<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/05_GES_Aware_Genomic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# COLAB NOTEBOOK FILE NAME:
# 05_GES_Aware_Genomic_RAG.ipynb
#
# EXPERIMENT 2 — STAGE 7A — CELL 7A0
# EXPLICIT GO/NO-GO DECISION, SCIENTIFIC BOUNDARY, PROTOCOL FREEZE,
# CHECKSUM PROTECTION, AND AUTHORIZATION OF THE RAG CORPUS PREFLIGHT
#
# This cell:
#   1. Verifies the completed Stage 6C final integrated manifest.
#   2. Confirms that Experiment 2 was not previously authorized or started.
#   3. Records a constrained GO decision for an Experiment 2 pilot.
#   4. Freezes the Experiment 2 design before corpus construction or LLM testing.
#   5. Does NOT construct the corpus, generate embeddings, call an LLM, or inspect RAG results.
#
# Scientific boundary:
#   - GES is treated only as a relative evidence-warning/prioritization index.
#   - GES is NOT treated as a calibrated probability.
#   - The inherited P(stable)=0.50 threshold will NOT be used as a RAG decision threshold.
#   - Low-stability evidence will NOT be removed in the primary intervention.
#   - Star-aware and combined-metadata-aware RAG are mandatory comparators.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PROJECT ROOT AND IMMUTABLE STAGE 6C FINAL MANIFEST
# --------------------------------------------------------------------------------------------------

NOTEBOOK_FILENAME = "05_GES_Aware_Genomic_RAG.ipynb"

ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
if not ROOT.exists():
    raise FileNotFoundError(
        f"Project directory was not found: {ROOT}\n"
        "Confirm that Google Drive is mounted and the project folder name is unchanged."
    )

STAGE6C_FINAL_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4k0_final_integrated_package_freeze_v1/"
    "stage6c_4k0_final_integrated_package_manifest_v1.json"
)

EXPECTED_STAGE6C_MANIFEST_SHA256 = (
    "de2089a245d80e5feafafbf2ceed3559b0028a8cc38deb5e89379ca67bb39fcb"
)

EXPECTED_STAGE6C_DECISION = (
    "PASS_STAGE6C_FINAL_INTEGRATED_PACKAGE_FROZEN_CHECKSUM_PROTECTED_"
    "SEMANTICALLY_READ_BACK_AND_FRESHLY_REVERIFIED"
)


# --------------------------------------------------------------------------------------------------
# 2. EXPERIMENT 2 OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

STAGE7_CONFIG_DIR = ROOT / "configs/stage7_rag"
STAGE7_QC_DIR = ROOT / "outputs/quality_checks/stage7_rag"
STAGE7_TABLE_DIR = ROOT / "outputs/tables/stage7_rag"
STAGE7_LOG_DIR = ROOT / "outputs/logs/stage7_rag"
STAGE7_DATA_DIR = ROOT / "data_processed/stage7_rag"
STAGE7_MODEL_DIR = ROOT / "models/stage7_rag"

for directory in [
    STAGE7_CONFIG_DIR,
    STAGE7_QC_DIR,
    STAGE7_TABLE_DIR,
    STAGE7_LOG_DIR,
    STAGE7_DATA_DIR,
    STAGE7_MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PROTOCOL_PATH = (
    STAGE7_CONFIG_DIR /
    "experiment2_ges_aware_rag_protocol_v1.json"
)

DECISION_PATH = (
    STAGE7_CONFIG_DIR /
    "experiment2_go_no_go_authorization_v1.json"
)

QC_PATH = (
    STAGE7_QC_DIR /
    "cell_7a0_experiment2_authorization_qc_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC AND STABLE-WRITE HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Return the SHA-256 digest of a file."""
    path = Path(path)

    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)

    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    """Return the standard .sha256 sidecar path."""
    path = Path(path)
    return path.with_name(path.name + ".sha256")


def read_sidecar_digest(path: Path) -> str:
    """Read the first SHA-256 digest from a sidecar."""
    text = Path(path).read_text(encoding="utf-8").strip()

    for token in text.replace("\n", " ").split():
        candidate = token.strip().lower()
        if len(candidate) == 64 and all(c in "0123456789abcdef" for c in candidate):
            return candidate

    raise ValueError(f"No valid SHA-256 digest found in sidecar: {path}")


def sidecar_is_valid(path: Path) -> bool:
    """Verify that a file has a matching .sha256 sidecar."""
    path = Path(path)
    sidecar = sidecar_path(path)

    return (
        path.exists()
        and sidecar.exists()
        and read_sidecar_digest(sidecar) == sha256_file(path)
    )


def json_native(value):
    """Convert common NumPy/Pandas objects into JSON-native values."""
    if isinstance(value, dict):
        return {str(key): json_native(item) for key, item in value.items()}

    if isinstance(value, (list, tuple)):
        return [json_native(item) for item in value]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if value is pd.NA:
        return None

    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    """
    Write a new frozen artifact atomically.

    On rerun:
      - accept byte-identical content;
      - refuse to overwrite nonidentical content.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}"
    )
    temporary.write_bytes(payload)

    temporary_hash = sha256_file(temporary)

    if path.exists():
        existing_hash = sha256_file(path)

        if existing_hash != temporary_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(
                "Refusing to overwrite a nonidentical frozen artifact:\n"
                f"{path}\n"
                f"Existing SHA-256: {existing_hash}\n"
                f"Proposed SHA-256: {temporary_hash}"
            )

        temporary.unlink(missing_ok=True)

    else:
        os.replace(temporary, path)

    return sha256_file(path)


def stable_write_json(path: Path, obj: dict) -> str:
    """Write deterministic JSON and return its SHA-256 digest."""
    payload = (
        json.dumps(
            json_native(obj),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")

    return stable_write_bytes(path, payload)


def write_sha256_sidecar(path: Path) -> Path:
    """Create or verify the standard SHA-256 sidecar."""
    path = Path(path)
    output = sidecar_path(path)

    payload = f"{sha256_file(path)}  {path.name}\n".encode("utf-8")
    stable_write_bytes(output, payload)

    return output


# --------------------------------------------------------------------------------------------------
# 4. VERIFY THE FINAL EXPERIMENT 1 / STAGE 6C BOUNDARY
# --------------------------------------------------------------------------------------------------

if not STAGE6C_FINAL_MANIFEST.exists():
    raise FileNotFoundError(
        "The final Stage 6C manifest was not found:\n"
        f"{STAGE6C_FINAL_MANIFEST}"
    )

observed_stage6c_hash = sha256_file(STAGE6C_FINAL_MANIFEST)

if observed_stage6c_hash != EXPECTED_STAGE6C_MANIFEST_SHA256:
    raise RuntimeError(
        "Final Stage 6C manifest SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_STAGE6C_MANIFEST_SHA256}\n"
        f"Observed: {observed_stage6c_hash}"
    )

if not sidecar_is_valid(STAGE6C_FINAL_MANIFEST):
    raise RuntimeError(
        "Final Stage 6C manifest sidecar verification failed:\n"
        f"{sidecar_path(STAGE6C_FINAL_MANIFEST)}"
    )

stage6c_manifest = json.loads(
    STAGE6C_FINAL_MANIFEST.read_text(encoding="utf-8")
)

if stage6c_manifest.get("decision") != EXPECTED_STAGE6C_DECISION:
    raise RuntimeError(
        "The Stage 6C terminal decision does not match the required PASS decision.\n"
        f"Observed decision: {stage6c_manifest.get('decision')}"
    )

stage6c_status = stage6c_manifest.get("stage6c_status", {})

if stage6c_status.get("experiment_2_started") is not False:
    raise RuntimeError(
        "The final Stage 6C manifest does not preserve "
        "experiment_2_started=False."
    )

if stage6c_status.get("experiment_2_authorized") is not False:
    raise RuntimeError(
        "The final Stage 6C manifest does not preserve "
        "experiment_2_authorized=False."
    )


# --------------------------------------------------------------------------------------------------
# 5. PRESERVE ONE CREATION TIMESTAMP ACROSS SAFE RERUNS
# --------------------------------------------------------------------------------------------------

existing_created_utc = None

for existing_path in [DECISION_PATH, PROTOCOL_PATH, QC_PATH]:
    if existing_path.exists():
        try:
            existing_object = json.loads(
                existing_path.read_text(encoding="utf-8")
            )
            existing_created_utc = existing_object.get("created_utc")

            if existing_created_utc:
                break

        except Exception:
            pass

CREATED_UTC = (
    existing_created_utc
    if existing_created_utc
    else datetime.now(timezone.utc).isoformat()
)


# --------------------------------------------------------------------------------------------------
# 6. FREEZE THE EXPERIMENT 2 PROTOCOL
# --------------------------------------------------------------------------------------------------

experiment2_protocol = {
    "protocol_name": "GES-aware genomic RAG intervention protocol",
    "protocol_version": "1.0.0",
    "created_utc": CREATED_UTC,
    "notebook_filename": NOTEBOOK_FILENAME,

    "study_title": (
        "GES-RAG: Temporal Validation and Stability-Aware Context Assembly "
        "for Reliable Genomic Question Answering"
    ),

    "experiment": {
        "number": 2,
        "name": "GES-aware genomic RAG intervention",
        "blueprint_stage": 7,
        "current_cell": "7A0",
        "current_activity": (
            "Explicit authorization and protocol freeze before corpus construction"
        ),
    },

    "stage6c_parent_boundary": {
        "manifest_path": str(STAGE6C_FINAL_MANIFEST),
        "manifest_sha256": observed_stage6c_hash,
        "terminal_decision": stage6c_manifest["decision"],
        "required_categories_verified": 14,
        "experiment2_started_in_parent_manifest": False,
        "experiment2_authorized_in_parent_manifest": False,
    },

    "go_no_go_decision": {
        "decision": "GO_CONSTRAINED_PILOT",
        "full_unrestricted_deployment_authorized": False,
        "clinical_use_authorized": False,
        "reason": (
            "Experiment 1 identified a modest and heterogeneous relative ranking "
            "signal, localized low-score enrichment, poor probability calibration, "
            "no primary AUPRC advantage over combined metadata, and substantial "
            "false-stable failure for newly emerging unresolved conflict. "
            "A controlled RAG experiment remains scientifically justified only as "
            "a constrained intervention study with strong comparators and explicit "
            "negative-result protections."
        ),
    },

    "scientific_interpretation_boundary": {
        "ges_role": "relative_warning_and_prioritization_index",
        "ges_is_calibrated_probability": False,
        "ges_is_clinical_decision_threshold": False,
        "ges_is_demonstrated_rag_safety_intervention": False,
        "use_inherited_pstable_0_50_threshold": False,
        "primary_hard_exclusion_of_low_stability_evidence": False,
        "allow_soft_warning_and_cautious_answering": True,
        "allow_abstention_when_evidence_policy_requires_it": True,
    },

    "corpus_design": {
        "source_release": "ClinVar January 2026",
        "embedded_data_cutoff": "2025-12-27",
        "primary_unit": "RCV-level variant-condition aggregate",
        "primary_genes": ["BRCA1", "BRCA2", "MLH1"],
        "exploratory_genes": ["EGFR"],
        "expected_evidence_packet_fields": [
            "gene",
            "variation_id",
            "vcv_accession",
            "rcv_accession",
            "condition_names",
            "condition_identifiers",
            "aggregate_classification",
            "normalized_classification_group",
            "review_status",
            "review_stars",
            "aggregate_conflict_flag",
            "last_evaluated",
            "submitter_count",
            "submitter_identifiers",
            "nested_scv_assertions",
            "source_release",
            "source_cutoff",
            "ges_stability_index",
            "ges_instability_risk_rank",
            "combined_metadata_risk",
        ],
        "answer_not_encoded_in_hidden_metadata": True,
    },

    "dataset_partition": {
        "partition_unit": "unique_variant_condition_pair",
        "development_fraction": 0.20,
        "locked_test_fraction": 0.80,
        "questions_for_same_variant_may_cross_partitions": False,
        "stratification_targets": [
            "high-stability consensus",
            "active conflict",
            "low-review or low-stability",
            "same-star different metadata",
            "temporally reclassified",
            "primary gene",
            "exploratory EGFR",
        ],
        "random_seed": 42,
    },

    "experimental_conditions": [
        {
            "condition_id": "A",
            "name": "standard_semantic_rag",
            "description": (
                "Semantic retrieval and ranking without star, combined-metadata, "
                "or GES intervention."
            ),
        },
        {
            "condition_id": "B",
            "name": "star_aware_rag",
            "description": (
                "Semantic candidate pool with ClinVar review-star-aware reranking."
            ),
        },
        {
            "condition_id": "C",
            "name": "combined_metadata_aware_rag",
            "description": (
                "Semantic candidate pool with the frozen combined-metadata "
                "heuristic used as the quality-ranking baseline."
            ),
        },
        {
            "condition_id": "D",
            "name": "ges_aware_reranking",
            "description": (
                "Semantic candidate pool reranked using the frozen GES relative "
                "stability signal; no hard deletion."
            ),
        },
        {
            "condition_id": "E",
            "name": "ges_soft_quality_gate",
            "description": (
                "GES-aware reranking plus explicit conflict disclosure, "
                "uncertainty warning, cautious answering, and permitted abstention."
            ),
        },
        {
            "condition_id": "F",
            "name": "random_score_control",
            "description": (
                "A random score with a distribution comparable to GES, used to "
                "test whether improvement comes merely from changing rank order."
            ),
        },
    ],

    "primary_rag_endpoint": {
        "name": "evidence_grounded_response_success",
        "type": "binary",
        "criteria_all_required": [
            "The factual or classification answer is correct.",
            "The cited evidence supports the answer.",
            "Existing conflict is disclosed when present.",
            "No unsupported clinical recommendation is made.",
            "The response is cautious or abstains when required by the frozen evidence policy.",
        ],
        "primary_comparison": (
            "ges_soft_quality_gate versus standard_semantic_rag"
        ),
        "important_secondary_comparisons": [
            "ges_soft_quality_gate versus star_aware_rag",
            "ges_soft_quality_gate versus combined_metadata_aware_rag",
            "ges_aware_reranking versus star_aware_rag",
            "ges_aware_reranking versus combined_metadata_aware_rag",
        ],
    },

    "secondary_outcomes": [
        "classification_accuracy",
        "conflict_detection_precision",
        "conflict_detection_recall",
        "conflict_detection_f1",
        "citation_precision",
        "citation_recall",
        "evidence_faithfulness",
        "unsupported_claim_rate",
        "appropriate_abstention_rate",
        "unsafe_overconfidence_rate",
        "selective_accuracy",
        "risk_coverage",
        "run_to_run_disagreement",
    ],

    "generation_reproducibility": {
        "fixed_model_required": True,
        "dated_model_version_required": True,
        "fixed_system_prompt": True,
        "temperature": 0,
        "fixed_context_length": True,
        "fixed_generation_limit": True,
        "fixed_output_schema": True,
        "planned_repetitions_per_question_condition": 3,
        "api_keys_must_not_be_saved_in_artifacts": True,
    },

    "required_ablation_and_stress_tests": [
        "GES without review stars",
        "Same-star subset",
        "Ranking versus warning-label separation",
        "Lost-in-the-middle position stress test",
        "Random-score control",
        "Hard-exclusion sensitivity analysis only",
    ],

    "negative_result_policy": {
        "primary_endpoint_will_not_be_replaced_post_hoc": True,
        "locked_test_prompts_will_not_be_modified_after_review": True,
        "failure_to_outperform_star_or_combined_metadata_will_be_reported": True,
        "secondary_comparisons_require_multiplicity_control_or_exploratory_label": True,
    },

    "safety_scope": {
        "operational_definition": (
            "Fewer unsupported claims, fewer failures to disclose conflicts, "
            "fewer overly definitive answers based on weak evidence, and more "
            "appropriate cautious answering or abstention."
        ),
        "patient_outcomes_measured": False,
        "treatment_safety_measured": False,
        "real_world_clinical_effectiveness_measured": False,
        "research_prototype_only": True,
    },

    "authorized_next_cell": {
        "cell_id": "7A1",
        "name": "T1 corpus-source verification and schema inventory",
        "authorized_actions": [
            "Verify the accepted T1 Parquet and its checksum-controlled freeze manifest.",
            "Inventory available aggregate and nested-SCV fields.",
            "Confirm one-row-per-RCV key integrity.",
            "Confirm classification-axis provenance and source cutoff.",
            "Determine which evidence-packet fields are directly derivable.",
            "Produce a read-only corpus-preflight report.",
        ],
        "prohibited_actions": [
            "Do not call an LLM.",
            "Do not create embeddings.",
            "Do not tune reranking weights.",
            "Do not create development or test answers.",
            "Do not inspect RAG performance.",
            "Do not modify any Experiment 1 artifact.",
        ],
    },
}


# --------------------------------------------------------------------------------------------------
# 7. WRITE AND CHECKSUM-FREEZE THE PROTOCOL
# --------------------------------------------------------------------------------------------------

protocol_sha256 = stable_write_json(
    PROTOCOL_PATH,
    experiment2_protocol,
)
write_sha256_sidecar(PROTOCOL_PATH)


# --------------------------------------------------------------------------------------------------
# 8. CREATE THE EXPLICIT EXPERIMENT 2 AUTHORIZATION RECORD
# --------------------------------------------------------------------------------------------------

authorization_record = {
    "record_name": "Experiment 2 explicit go/no-go authorization",
    "record_version": "1.0.0",
    "created_utc": CREATED_UTC,
    "notebook_filename": NOTEBOOK_FILENAME,
    "cell_id": "7A0",

    "parent_stage6c_manifest": {
        "path": str(STAGE6C_FINAL_MANIFEST),
        "sha256": observed_stage6c_hash,
        "sidecar_verified": True,
        "decision": stage6c_manifest["decision"],
    },

    "experiment2_protocol": {
        "path": str(PROTOCOL_PATH),
        "sha256": protocol_sha256,
        "sidecar_verified": sidecar_is_valid(PROTOCOL_PATH),
    },

    "decision": "GO_CONSTRAINED_PILOT",

    "authorization_status": {
        "experiment_2_authorized": True,
        "experiment_2_scientific_execution_started": False,
        "stage7a1_corpus_preflight_authorized": True,
        "llm_generation_authorized": False,
        "embedding_construction_authorized": False,
        "rag_performance_analysis_authorized": False,
        "clinical_use_authorized": False,
    },

    "mandatory_conditions": [
        "GES must remain a relative warning/prioritization index.",
        "The P(stable)=0.50 threshold must not be used as a RAG decision threshold.",
        "The primary intervention must not hard-delete low-stability evidence.",
        "Star-aware and combined-metadata-aware RAG must remain mandatory comparators.",
        "The primary endpoint must remain evidence-grounded response success.",
        "The development and locked-test split must occur by variant-condition pair.",
        "No locked-test prompt or endpoint may be changed after results are examined.",
    ],

    "next_authorized_cell": "7A1",

    "terminal_decision": (
        "PASS_EXPERIMENT2_CONSTRAINED_PILOT_AUTHORIZED_PROTOCOL_FROZEN_"
        "CHECKSUM_PROTECTED_STAGE7A1_CORPUS_PREFLIGHT_ONLY"
    ),
}

decision_sha256 = stable_write_json(
    DECISION_PATH,
    authorization_record,
)
write_sha256_sidecar(DECISION_PATH)


# --------------------------------------------------------------------------------------------------
# 9. CREATE CELL 7A0 QC RECORD
# --------------------------------------------------------------------------------------------------

qc_checks = {
    "project_root_exists": ROOT.exists(),
    "stage6c_final_manifest_exists": STAGE6C_FINAL_MANIFEST.exists(),
    "stage6c_manifest_hash_matches": (
        observed_stage6c_hash == EXPECTED_STAGE6C_MANIFEST_SHA256
    ),
    "stage6c_manifest_sidecar_valid": sidecar_is_valid(
        STAGE6C_FINAL_MANIFEST
    ),
    "stage6c_terminal_decision_matches": (
        stage6c_manifest.get("decision") == EXPECTED_STAGE6C_DECISION
    ),
    "experiment2_was_not_started_by_stage6c": (
        stage6c_status.get("experiment_2_started") is False
    ),
    "experiment2_was_not_authorized_by_stage6c": (
        stage6c_status.get("experiment_2_authorized") is False
    ),
    "experiment2_protocol_written": PROTOCOL_PATH.exists(),
    "experiment2_protocol_sidecar_valid": sidecar_is_valid(PROTOCOL_PATH),
    "authorization_record_written": DECISION_PATH.exists(),
    "authorization_record_sidecar_valid": sidecar_is_valid(DECISION_PATH),
    "decision_is_constrained_pilot": (
        authorization_record["decision"] == "GO_CONSTRAINED_PILOT"
    ),
    "clinical_use_not_authorized": (
        authorization_record["authorization_status"][
            "clinical_use_authorized"
        ] is False
    ),
    "llm_generation_not_yet_authorized": (
        authorization_record["authorization_status"][
            "llm_generation_authorized"
        ] is False
    ),
    "next_authorized_cell_is_7a1": (
        authorization_record["next_authorized_cell"] == "7A1"
    ),
}

failed_checks = [
    name for name, passed in qc_checks.items() if passed is not True
]

qc_record = {
    "qc_name": "Cell 7A0 Experiment 2 authorization QC",
    "qc_version": "1.0.0",
    "created_utc": CREATED_UTC,
    "notebook_filename": NOTEBOOK_FILENAME,
    "cell_id": "7A0",
    "checks": qc_checks,
    "checks_total": len(qc_checks),
    "checks_passed": len(qc_checks) - len(failed_checks),
    "checks_failed": len(failed_checks),
    "failed_check_names": failed_checks,
    "protocol_sha256": protocol_sha256,
    "authorization_sha256": decision_sha256,
    "decision": (
        "PASS" if not failed_checks else "FAIL"
    ),
}

qc_sha256 = stable_write_json(QC_PATH, qc_record)
write_sha256_sidecar(QC_PATH)

if failed_checks:
    raise RuntimeError(
        "Cell 7A0 QC failed:\n- " + "\n- ".join(failed_checks)
    )


# --------------------------------------------------------------------------------------------------
# 10. FRESH SEMANTIC READBACK AND CRYPTOGRAPHIC REVERIFICATION
# --------------------------------------------------------------------------------------------------

fresh_protocol = json.loads(
    PROTOCOL_PATH.read_text(encoding="utf-8")
)

fresh_authorization = json.loads(
    DECISION_PATH.read_text(encoding="utf-8")
)

fresh_qc = json.loads(
    QC_PATH.read_text(encoding="utf-8")
)

if not sidecar_is_valid(PROTOCOL_PATH):
    raise RuntimeError("Fresh protocol sidecar verification failed.")

if not sidecar_is_valid(DECISION_PATH):
    raise RuntimeError("Fresh authorization sidecar verification failed.")

if not sidecar_is_valid(QC_PATH):
    raise RuntimeError("Fresh QC sidecar verification failed.")

if fresh_protocol["go_no_go_decision"]["decision"] != "GO_CONSTRAINED_PILOT":
    raise RuntimeError("Fresh protocol decision readback mismatch.")

if fresh_authorization["authorization_status"]["experiment_2_authorized"] is not True:
    raise RuntimeError("Fresh Experiment 2 authorization readback mismatch.")

if fresh_authorization["authorization_status"]["llm_generation_authorized"] is not False:
    raise RuntimeError("LLM generation was incorrectly authorized by Cell 7A0.")

if fresh_authorization["next_authorized_cell"] != "7A1":
    raise RuntimeError("Next authorized cell readback mismatch.")

if fresh_qc["checks_failed"] != 0:
    raise RuntimeError("Fresh QC readback reports failed checks.")


# --------------------------------------------------------------------------------------------------
# 11. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

separator = "=" * 120

print("\n" + separator)
print("EXPERIMENT 2 — STAGE 7A — CELL 7A0")
print("EXPLICIT GO/NO-GO DECISION AND PROTOCOL FREEZE")
print(separator)

print(f"Notebook                                  : {NOTEBOOK_FILENAME}")
print(f"Project root                              : {ROOT}")
print()
print("PARENT EXPERIMENT 1 BOUNDARY")
print(f"Stage 6C manifest                         : {STAGE6C_FINAL_MANIFEST.name}")
print(f"Stage 6C manifest SHA-256                 : {observed_stage6c_hash}")
print(f"Stage 6C sidecar                          : PASS")
print(f"Stage 6C terminal decision                : PASS")
print()
print("EXPERIMENT 2 DECISION")
print(f"Decision                                  : GO_CONSTRAINED_PILOT")
print(f"Experiment 2 authorized                   : YES")
print(f"Scientific RAG execution started          : NO")
print(f"LLM generation authorized                 : NO")
print(f"Embedding generation authorized           : NO")
print(f"Clinical use authorized                   : NO")
print()
print("SCIENTIFIC BOUNDARY")
print("GES interpretation                        : Relative warning/prioritization index")
print("GES treated as calibrated probability     : NO")
print("Use inherited P(stable)=0.50 threshold     : NO")
print("Primary hard evidence exclusion           : NO")
print("Star-aware comparator mandatory            : YES")
print("Combined-metadata comparator mandatory     : YES")
print()
print("FROZEN OUTPUTS")
print(f"Protocol                                   : {PROTOCOL_PATH}")
print(f"Protocol SHA-256                           : {protocol_sha256}")
print(f"Authorization                              : {DECISION_PATH}")
print(f"Authorization SHA-256                      : {decision_sha256}")
print(f"QC record                                  : {QC_PATH}")
print(f"QC SHA-256                                 : {qc_sha256}")
print(f"QC checks                                  : {len(qc_checks)}/{len(qc_checks)} PASS")
print()
print("NEXT AUTHORIZED CELL")
print("Cell 7A1                                  : T1 corpus-source verification")
print("                                              and schema inventory")
print()
print(
    "FINAL DECISION                            : "
    "PASS_EXPERIMENT2_CONSTRAINED_PILOT_AUTHORIZED_"
    "PROTOCOL_FROZEN_CHECKSUM_PROTECTED_"
    "STAGE7A1_CORPUS_PREFLIGHT_ONLY"
)
print(separator)

Mounted at /content/drive

EXPERIMENT 2 — STAGE 7A — CELL 7A0
EXPLICIT GO/NO-GO DECISION AND PROTOCOL FREEZE
Notebook                                  : 05_GES_Aware_Genomic_RAG.ipynb
Project root                              : /content/drive/MyDrive/GES_RAG_Temporal_Study

PARENT EXPERIMENT 1 BOUNDARY
Stage 6C manifest                         : stage6c_4k0_final_integrated_package_manifest_v1.json
Stage 6C manifest SHA-256                 : de2089a245d80e5feafafbf2ceed3559b0028a8cc38deb5e89379ca67bb39fcb
Stage 6C sidecar                          : PASS
Stage 6C terminal decision                : PASS

EXPERIMENT 2 DECISION
Decision                                  : GO_CONSTRAINED_PILOT
Experiment 2 authorized                   : YES
Scientific RAG execution started          : NO
LLM generation authorized                 : NO
Embedding generation authorized           : NO
Clinical use authorized                   : NO

SCIENTIFIC BOUNDARY
GES interpretation                        : Re

In [3]:
# ==================================================================================================
# EXPERIMENT 2 — STAGE 7A — CELL 7A1
# FROZEN T1 CORPUS-SOURCE VERIFICATION, TOP-LEVEL SCHEMA INVENTORY,
# NESTED-SCV KEY INVENTORY, EVIDENCE-PACKET DERIVABILITY AUDIT,
# CHECKSUM PROTECTION, AND AUTHORIZATION OF CELL 7A2
#
# This cell:
#   1. Re-verifies the Cell 7A0 protocol and authorization package.
#   2. Re-verifies the accepted January 2026 T1 ClinVar RCV Parquet.
#   3. Re-verifies the T1 extraction-validation freeze manifest.
#   4. Confirms rows, columns, unique RCV keys, genes, classification axes,
#      nested-SCV accounting, conflict counts, and accepted condition-ID missingness.
#   5. Inventories all top-level Parquet fields.
#   6. Inventories all keys contained in nested SCV records.
#   7. Determines which planned RAG evidence-packet fields are:
#         - directly available;
#         - deterministically derivable;
#         - dependent on later frozen-model scoring.
#   8. Freezes the Stage 7A1 preflight package.
#
# This cell DOES NOT:
#   - construct the RAG corpus;
#   - apply GES or combined-metadata scores to T1;
#   - create embeddings;
#   - create development/test partitions;
#   - generate questions;
#   - call an LLM;
#   - inspect RAG performance;
#   - modify any Experiment 1 artifact.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. PROJECT ROOT AND CELL 7A0 AUTHORIZATION PACKAGE
# --------------------------------------------------------------------------------------------------

NOTEBOOK_FILENAME = "05_GES_Aware_Genomic_RAG.ipynb"
CELL_ID = "7A1"

ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

if not ROOT.exists():
    raise FileNotFoundError(
        f"Project root was not found:\n{ROOT}"
    )

STAGE7_CONFIG_DIR = ROOT / "configs" / "stage7_rag"
STAGE7_QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"
STAGE7_TABLE_DIR = ROOT / "outputs" / "tables" / "stage7_rag"

for directory in [
    STAGE7_CONFIG_DIR,
    STAGE7_QC_DIR,
    STAGE7_TABLE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PROTOCOL_PATH = (
    STAGE7_CONFIG_DIR
    / "experiment2_ges_aware_rag_protocol_v1.json"
)

AUTHORIZATION_PATH = (
    STAGE7_CONFIG_DIR
    / "experiment2_go_no_go_authorization_v1.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "db0fb3f197356118fd069c36e4527730f4bb27ebece709038f51b522cb6aa332"
)

EXPECTED_AUTHORIZATION_SHA256 = (
    "7ea8f5f8a663ddc82a6434e9f08b6da2f3e70981cb4974270f5ec1ed4cf2d91f"
)

EXPECTED_AUTHORIZATION_DECISION = (
    "PASS_EXPERIMENT2_CONSTRAINED_PILOT_AUTHORIZED_PROTOCOL_FROZEN_"
    "CHECKSUM_PROTECTED_STAGE7A1_CORPUS_PREFLIGHT_ONLY"
)


# --------------------------------------------------------------------------------------------------
# 2. IMMUTABLE T1 SOURCE PACKAGE
# --------------------------------------------------------------------------------------------------

T1_PARQUET = (
    ROOT
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

T1_VALIDATION_REPORT = (
    ROOT
    / "outputs"
    / "quality_checks"
    / "clinvar_t1_target_gene_rcv_field_validation_v1.json"
)

T1_FREEZE_MANIFEST = (
    ROOT
    / "configs"
    / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
)

EXPECTED_T1_PARQUET_SHA256 = (
    "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"
)

EXPECTED_T1_MANIFEST_SHA256 = (
    "7eaeff0fee3df96973130f721d6c1f2a02fd9108e85af7b2743cdd7750a4372e"
)

EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36
EXPECTED_T1_NESTED_SCVS = 145_400
EXPECTED_T1_CONFLICT_POSITIVE = 6_602
EXPECTED_T1_EMPTY_CONDITION_IDS = 730
EXPECTED_T1_CUTOFF = "2025-12-27"

EXPECTED_GENE_COUNTS = {
    "BRCA1": 32_603,
    "BRCA2": 49_221,
    "MLH1": 13_684,
    "EGFR": 5_412,
}

EXPECTED_STUDY_SCOPE_COUNTS = {
    "primary": 95_508,
    "exploratory": 5_412,
}

EXPECTED_AXIS_COUNTS = {
    "GermlineClassification": 97_526,
    "OncogenicityClassification": 52,
    "SomaticClinicalImpact": 25,
    "NoClassification": 3_317,
}


# --------------------------------------------------------------------------------------------------
# 3. CELL 7A1 OUTPUT PACKAGE
# --------------------------------------------------------------------------------------------------

TOP_LEVEL_SCHEMA_CSV = (
    STAGE7_TABLE_DIR
    / "cell_7a1_t1_top_level_schema_inventory_v1.csv"
)

NESTED_SCV_SCHEMA_CSV = (
    STAGE7_TABLE_DIR
    / "cell_7a1_t1_nested_scv_key_inventory_v1.csv"
)

DERIVABILITY_CSV = (
    STAGE7_TABLE_DIR
    / "cell_7a1_evidence_packet_derivability_v1.csv"
)

PREFLIGHT_REPORT_PATH = (
    STAGE7_QC_DIR
    / "cell_7a1_t1_corpus_source_preflight_v1.json"
)

QC_PATH = (
    STAGE7_QC_DIR
    / "cell_7a1_t1_corpus_source_preflight_qc_v1.json"
)

MANIFEST_PATH = (
    STAGE7_CONFIG_DIR
    / "cell_7a1_t1_corpus_source_preflight_manifest_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 4. CRYPTOGRAPHIC AND STABLE-WRITE HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """Return the SHA-256 checksum of a file."""
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    """Return the standard filename.ext.sha256 sidecar path."""
    path = Path(path)
    return path.with_name(path.name + ".sha256")


def read_sidecar_digest(path: Path) -> str:
    """Extract the first valid SHA-256 digest from a sidecar."""
    text = Path(path).read_text(
        encoding="utf-8"
    ).strip()

    for token in text.replace("\n", " ").split():
        candidate = token.strip().lower()

        if (
            len(candidate) == 64
            and all(
                character in "0123456789abcdef"
                for character in candidate
            )
        ):
            return candidate

    raise ValueError(
        f"No valid SHA-256 digest was found in:\n{path}"
    )


def sidecar_is_valid(
    path: Path,
    required: bool = True,
) -> bool:
    """
    Verify a file's standard .sha256 sidecar.

    When required=False, absence of the sidecar is allowed.
    """
    path = Path(path)
    sidecar = sidecar_path(path)

    if not path.exists():
        return False

    if not sidecar.exists():
        return not required

    return (
        read_sidecar_digest(sidecar)
        == sha256_file(path)
    )


def json_native(value):
    """Convert common scientific Python objects to JSON-native values."""
    if isinstance(value, dict):
        return {
            str(key): json_native(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            json_native(item)
            for item in value
        ]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return (
            None
            if not np.isfinite(value)
            else float(value)
        )

    if isinstance(value, np.bool_):
        return bool(value)

    if value is pd.NA:
        return None

    return value


def stable_write_bytes(
    path: Path,
    payload: bytes,
) -> str:
    """
    Atomically write a new artifact.

    A rerun accepts byte-identical content and refuses to overwrite
    nonidentical frozen content.
    """
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}"
    )

    temporary_path.write_bytes(payload)
    proposed_sha256 = sha256_file(temporary_path)

    if path.exists():
        existing_sha256 = sha256_file(path)

        if existing_sha256 != proposed_sha256:
            temporary_path.unlink(
                missing_ok=True
            )

            raise RuntimeError(
                "Refusing to overwrite a nonidentical frozen artifact.\n"
                f"Artifact: {path}\n"
                f"Existing SHA-256: {existing_sha256}\n"
                f"Proposed SHA-256: {proposed_sha256}"
            )

        temporary_path.unlink(
            missing_ok=True
        )

    else:
        os.replace(
            temporary_path,
            path,
        )

    return sha256_file(path)


def stable_write_json(
    path: Path,
    obj: dict,
) -> str:
    """Write deterministic, sorted JSON."""
    payload = (
        json.dumps(
            json_native(obj),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")

    return stable_write_bytes(
        path,
        payload,
    )


def stable_write_dataframe_csv(
    path: Path,
    dataframe: pd.DataFrame,
) -> str:
    """Write a deterministic CSV file."""
    payload = dataframe.to_csv(
        index=False,
        lineterminator="\n",
    ).encode("utf-8")

    return stable_write_bytes(
        path,
        payload,
    )


def write_sha256_sidecar(
    path: Path,
) -> Path:
    """Create or verify a standard SHA-256 sidecar."""
    path = Path(path)
    output_path = sidecar_path(path)

    payload = (
        f"{sha256_file(path)}  {path.name}\n"
    ).encode("utf-8")

    stable_write_bytes(
        output_path,
        payload,
    )

    return output_path


def parse_json_value(value):
    """Parse a JSON-serialized value while accepting an already parsed object."""
    if isinstance(
        value,
        (list, dict),
    ):
        return value

    if value is None or value is pd.NA:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return json.loads(
        str(value)
    )


def recursively_contains_scalar(
    obj,
    target,
) -> bool:
    """Search nested dictionaries/lists for an exact scalar value."""
    if isinstance(obj, dict):
        return any(
            recursively_contains_scalar(
                value,
                target,
            )
            for value in obj.values()
        )

    if isinstance(obj, list):
        return any(
            recursively_contains_scalar(
                value,
                target,
            )
            for value in obj
        )

    return obj == target


def first_existing_column(
    columns,
    preferred_names,
    semantic_tokens=None,
):
    """Resolve a column by preferred exact name or semantic tokens."""
    columns = list(columns)

    for name in preferred_names:
        if name in columns:
            return name

    if semantic_tokens:
        matches = []

        for column in columns:
            normalized = column.lower()

            if all(
                token.lower() in normalized
                for token in semantic_tokens
            ):
                matches.append(column)

        if len(matches) == 1:
            return matches[0]

    return None


# --------------------------------------------------------------------------------------------------
# 5. VERIFY THE CELL 7A0 AUTHORIZATION PACKAGE
# --------------------------------------------------------------------------------------------------

for required_path in [
    PROTOCOL_PATH,
    AUTHORIZATION_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Cell 7A0 artifact was not found:\n{required_path}"
        )

observed_protocol_sha256 = sha256_file(
    PROTOCOL_PATH
)

observed_authorization_sha256 = sha256_file(
    AUTHORIZATION_PATH
)

if observed_protocol_sha256 != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError(
        "Experiment 2 protocol SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_PROTOCOL_SHA256}\n"
        f"Observed: {observed_protocol_sha256}"
    )

if observed_authorization_sha256 != EXPECTED_AUTHORIZATION_SHA256:
    raise RuntimeError(
        "Experiment 2 authorization SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_AUTHORIZATION_SHA256}\n"
        f"Observed: {observed_authorization_sha256}"
    )

if not sidecar_is_valid(
    PROTOCOL_PATH,
    required=True,
):
    raise RuntimeError(
        "Experiment 2 protocol sidecar verification failed."
    )

if not sidecar_is_valid(
    AUTHORIZATION_PATH,
    required=True,
):
    raise RuntimeError(
        "Experiment 2 authorization sidecar verification failed."
    )

experiment2_protocol = json.loads(
    PROTOCOL_PATH.read_text(
        encoding="utf-8"
    )
)

authorization_record = json.loads(
    AUTHORIZATION_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    authorization_record.get("terminal_decision")
    != EXPECTED_AUTHORIZATION_DECISION
):
    raise RuntimeError(
        "Cell 7A0 terminal authorization decision mismatch."
    )

if (
    authorization_record
    .get("authorization_status", {})
    .get("experiment_2_authorized")
    is not True
):
    raise RuntimeError(
        "Experiment 2 was not explicitly authorized."
    )

if (
    authorization_record
    .get("authorization_status", {})
    .get("stage7a1_corpus_preflight_authorized")
    is not True
):
    raise RuntimeError(
        "Cell 7A1 corpus preflight was not authorized."
    )

if (
    authorization_record
    .get("authorization_status", {})
    .get("llm_generation_authorized")
    is not False
):
    raise RuntimeError(
        "Cell 7A0 unexpectedly authorized LLM generation."
    )

if (
    authorization_record
    .get("authorization_status", {})
    .get("embedding_construction_authorized")
    is not False
):
    raise RuntimeError(
        "Cell 7A0 unexpectedly authorized embedding construction."
    )

if (
    authorization_record.get("next_authorized_cell")
    != "7A1"
):
    raise RuntimeError(
        "The next authorized cell in the Cell 7A0 record is not 7A1."
    )


# --------------------------------------------------------------------------------------------------
# 6. REVERIFY THE FINAL STAGE 6C PARENT BOUNDARY
# --------------------------------------------------------------------------------------------------

parent_information = authorization_record.get(
    "parent_stage6c_manifest",
    {},
)

parent_stage6c_path = Path(
    parent_information.get(
        "path",
        "",
    )
)

parent_stage6c_expected_sha256 = (
    parent_information.get("sha256")
)

if not parent_stage6c_path.exists():
    raise FileNotFoundError(
        "The parent Stage 6C final manifest referenced by Cell 7A0 "
        f"was not found:\n{parent_stage6c_path}"
    )

parent_stage6c_observed_sha256 = sha256_file(
    parent_stage6c_path
)

if (
    parent_stage6c_observed_sha256
    != parent_stage6c_expected_sha256
):
    raise RuntimeError(
        "Parent Stage 6C manifest checksum mismatch."
    )

if not sidecar_is_valid(
    parent_stage6c_path,
    required=True,
):
    raise RuntimeError(
        "Parent Stage 6C manifest sidecar verification failed."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY THE ACCEPTED T1 SOURCE PACKAGE
# --------------------------------------------------------------------------------------------------

for required_path in [
    T1_PARQUET,
    T1_VALIDATION_REPORT,
    T1_FREEZE_MANIFEST,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required frozen T1 artifact was not found:\n{required_path}"
        )

observed_t1_parquet_sha256 = sha256_file(
    T1_PARQUET
)

observed_t1_manifest_sha256 = sha256_file(
    T1_FREEZE_MANIFEST
)

observed_t1_validation_sha256 = sha256_file(
    T1_VALIDATION_REPORT
)

if (
    observed_t1_parquet_sha256
    != EXPECTED_T1_PARQUET_SHA256
):
    raise RuntimeError(
        "Accepted T1 Parquet SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_T1_PARQUET_SHA256}\n"
        f"Observed: {observed_t1_parquet_sha256}"
    )

if (
    observed_t1_manifest_sha256
    != EXPECTED_T1_MANIFEST_SHA256
):
    raise RuntimeError(
        "T1 extraction-validation manifest SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_T1_MANIFEST_SHA256}\n"
        f"Observed: {observed_t1_manifest_sha256}"
    )

if not sidecar_is_valid(
    T1_FREEZE_MANIFEST,
    required=True,
):
    raise RuntimeError(
        "T1 freeze-manifest sidecar verification failed."
    )

# The original T1 Parquet and validation report may or may not have standard
# sidecars. When present, they must be valid.
if not sidecar_is_valid(
    T1_PARQUET,
    required=False,
):
    raise RuntimeError(
        "A T1 Parquet sidecar exists but does not match the Parquet."
    )

if not sidecar_is_valid(
    T1_VALIDATION_REPORT,
    required=False,
):
    raise RuntimeError(
        "A T1 validation-report sidecar exists but does not match the report."
    )

t1_freeze_manifest = json.loads(
    T1_FREEZE_MANIFEST.read_text(
        encoding="utf-8"
    )
)

t1_validation_report = json.loads(
    T1_VALIDATION_REPORT.read_text(
        encoding="utf-8"
    )
)

if not recursively_contains_scalar(
    t1_freeze_manifest,
    "ACCEPTED_AND_FROZEN",
):
    raise RuntimeError(
        "The T1 freeze manifest does not contain the required "
        "ACCEPTED_AND_FROZEN decision."
    )


# --------------------------------------------------------------------------------------------------
# 8. PARQUET METADATA AND COMPLETE DATAFRAME LOAD
# --------------------------------------------------------------------------------------------------

parquet_file = pq.ParquetFile(
    T1_PARQUET
)

parquet_metadata = parquet_file.metadata
arrow_schema = parquet_file.schema_arrow
schema_columns = list(
    arrow_schema.names
)

if (
    parquet_metadata.num_rows
    != EXPECTED_T1_ROWS
):
    raise RuntimeError(
        "T1 Parquet row count mismatch.\n"
        f"Expected: {EXPECTED_T1_ROWS:,}\n"
        f"Observed: {parquet_metadata.num_rows:,}"
    )

if (
    parquet_metadata.num_columns
    != EXPECTED_T1_COLUMNS
):
    raise RuntimeError(
        "T1 Parquet column count mismatch.\n"
        f"Expected: {EXPECTED_T1_COLUMNS}\n"
        f"Observed: {parquet_metadata.num_columns}"
    )

t1 = pd.read_parquet(
    T1_PARQUET
).copy()

if len(t1) != EXPECTED_T1_ROWS:
    raise RuntimeError(
        "Loaded T1 dataframe row count does not match Parquet metadata."
    )

if list(t1.columns) != schema_columns:
    raise RuntimeError(
        "Loaded dataframe column order does not match the Parquet schema."
    )


# --------------------------------------------------------------------------------------------------
# 9. RESOLVE IMPORTANT SOURCE COLUMNS
# --------------------------------------------------------------------------------------------------

RCV_COLUMN = first_existing_column(
    schema_columns,
    ["rcv_accession"],
    ["rcv", "accession"],
)

VARIATION_ID_COLUMN = first_existing_column(
    schema_columns,
    ["variation_id"],
    ["variation", "id"],
)

VCV_COLUMN = first_existing_column(
    schema_columns,
    ["vcv_accession"],
    ["vcv", "accession"],
)

GENE_COLUMN = first_existing_column(
    schema_columns,
    ["target_genes_json"],
    ["target", "gene", "json"],
)

STUDY_SCOPE_COLUMN = first_existing_column(
    schema_columns,
    ["study_scope"],
    ["study", "scope"],
)

TIMEPOINT_COLUMN = first_existing_column(
    schema_columns,
    ["timepoint"],
    ["timepoint"],
)

RELEASE_LABEL_COLUMN = first_existing_column(
    schema_columns,
    ["release_label"],
    ["release", "label"],
)

CUTOFF_COLUMN = first_existing_column(
    schema_columns,
    [
        "embedded_data_cutoff",
        "embedded_cutoff",
        "release_embedded_cutoff",
    ],
    ["embedded", "cutoff"],
)

SCV_RECORDS_COLUMN = first_existing_column(
    schema_columns,
    ["scv_records_json"],
    ["scv", "records", "json"],
)

SCV_COUNT_COLUMN = first_existing_column(
    schema_columns,
    ["scv_count_xml"],
    ["scv", "count"],
)

CONDITION_IDS_COLUMN = first_existing_column(
    schema_columns,
    ["condition_ids_json"],
    ["condition", "id", "json"],
)

CONFLICT_COLUMN = first_existing_column(
    schema_columns,
    ["aggregate_conflict_flag"],
    ["aggregate", "conflict", "flag"],
)

required_resolved_columns = {
    "RCV_COLUMN": RCV_COLUMN,
    "VARIATION_ID_COLUMN": VARIATION_ID_COLUMN,
    "VCV_COLUMN": VCV_COLUMN,
    "GENE_COLUMN": GENE_COLUMN,
    "STUDY_SCOPE_COLUMN": STUDY_SCOPE_COLUMN,
    "TIMEPOINT_COLUMN": TIMEPOINT_COLUMN,
    "RELEASE_LABEL_COLUMN": RELEASE_LABEL_COLUMN,
    "CUTOFF_COLUMN": CUTOFF_COLUMN,
    "SCV_RECORDS_COLUMN": SCV_RECORDS_COLUMN,
    "SCV_COUNT_COLUMN": SCV_COUNT_COLUMN,
    "CONDITION_IDS_COLUMN": CONDITION_IDS_COLUMN,
    "CONFLICT_COLUMN": CONFLICT_COLUMN,
}

unresolved_required_columns = [
    label
    for label, column in required_resolved_columns.items()
    if column is None
]

if unresolved_required_columns:
    raise KeyError(
        "Required T1 source fields could not be resolved:\n- "
        + "\n- ".join(unresolved_required_columns)
    )


# --------------------------------------------------------------------------------------------------
# 10. RCV, IDENTIFIER, TIMEPOINT, AND CUTOFF VALIDATION
# --------------------------------------------------------------------------------------------------

normalized_rcv = (
    t1[RCV_COLUMN]
    .astype("string")
    .str.strip()
    .str.upper()
)

missing_rcv_count = int(
    normalized_rcv.isna().sum()
)

blank_rcv_count = int(
    normalized_rcv.fillna("").eq("").sum()
)

duplicate_rcv_count = int(
    normalized_rcv.duplicated(
        keep=False
    ).sum()
)

malformed_rcv_count = int(
    (
        ~normalized_rcv
        .fillna("")
        .str.fullmatch(
            r"RCV\d+",
            na=False,
        )
    ).sum()
)

unique_rcv_count = int(
    normalized_rcv.nunique(
        dropna=False
    )
)

missing_variation_id_count = int(
    t1[VARIATION_ID_COLUMN].isna().sum()
)

missing_vcv_count = int(
    t1[VCV_COLUMN].isna().sum()
)

observed_timepoints = sorted(
    t1[TIMEPOINT_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

observed_release_labels = sorted(
    t1[RELEASE_LABEL_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

observed_cutoffs = sorted(
    t1[CUTOFF_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)


# --------------------------------------------------------------------------------------------------
# 11. GENE AND STUDY-SCOPE VALIDATION
# --------------------------------------------------------------------------------------------------

allowed_genes = set(
    EXPECTED_GENE_COUNTS
)

gene_parse_errors = 0
gene_structure_errors = 0
normalized_genes = []

for value in t1[GENE_COLUMN]:
    try:
        parsed = parse_json_value(
            value
        )
    except Exception:
        gene_parse_errors += 1
        normalized_genes.append(
            None
        )
        continue

    if not isinstance(parsed, list):
        gene_structure_errors += 1
        normalized_genes.append(
            None
        )
        continue

    genes = sorted({
        str(gene).strip().upper()
        for gene in parsed
        if str(gene).strip()
    })

    if (
        len(genes) != 1
        or genes[0] not in allowed_genes
    ):
        gene_structure_errors += 1
        normalized_genes.append(
            None
        )
        continue

    normalized_genes.append(
        genes[0]
    )

normalized_gene_series = pd.Series(
    normalized_genes,
    index=t1.index,
    dtype="string",
)

observed_gene_counts = {
    str(key): int(value)
    for key, value in (
        normalized_gene_series
        .value_counts(
            dropna=False
        )
        .sort_index()
        .items()
    )
    if not pd.isna(key)
}

observed_study_scope_counts = {
    str(key): int(value)
    for key, value in (
        t1[STUDY_SCOPE_COLUMN]
        .astype("string")
        .str.strip()
        .str.lower()
        .value_counts(
            dropna=False
        )
        .sort_index()
        .items()
    )
    if not pd.isna(key)
}


# --------------------------------------------------------------------------------------------------
# 12. IDENTIFY AND VALIDATE THE SELECTED CLASSIFICATION-AXIS COLUMN
# --------------------------------------------------------------------------------------------------

axis_candidate_columns = [
    column
    for column in schema_columns
    if "axis" in column.lower()
]

selected_axis_column = None

for column in axis_candidate_columns:
    values = (
        t1[column]
        .dropna()
        .astype(str)
        .str.strip()
    )

    observed_counts = {
        str(key): int(value)
        for key, value in (
            values
            .value_counts()
            .sort_index()
            .items()
        )
    }

    if observed_counts == EXPECTED_AXIS_COUNTS:
        selected_axis_column = column
        break

if selected_axis_column is None:
    raise RuntimeError(
        "The selected T1 classification-axis column could not be "
        "identified by its frozen value counts.\n"
        f"Axis candidates found: {axis_candidate_columns}"
    )

classification_axis_counts = {
    str(key): int(value)
    for key, value in (
        t1[selected_axis_column]
        .astype("string")
        .str.strip()
        .value_counts()
        .sort_index()
        .items()
    )
}

axis_provenance_columns = [
    column
    for column in axis_candidate_columns
    if column != selected_axis_column
]


# --------------------------------------------------------------------------------------------------
# 13. JSON-FIELD VALIDATION AND NESTED-SCV KEY INVENTORY
# --------------------------------------------------------------------------------------------------

json_columns = [
    column
    for column in schema_columns
    if column.lower().endswith("_json")
]

json_validation_summary = {}
json_parse_error_total = 0
json_type_error_total = 0

nested_total_records = 0
nested_non_dictionary_records = 0
nested_key_presence = Counter()
nested_key_nonnull = Counter()
nested_key_type_counts = defaultdict(
    Counter
)

condition_id_empty_count = 0
condition_id_parse_errors = 0

for json_column in json_columns:
    parse_errors = 0
    list_count = 0
    dictionary_count = 0
    null_count = 0
    other_type_count = 0

    for value in t1[json_column]:
        try:
            parsed = parse_json_value(
                value
            )
        except Exception:
            parse_errors += 1
            continue

        if parsed is None:
            null_count += 1

        elif isinstance(parsed, list):
            list_count += 1

        elif isinstance(parsed, dict):
            dictionary_count += 1

        else:
            other_type_count += 1

        if json_column == CONDITION_IDS_COLUMN:
            if isinstance(parsed, list):
                if len(parsed) == 0:
                    condition_id_empty_count += 1
            else:
                condition_id_parse_errors += 1

        if json_column == SCV_RECORDS_COLUMN:
            if not isinstance(parsed, list):
                continue

            nested_total_records += len(
                parsed
            )

            for nested_record in parsed:
                if not isinstance(
                    nested_record,
                    dict,
                ):
                    nested_non_dictionary_records += 1
                    continue

                for key, nested_value in nested_record.items():
                    key = str(key)

                    nested_key_presence[key] += 1

                    if nested_value is not None:
                        nested_key_nonnull[key] += 1

                    nested_key_type_counts[key][
                        type(nested_value).__name__
                    ] += 1

    json_parse_error_total += parse_errors
    json_type_error_total += other_type_count

    json_validation_summary[json_column] = {
        "rows": int(len(t1)),
        "parse_errors": int(parse_errors),
        "list_values": int(list_count),
        "dictionary_values": int(dictionary_count),
        "null_values": int(null_count),
        "other_top_level_types": int(other_type_count),
    }


# --------------------------------------------------------------------------------------------------
# 14. RECONCILE SAVED AND PARSED SCV COUNTS
# --------------------------------------------------------------------------------------------------

numeric_scv_counts = pd.to_numeric(
    t1[SCV_COUNT_COLUMN],
    errors="raise",
)

saved_scv_count_total = int(
    numeric_scv_counts.sum()
)

scv_count_json_mismatch_count = 0

for saved_count, value in zip(
    numeric_scv_counts.tolist(),
    t1[SCV_RECORDS_COLUMN].tolist(),
):
    parsed = parse_json_value(
        value
    )

    if (
        not isinstance(parsed, list)
        or int(saved_count) != len(parsed)
    ):
        scv_count_json_mismatch_count += 1


# --------------------------------------------------------------------------------------------------
# 15. CONFLICT AND ACCEPTED CONDITION-ID MISSINGNESS ACCOUNTING
# --------------------------------------------------------------------------------------------------

conflict_values = (
    t1[CONFLICT_COLUMN]
    .astype("boolean")
)

observed_conflict_positive = int(
    conflict_values.fillna(False).sum()
)

missing_conflict_values = int(
    conflict_values.isna().sum()
)


# --------------------------------------------------------------------------------------------------
# 16. TOP-LEVEL SCHEMA INVENTORY
# --------------------------------------------------------------------------------------------------

def infer_top_level_role(
    column_name: str,
) -> str:
    name = column_name.lower()

    if "source" in name or "release" in name or "cutoff" in name or "timepoint" in name:
        return "provenance"

    if "rcv" in name or "vcv" in name or "variation_id" in name:
        return "identifier"

    if "gene" in name or "scope" in name:
        return "gene_and_scope"

    if "condition" in name or "trait" in name:
        return "condition"

    if "classification" in name or "review" in name or "axis" in name:
        return "interpretation"

    if "conflict" in name or "disagreement" in name:
        return "conflict_and_disagreement"

    if "submitter" in name or "scv" in name:
        return "submitted_evidence"

    if "measure" in name:
        return "variant_structure"

    return "other"


schema_inventory_rows = []

for ordinal, column in enumerate(
    schema_columns,
):
    series = t1[column]

    null_count = int(
        series.isna().sum()
    )

    blank_string_count = 0

    if (
        pd.api.types.is_object_dtype(series.dtype)
        or pd.api.types.is_string_dtype(series.dtype)
    ):
        blank_string_count = int(
            series.astype("string")
            .fillna("")
            .str.strip()
            .eq("")
            .sum()
        )

    try:
        unique_count = int(
            series.nunique(
                dropna=True
            )
        )
    except Exception:
        unique_count = None

    schema_inventory_rows.append({
        "ordinal_position": int(ordinal),
        "column_name": column,
        "parquet_type": str(
            arrow_schema.field(column).type
        ),
        "pandas_dtype": str(
            series.dtype
        ),
        "row_count": int(len(series)),
        "non_null_count": int(
            len(series) - null_count
        ),
        "null_count": null_count,
        "blank_string_count": blank_string_count,
        "unique_non_null_count": unique_count,
        "json_serialized": bool(
            column in json_columns
        ),
        "inferred_role": infer_top_level_role(
            column
        ),
        "selected_classification_axis_field": bool(
            column == selected_axis_column
        ),
    })

top_level_schema_inventory = (
    pd.DataFrame(
        schema_inventory_rows
    )
    .sort_values(
        "ordinal_position",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 17. NESTED-SCV KEY INVENTORY
# --------------------------------------------------------------------------------------------------

nested_schema_rows = []

for key in sorted(
    nested_key_presence
):
    type_counts = {
        str(type_name): int(count)
        for type_name, count in sorted(
            nested_key_type_counts[key].items()
        )
    }

    present_count = int(
        nested_key_presence[key]
    )

    nonnull_count = int(
        nested_key_nonnull[key]
    )

    nested_schema_rows.append({
        "nested_key": key,
        "total_nested_scv_records": int(
            nested_total_records
        ),
        "records_containing_key": present_count,
        "records_missing_key": int(
            nested_total_records - present_count
        ),
        "nonnull_value_count": nonnull_count,
        "null_value_count_when_present": int(
            present_count - nonnull_count
        ),
        "presence_fraction": (
            float(
                present_count
                / nested_total_records
            )
            if nested_total_records
            else None
        ),
        "nonnull_fraction": (
            float(
                nonnull_count
                / nested_total_records
            )
            if nested_total_records
            else None
        ),
        "observed_python_types_json": json.dumps(
            type_counts,
            sort_keys=True,
            separators=(",", ":"),
        ),
    })

nested_scv_schema_inventory = (
    pd.DataFrame(
        nested_schema_rows
    )
    .sort_values(
        "nested_key",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 18. EVIDENCE-PACKET DERIVABILITY AUDIT
# --------------------------------------------------------------------------------------------------

def derivability_row(
    packet_field,
    source_columns,
    derivability_status,
    planned_operation,
    scientific_note,
):
    source_columns = list(
        source_columns
    )

    missing_source_columns = [
        column
        for column in source_columns
        if column not in schema_columns
    ]

    return {
        "packet_field": packet_field,
        "source_columns": "|".join(
            source_columns
        ),
        "all_source_columns_present": (
            len(missing_source_columns) == 0
        ),
        "missing_source_columns": "|".join(
            missing_source_columns
        ),
        "derivability_status": derivability_status,
        "planned_operation": planned_operation,
        "scientific_note": scientific_note,
    }


derivability_rows = [
    derivability_row(
        "evidence_packet_id",
        [RCV_COLUMN],
        "DETERMINISTICALLY_DERIVABLE",
        "Create a versioned identifier from the frozen RCV accession.",
        "The RCV remains the primary variant-condition unit.",
    ),
    derivability_row(
        "gene",
        [GENE_COLUMN],
        "DIRECT_FROM_T1_JSON",
        "Read the single exact target gene from target_genes_json.",
        "BRCA1, BRCA2, and MLH1 are primary; EGFR remains exploratory.",
    ),
    derivability_row(
        "variation_id",
        [VARIATION_ID_COLUMN],
        "DIRECT_FROM_T1",
        "Copy the frozen T1 VariationID.",
        "Variant identity must not replace the RCV-level condition unit.",
    ),
    derivability_row(
        "vcv_accession",
        [VCV_COLUMN],
        "DIRECT_FROM_T1",
        "Copy the frozen T1 VCV accession.",
        "Used as variant-level provenance and continuity metadata.",
    ),
    derivability_row(
        "rcv_accession",
        [RCV_COLUMN],
        "DIRECT_FROM_T1",
        "Copy the frozen T1 RCV accession.",
        "Primary evidence-packet key.",
    ),
    derivability_row(
        "condition_names",
        ["condition_names_json"],
        "DIRECT_FROM_T1_JSON",
        "Parse the frozen condition-name list.",
        "Names remain available even when structured condition identifiers are absent.",
    ),
    derivability_row(
        "condition_identifiers",
        [CONDITION_IDS_COLUMN],
        "DIRECT_FROM_T1_JSON_WITH_ACCEPTED_MISSINGNESS",
        "Parse the frozen condition-ID list without imputation.",
        "Exactly 730 records are expected to have empty structured condition identifiers.",
    ),
    derivability_row(
        "structured_trait_records",
        ["trait_records_json"],
        "DIRECT_FROM_T1_JSON",
        "Parse structured trait records.",
        "Preserves condition evidence beyond flattened names and identifiers.",
    ),
    derivability_row(
        "aggregate_classification",
        ["aggregate_classification"],
        "DIRECT_FROM_T1",
        "Copy the selected-axis aggregate classification.",
        "Classification axis must accompany the value.",
    ),
    derivability_row(
        "aggregate_classification_group",
        ["aggregate_classification_group"],
        "DIRECT_FROM_T1",
        "Copy the normalized aggregate classification group.",
        "NoClassification and non-germline axes must remain explicit.",
    ),
    derivability_row(
        "classification_axis",
        [selected_axis_column],
        "DIRECT_FROM_T1",
        "Copy the selected current-schema classification axis.",
        "Prevents germline, oncogenicity, somatic-impact, and no-classification states from being mixed.",
    ),
    derivability_row(
        "classification_axis_provenance",
        axis_provenance_columns,
        (
            "DIRECT_FROM_T1"
            if axis_provenance_columns
            else "AVAILABLE_THROUGH_SELECTED_AXIS_ONLY"
        ),
        "Retain available classification-axis provenance fields.",
        "Current ClinVar may contain more than one classification container.",
    ),
    derivability_row(
        "review_status",
        ["aggregate_review_status"],
        "DIRECT_FROM_T1",
        "Copy the frozen aggregate review status.",
        "Review status is evidence metadata, not an independent clinical outcome.",
    ),
    derivability_row(
        "review_stars",
        ["aggregate_review_stars"],
        "DIRECT_FROM_T1",
        "Copy the frozen aggregate star level.",
        "Required for the mandatory star-aware RAG comparator.",
    ),
    derivability_row(
        "aggregate_conflict",
        [CONFLICT_COLUMN],
        "DIRECT_FROM_T1",
        "Copy the authoritative aggregate conflict flag.",
        "Conflict disclosure is required in the primary RAG endpoint.",
    ),
    derivability_row(
        "scv_group_disagreement",
        ["scv_group_disagreement_flag"],
        "DIRECT_FROM_T1",
        "Copy the nested-submission disagreement indicator.",
        "This is related to but distinct from aggregate ClinVar conflict.",
    ),
    derivability_row(
        "last_evaluated",
        ["aggregate_last_evaluated"],
        "DIRECT_FROM_T1_WITH_MISSINGNESS",
        "Copy the source value without imputing unavailable dates.",
        "Date missingness must be preserved explicitly.",
    ),
    derivability_row(
        "submitter_count",
        ["unique_submitter_count_xml"],
        "DIRECT_FROM_T1",
        "Copy the unique submitter count.",
        "Used for descriptive evidence strength and later frozen feature reconstruction.",
    ),
    derivability_row(
        "submitter_names",
        ["submitter_names_json"],
        "DIRECT_FROM_T1_JSON",
        "Parse the frozen submitter-name list.",
        "Do not infer organization identity from names.",
    ),
    derivability_row(
        "submitter_identifiers",
        ["submitter_ids_json"],
        "DIRECT_FROM_T1_JSON",
        "Parse the frozen primary organization-ID list.",
        "Primary OrgID availability was validated during Stage 2.",
    ),
    derivability_row(
        "nested_scv_assertions",
        [SCV_RECORDS_COLUMN],
        "DIRECT_FROM_T1_JSON",
        "Parse the complete nested SCV list.",
        "Nested field-level missingness must remain visible in later evidence formatting.",
    ),
    derivability_row(
        "scv_classification_counts",
        ["scv_classification_counts_json"],
        "DIRECT_FROM_T1_JSON",
        "Parse saved submitted-classification counts.",
        "Useful for evidence-distribution summaries.",
    ),
    derivability_row(
        "scv_classification_group_counts",
        ["scv_classification_group_counts_json"],
        "DIRECT_FROM_T1_JSON",
        "Parse saved normalized classification-group counts.",
        "Useful for disagreement and entropy reconstruction.",
    ),
    derivability_row(
        "source_release",
        [RELEASE_LABEL_COLUMN],
        "DIRECT_FROM_T1",
        "Copy the frozen January 2026 release label.",
        "The archive label must remain separate from the embedded cutoff.",
    ),
    derivability_row(
        "source_cutoff",
        [CUTOFF_COLUMN],
        "DIRECT_FROM_T1",
        "Copy the frozen embedded cutoff date.",
        "Expected biological data boundary is 2025-12-27.",
    ),
    derivability_row(
        "source_filename",
        ["source_filename"],
        "DIRECT_FROM_T1",
        "Copy the immutable source filename.",
        "Supports record-level provenance.",
    ),
    derivability_row(
        "source_sha256",
        ["source_sha256"],
        "DIRECT_FROM_T1",
        "Copy the immutable raw-source SHA-256.",
        "Supports source-release provenance.",
    ),
    derivability_row(
        "semantic_evidence_text",
        [
            GENE_COLUMN,
            RCV_COLUMN,
            "condition_names_json",
            "aggregate_classification",
            "aggregate_review_status",
            CONFLICT_COLUMN,
            SCV_RECORDS_COLUMN,
        ],
        "DETERMINISTICALLY_DERIVABLE_LATER",
        "Create versioned evidence text only after a corpus-formatting policy is frozen.",
        "Cell 7A1 does not construct or tokenize evidence text.",
    ),
    derivability_row(
        "ges_stability_index",
        [],
        "REQUIRES_FROZEN_MODEL_APPLICATION",
        "Apply the checksum-verified Stage 4C feature and model specification to T1 in a later authorized cell.",
        "Must remain a relative warning/prioritization index rather than a calibrated probability.",
    ),
    derivability_row(
        "ges_instability_risk_rank",
        [],
        "REQUIRES_GES_SCORING",
        "Create rank-based risk metadata only after T1 GES scoring is frozen.",
        "The inherited 0.50 probability threshold is prohibited.",
    ),
    derivability_row(
        "no_star_ges_index",
        [],
        "REQUIRES_FROZEN_MODEL_APPLICATION",
        "Apply the checksum-verified no-star Stage 4C pipeline to T1.",
        "Required for the prespecified no-star ablation.",
    ),
    derivability_row(
        "combined_metadata_risk",
        [],
        "REQUIRES_FROZEN_POLICY_APPLICATION",
        "Apply the checksum-verified Stage 6A combined-metadata policy to T1.",
        "This is a mandatory strong comparator because it matched or exceeded Full GES on primary temporal AUPRC.",
    ),
]

evidence_packet_derivability = (
    pd.DataFrame(
        derivability_rows
    )
    .sort_values(
        "packet_field",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 19. PRESERVE ONE CREATION TIMESTAMP ACROSS SAFE RERUNS
# --------------------------------------------------------------------------------------------------

existing_created_utc = None

for existing_json_path in [
    MANIFEST_PATH,
    QC_PATH,
    PREFLIGHT_REPORT_PATH,
]:
    if existing_json_path.exists():
        try:
            existing_object = json.loads(
                existing_json_path.read_text(
                    encoding="utf-8"
                )
            )

            existing_created_utc = (
                existing_object.get("created_utc")
            )

            if existing_created_utc:
                break

        except Exception:
            pass

CREATED_UTC = (
    existing_created_utc
    if existing_created_utc
    else datetime.now(
        timezone.utc
    ).isoformat()
)


# --------------------------------------------------------------------------------------------------
# 20. WRITE THE THREE VERSIONED INVENTORY TABLES
# --------------------------------------------------------------------------------------------------

top_level_schema_sha256 = stable_write_dataframe_csv(
    TOP_LEVEL_SCHEMA_CSV,
    top_level_schema_inventory,
)

write_sha256_sidecar(
    TOP_LEVEL_SCHEMA_CSV
)

nested_scv_schema_sha256 = stable_write_dataframe_csv(
    NESTED_SCV_SCHEMA_CSV,
    nested_scv_schema_inventory,
)

write_sha256_sidecar(
    NESTED_SCV_SCHEMA_CSV
)

derivability_sha256 = stable_write_dataframe_csv(
    DERIVABILITY_CSV,
    evidence_packet_derivability,
)

write_sha256_sidecar(
    DERIVABILITY_CSV
)


# --------------------------------------------------------------------------------------------------
# 21. CREATE THE CELL 7A1 PREFLIGHT REPORT
# --------------------------------------------------------------------------------------------------

derivability_status_counts = {
    str(key): int(value)
    for key, value in (
        evidence_packet_derivability[
            "derivability_status"
        ]
        .value_counts()
        .sort_index()
        .items()
    )
}

preflight_report = {
    "report_name": (
        "Cell 7A1 T1 corpus-source verification and schema inventory"
    ),
    "report_version": "1.0.0",
    "created_utc": CREATED_UTC,
    "notebook_filename": NOTEBOOK_FILENAME,
    "cell_id": CELL_ID,

    "authorization": {
        "protocol_path": str(
            PROTOCOL_PATH
        ),
        "protocol_sha256": observed_protocol_sha256,
        "authorization_path": str(
            AUTHORIZATION_PATH
        ),
        "authorization_sha256": observed_authorization_sha256,
        "authorization_decision": (
            authorization_record[
                "terminal_decision"
            ]
        ),
        "llm_generation_authorized": False,
        "embedding_construction_authorized": False,
    },

    "parent_stage6c_boundary": {
        "path": str(
            parent_stage6c_path
        ),
        "sha256": parent_stage6c_observed_sha256,
        "sidecar_verified": True,
    },

    "t1_source_package": {
        "parquet_path": str(
            T1_PARQUET
        ),
        "parquet_sha256": observed_t1_parquet_sha256,
        "parquet_sidecar_present": sidecar_path(
            T1_PARQUET
        ).exists(),
        "validation_report_path": str(
            T1_VALIDATION_REPORT
        ),
        "validation_report_sha256": observed_t1_validation_sha256,
        "freeze_manifest_path": str(
            T1_FREEZE_MANIFEST
        ),
        "freeze_manifest_sha256": observed_t1_manifest_sha256,
        "freeze_manifest_sidecar_verified": True,
        "administrative_decision": "ACCEPTED_AND_FROZEN",
    },

    "parquet_structure": {
        "rows": int(
            parquet_metadata.num_rows
        ),
        "columns": int(
            parquet_metadata.num_columns
        ),
        "row_groups": int(
            parquet_metadata.num_row_groups
        ),
        "schema_columns": schema_columns,
    },

    "resolved_columns": required_resolved_columns,

    "identifier_accounting": {
        "unique_rcv_count": unique_rcv_count,
        "missing_rcv_count": missing_rcv_count,
        "blank_rcv_count": blank_rcv_count,
        "duplicate_rcv_record_count": duplicate_rcv_count,
        "malformed_rcv_count": malformed_rcv_count,
        "missing_variation_id_count": missing_variation_id_count,
        "missing_vcv_count": missing_vcv_count,
    },

    "provenance_accounting": {
        "observed_timepoints": observed_timepoints,
        "observed_release_labels": observed_release_labels,
        "observed_embedded_cutoffs": observed_cutoffs,
    },

    "gene_and_scope_accounting": {
        "gene_parse_errors": gene_parse_errors,
        "gene_structure_errors": gene_structure_errors,
        "observed_gene_counts": observed_gene_counts,
        "expected_gene_counts": EXPECTED_GENE_COUNTS,
        "observed_study_scope_counts": observed_study_scope_counts,
        "expected_study_scope_counts": EXPECTED_STUDY_SCOPE_COUNTS,
    },

    "classification_axis_accounting": {
        "selected_axis_column": selected_axis_column,
        "axis_provenance_columns": axis_provenance_columns,
        "observed_axis_counts": classification_axis_counts,
        "expected_axis_counts": EXPECTED_AXIS_COUNTS,
    },

    "nested_scv_accounting": {
        "saved_scv_count_total": saved_scv_count_total,
        "parsed_nested_scv_total": nested_total_records,
        "expected_nested_scv_total": EXPECTED_T1_NESTED_SCVS,
        "scv_count_json_mismatch_count": scv_count_json_mismatch_count,
        "nested_non_dictionary_record_count": nested_non_dictionary_records,
        "nested_unique_key_count": int(
            len(nested_key_presence)
        ),
    },

    "json_accounting": {
        "json_columns": json_columns,
        "json_column_count": int(
            len(json_columns)
        ),
        "total_json_parse_errors": int(
            json_parse_error_total
        ),
        "total_other_top_level_type_values": int(
            json_type_error_total
        ),
        "by_column": json_validation_summary,
    },

    "conflict_and_condition_accounting": {
        "aggregate_conflict_positive": observed_conflict_positive,
        "expected_aggregate_conflict_positive": EXPECTED_T1_CONFLICT_POSITIVE,
        "missing_conflict_values": missing_conflict_values,
        "empty_structured_condition_id_rows": condition_id_empty_count,
        "expected_empty_structured_condition_id_rows": EXPECTED_T1_EMPTY_CONDITION_IDS,
        "condition_id_parse_or_type_errors": condition_id_parse_errors,
    },

    "inventory_artifacts": {
        "top_level_schema_csv": str(
            TOP_LEVEL_SCHEMA_CSV
        ),
        "top_level_schema_sha256": top_level_schema_sha256,
        "nested_scv_schema_csv": str(
            NESTED_SCV_SCHEMA_CSV
        ),
        "nested_scv_schema_sha256": nested_scv_schema_sha256,
        "evidence_packet_derivability_csv": str(
            DERIVABILITY_CSV
        ),
        "evidence_packet_derivability_sha256": derivability_sha256,
    },

    "derivability_summary": {
        "planned_packet_field_count": int(
            len(evidence_packet_derivability)
        ),
        "status_counts": derivability_status_counts,
        "corpus_constructed": False,
        "ges_score_applied": False,
        "combined_metadata_score_applied": False,
        "embeddings_created": False,
        "questions_created": False,
        "llm_called": False,
        "rag_performance_examined": False,
    },

    "next_authorized_cell": {
        "cell_id": "7A2",
        "name": (
            "Frozen T1 feature-reconstruction and model-applicability preflight"
        ),
        "authorized_actions": [
            "Verify Stage 4A T0 feature definitions and frozen feature schema.",
            "Verify Stage 4B transformations and weak-label specifications without creating new labels.",
            "Verify the Stage 4C Full GES and no-star model packages.",
            "Verify the Stage 6A combined-metadata comparator policy.",
            "Map every required frozen feature to the accepted T1 source.",
            "Quantify T1 feature missingness and out-of-range values.",
            "Determine whether the frozen pipelines can be applied to T1 without refitting.",
            "Produce a read-only model-applicability preflight package.",
        ],
        "prohibited_actions": [
            "Do not refit or recalibrate either GES model.",
            "Do not optimize thresholds or weights.",
            "Do not use the inherited P(stable)=0.50 threshold.",
            "Do not construct final T1 GES scores in Cell 7A2.",
            "Do not create embeddings.",
            "Do not call an LLM.",
            "Do not create development or locked-test questions.",
            "Do not inspect RAG performance.",
        ],
    },

    "decision": (
        "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_SCHEMA_AND_"
        "NESTED_EVIDENCE_INVENTORIED_DERIVABILITY_AUDITED"
    ),
}

preflight_report_sha256 = stable_write_json(
    PREFLIGHT_REPORT_PATH,
    preflight_report,
)

write_sha256_sidecar(
    PREFLIGHT_REPORT_PATH
)


# --------------------------------------------------------------------------------------------------
# 22. CELL 7A1 QUALITY-CONTROL CHECKS
# --------------------------------------------------------------------------------------------------

qc_checks = {
    "experiment2_protocol_hash_matches": (
        observed_protocol_sha256
        == EXPECTED_PROTOCOL_SHA256
    ),
    "experiment2_protocol_sidecar_valid": sidecar_is_valid(
        PROTOCOL_PATH,
        required=True,
    ),
    "experiment2_authorization_hash_matches": (
        observed_authorization_sha256
        == EXPECTED_AUTHORIZATION_SHA256
    ),
    "experiment2_authorization_sidecar_valid": sidecar_is_valid(
        AUTHORIZATION_PATH,
        required=True,
    ),
    "stage7a1_was_explicitly_authorized": (
        authorization_record.get(
            "next_authorized_cell"
        )
        == "7A1"
    ),
    "llm_generation_remains_unauthorized": (
        authorization_record[
            "authorization_status"
        ]["llm_generation_authorized"]
        is False
    ),
    "embedding_construction_remains_unauthorized": (
        authorization_record[
            "authorization_status"
        ]["embedding_construction_authorized"]
        is False
    ),
    "parent_stage6c_manifest_reverified": (
        parent_stage6c_observed_sha256
        == parent_stage6c_expected_sha256
    ),
    "parent_stage6c_sidecar_valid": sidecar_is_valid(
        parent_stage6c_path,
        required=True,
    ),
    "t1_parquet_exists": T1_PARQUET.exists(),
    "t1_parquet_hash_matches": (
        observed_t1_parquet_sha256
        == EXPECTED_T1_PARQUET_SHA256
    ),
    "t1_freeze_manifest_hash_matches": (
        observed_t1_manifest_sha256
        == EXPECTED_T1_MANIFEST_SHA256
    ),
    "t1_freeze_manifest_sidecar_valid": sidecar_is_valid(
        T1_FREEZE_MANIFEST,
        required=True,
    ),
    "t1_administrative_decision_accepted_and_frozen": (
        recursively_contains_scalar(
            t1_freeze_manifest,
            "ACCEPTED_AND_FROZEN",
        )
    ),
    "t1_row_count_matches": (
        len(t1)
        == EXPECTED_T1_ROWS
    ),
    "t1_column_count_matches": (
        len(t1.columns)
        == EXPECTED_T1_COLUMNS
    ),
    "t1_rcv_keys_complete": (
        missing_rcv_count == 0
        and blank_rcv_count == 0
    ),
    "t1_rcv_keys_unique": (
        unique_rcv_count
        == EXPECTED_T1_ROWS
        and duplicate_rcv_count == 0
    ),
    "t1_rcv_format_valid": (
        malformed_rcv_count == 0
    ),
    "t1_variation_ids_complete": (
        missing_variation_id_count == 0
    ),
    "t1_vcv_accessions_complete": (
        missing_vcv_count == 0
    ),
    "t1_timepoint_is_t1_only": (
        observed_timepoints == ["T1"]
    ),
    "t1_embedded_cutoff_matches": (
        observed_cutoffs
        == [EXPECTED_T1_CUTOFF]
    ),
    "t1_gene_json_parses": (
        gene_parse_errors == 0
    ),
    "t1_gene_structure_valid": (
        gene_structure_errors == 0
    ),
    "t1_gene_counts_match": (
        observed_gene_counts
        == EXPECTED_GENE_COUNTS
    ),
    "t1_study_scope_counts_match": (
        observed_study_scope_counts
        == EXPECTED_STUDY_SCOPE_COUNTS
    ),
    "t1_selected_axis_identified": (
        selected_axis_column
        is not None
    ),
    "t1_axis_counts_match": (
        classification_axis_counts
        == EXPECTED_AXIS_COUNTS
    ),
    "all_json_values_parse": (
        json_parse_error_total == 0
    ),
    "all_json_top_level_types_supported": (
        json_type_error_total == 0
    ),
    "saved_scv_total_matches": (
        saved_scv_count_total
        == EXPECTED_T1_NESTED_SCVS
    ),
    "parsed_scv_total_matches": (
        nested_total_records
        == EXPECTED_T1_NESTED_SCVS
    ),
    "saved_and_parsed_scv_counts_reconcile": (
        scv_count_json_mismatch_count
        == 0
    ),
    "nested_scv_records_are_dictionaries": (
        nested_non_dictionary_records
        == 0
    ),
    "aggregate_conflict_count_matches": (
        observed_conflict_positive
        == EXPECTED_T1_CONFLICT_POSITIVE
    ),
    "aggregate_conflict_values_complete": (
        missing_conflict_values == 0
    ),
    "accepted_condition_id_missingness_matches": (
        condition_id_empty_count
        == EXPECTED_T1_EMPTY_CONDITION_IDS
    ),
    "condition_id_json_type_valid": (
        condition_id_parse_errors == 0
    ),
    "top_level_schema_inventory_written": (
        TOP_LEVEL_SCHEMA_CSV.exists()
        and sidecar_is_valid(
            TOP_LEVEL_SCHEMA_CSV,
            required=True,
        )
    ),
    "nested_scv_schema_inventory_written": (
        NESTED_SCV_SCHEMA_CSV.exists()
        and sidecar_is_valid(
            NESTED_SCV_SCHEMA_CSV,
            required=True,
        )
    ),
    "derivability_inventory_written": (
        DERIVABILITY_CSV.exists()
        and sidecar_is_valid(
            DERIVABILITY_CSV,
            required=True,
        )
    ),
    "preflight_report_written": (
        PREFLIGHT_REPORT_PATH.exists()
        and sidecar_is_valid(
            PREFLIGHT_REPORT_PATH,
            required=True,
        )
    ),
    "corpus_not_constructed": (
        preflight_report[
            "derivability_summary"
        ]["corpus_constructed"]
        is False
    ),
    "ges_not_applied": (
        preflight_report[
            "derivability_summary"
        ]["ges_score_applied"]
        is False
    ),
    "combined_metadata_not_applied": (
        preflight_report[
            "derivability_summary"
        ]["combined_metadata_score_applied"]
        is False
    ),
    "llm_not_called": (
        preflight_report[
            "derivability_summary"
        ]["llm_called"]
        is False
    ),
    "next_authorized_cell_is_7a2": (
        preflight_report[
            "next_authorized_cell"
        ]["cell_id"]
        == "7A2"
    ),
}

failed_checks = [
    check_name
    for check_name, passed in qc_checks.items()
    if passed is not True
]

qc_record = {
    "qc_name": (
        "Cell 7A1 T1 corpus-source and schema-inventory QC"
    ),
    "qc_version": "1.0.0",
    "created_utc": CREATED_UTC,
    "notebook_filename": NOTEBOOK_FILENAME,
    "cell_id": CELL_ID,
    "checks": qc_checks,
    "checks_total": int(
        len(qc_checks)
    ),
    "checks_passed": int(
        len(qc_checks)
        - len(failed_checks)
    ),
    "checks_failed": int(
        len(failed_checks)
    ),
    "failed_check_names": failed_checks,
    "t1_parquet_sha256": observed_t1_parquet_sha256,
    "t1_freeze_manifest_sha256": observed_t1_manifest_sha256,
    "top_level_schema_sha256": top_level_schema_sha256,
    "nested_scv_schema_sha256": nested_scv_schema_sha256,
    "derivability_sha256": derivability_sha256,
    "preflight_report_sha256": preflight_report_sha256,
    "decision": (
        "PASS"
        if not failed_checks
        else "FAIL"
    ),
}

qc_sha256 = stable_write_json(
    QC_PATH,
    qc_record,
)

write_sha256_sidecar(
    QC_PATH
)

if failed_checks:
    raise RuntimeError(
        "Cell 7A1 QC failed:\n- "
        + "\n- ".join(
            failed_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 23. CREATE THE CELL 7A1 FREEZE MANIFEST
# --------------------------------------------------------------------------------------------------

manifest_record = {
    "manifest_name": (
        "Cell 7A1 T1 corpus-source verification and schema-inventory manifest"
    ),
    "manifest_version": "1.0.0",
    "created_utc": CREATED_UTC,
    "notebook_filename": NOTEBOOK_FILENAME,
    "cell_id": CELL_ID,

    "authorization_lineage": {
        "experiment2_protocol_path": str(
            PROTOCOL_PATH
        ),
        "experiment2_protocol_sha256": observed_protocol_sha256,
        "experiment2_authorization_path": str(
            AUTHORIZATION_PATH
        ),
        "experiment2_authorization_sha256": observed_authorization_sha256,
        "parent_stage6c_manifest_path": str(
            parent_stage6c_path
        ),
        "parent_stage6c_manifest_sha256": parent_stage6c_observed_sha256,
    },

    "immutable_t1_inputs": [
        {
            "role": "accepted_t1_harmonized_parquet",
            "path": str(
                T1_PARQUET
            ),
            "sha256": observed_t1_parquet_sha256,
            "rows": EXPECTED_T1_ROWS,
            "columns": EXPECTED_T1_COLUMNS,
        },
        {
            "role": "t1_field_validation_report",
            "path": str(
                T1_VALIDATION_REPORT
            ),
            "sha256": observed_t1_validation_sha256,
        },
        {
            "role": "t1_extraction_validation_freeze_manifest",
            "path": str(
                T1_FREEZE_MANIFEST
            ),
            "sha256": observed_t1_manifest_sha256,
            "sidecar_verified": True,
        },
    ],

    "cell_7a1_outputs": [
        {
            "role": "top_level_schema_inventory",
            "path": str(
                TOP_LEVEL_SCHEMA_CSV
            ),
            "sha256": top_level_schema_sha256,
            "sidecar_verified": True,
        },
        {
            "role": "nested_scv_key_inventory",
            "path": str(
                NESTED_SCV_SCHEMA_CSV
            ),
            "sha256": nested_scv_schema_sha256,
            "sidecar_verified": True,
        },
        {
            "role": "evidence_packet_derivability_audit",
            "path": str(
                DERIVABILITY_CSV
            ),
            "sha256": derivability_sha256,
            "sidecar_verified": True,
        },
        {
            "role": "corpus_source_preflight_report",
            "path": str(
                PREFLIGHT_REPORT_PATH
            ),
            "sha256": preflight_report_sha256,
            "sidecar_verified": True,
        },
        {
            "role": "cell_7a1_qc",
            "path": str(
                QC_PATH
            ),
            "sha256": qc_sha256,
            "sidecar_verified": True,
        },
    ],

    "verified_accounting": {
        "rows": EXPECTED_T1_ROWS,
        "columns": EXPECTED_T1_COLUMNS,
        "unique_rcv_keys": unique_rcv_count,
        "nested_scv_records": nested_total_records,
        "gene_counts": observed_gene_counts,
        "study_scope_counts": observed_study_scope_counts,
        "classification_axis_column": selected_axis_column,
        "classification_axis_counts": classification_axis_counts,
        "aggregate_conflict_positive": observed_conflict_positive,
        "empty_structured_condition_id_rows": condition_id_empty_count,
        "json_parse_errors": json_parse_error_total,
        "nested_unique_keys": int(
            len(nested_key_presence)
        ),
        "qc_checks_passed": int(
            len(qc_checks)
        ),
        "qc_checks_failed": 0,
    },

    "scientific_boundary": {
        "t1_source_accepted_and_frozen": True,
        "t1_schema_inventory_complete": True,
        "nested_scv_key_inventory_complete": True,
        "evidence_packet_derivability_audited": True,
        "rag_corpus_constructed": False,
        "ges_applied_to_t1": False,
        "combined_metadata_applied_to_t1": False,
        "embeddings_created": False,
        "question_set_created": False,
        "llm_called": False,
        "rag_performance_examined": False,
        "clinical_use_authorized": False,
    },

    "next_authorized_cell": {
        "cell_id": "7A2",
        "name": (
            "Frozen T1 feature-reconstruction and model-applicability preflight"
        ),
        "execution_type": (
            "read_only_preflight_against_frozen_stage4_and_stage6a_artifacts"
        ),
    },

    "terminal_decision": (
        "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_CHECKSUM_PROTECTED_"
        "TOP_LEVEL_AND_NESTED_SCHEMA_INVENTORIED_EVIDENCE_PACKET_"
        "DERIVABILITY_AUDITED_STAGE7A2_PREFLIGHT_ONLY"
    ),
}

manifest_sha256 = stable_write_json(
    MANIFEST_PATH,
    manifest_record,
)

write_sha256_sidecar(
    MANIFEST_PATH
)


# --------------------------------------------------------------------------------------------------
# 24. FRESH READBACK AND CRYPTOGRAPHIC REVERIFICATION
# --------------------------------------------------------------------------------------------------

fresh_preflight = json.loads(
    PREFLIGHT_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

fresh_qc = json.loads(
    QC_PATH.read_text(
        encoding="utf-8"
    )
)

fresh_manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

for output_path in [
    TOP_LEVEL_SCHEMA_CSV,
    NESTED_SCV_SCHEMA_CSV,
    DERIVABILITY_CSV,
    PREFLIGHT_REPORT_PATH,
    QC_PATH,
    MANIFEST_PATH,
]:
    if not sidecar_is_valid(
        output_path,
        required=True,
    ):
        raise RuntimeError(
            "Fresh output sidecar verification failed:\n"
            f"{output_path}"
        )

fresh_top_level_inventory = pd.read_csv(
    TOP_LEVEL_SCHEMA_CSV
)

fresh_nested_inventory = pd.read_csv(
    NESTED_SCV_SCHEMA_CSV
)

fresh_derivability_inventory = pd.read_csv(
    DERIVABILITY_CSV
)

if len(fresh_top_level_inventory) != EXPECTED_T1_COLUMNS:
    raise RuntimeError(
        "Fresh top-level schema inventory row count mismatch."
    )

if len(fresh_nested_inventory) != len(nested_key_presence):
    raise RuntimeError(
        "Fresh nested-SCV schema inventory row count mismatch."
    )

if (
    fresh_qc.get("checks_failed")
    != 0
):
    raise RuntimeError(
        "Fresh Cell 7A1 QC readback reports failed checks."
    )

if (
    fresh_preflight.get("decision")
    != (
        "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_SCHEMA_AND_"
        "NESTED_EVIDENCE_INVENTORIED_DERIVABILITY_AUDITED"
    )
):
    raise RuntimeError(
        "Fresh preflight decision readback mismatch."
    )

if (
    fresh_manifest.get("next_authorized_cell", {}).get("cell_id")
    != "7A2"
):
    raise RuntimeError(
        "Fresh manifest next-cell authorization mismatch."
    )

if (
    fresh_manifest
    .get("scientific_boundary", {})
    .get("llm_called")
    is not False
):
    raise RuntimeError(
        "Fresh manifest incorrectly reports LLM execution."
    )


# --------------------------------------------------------------------------------------------------
# 25. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

separator = "=" * 128

direct_or_derivable_count = int(
    evidence_packet_derivability[
        "derivability_status"
    ]
    .str.startswith(
        (
            "DIRECT",
            "DETERMINISTICALLY",
            "AVAILABLE",
        )
    )
    .sum()
)

requires_later_scoring_count = int(
    evidence_packet_derivability[
        "derivability_status"
    ]
    .str.startswith(
        "REQUIRES"
    )
    .sum()
)

print("\n" + separator)
print("EXPERIMENT 2 — STAGE 7A — CELL 7A1")
print("FROZEN T1 CORPUS-SOURCE VERIFICATION AND SCHEMA INVENTORY")
print(separator)

print(f"Notebook                                      : {NOTEBOOK_FILENAME}")
print(f"Project root                                  : {ROOT}")
print()

print("CELL 7A0 AUTHORIZATION")
print(f"Experiment 2 protocol SHA-256                 : {observed_protocol_sha256}")
print(f"Experiment 2 authorization SHA-256            : {observed_authorization_sha256}")
print("Cell 7A1 explicitly authorized                : YES")
print("LLM generation authorized                     : NO")
print("Embedding construction authorized             : NO")
print()

print("FROZEN T1 SOURCE")
print(f"T1 Parquet                                    : {T1_PARQUET}")
print(f"T1 Parquet SHA-256                            : {observed_t1_parquet_sha256}")
print(f"T1 freeze manifest SHA-256                    : {observed_t1_manifest_sha256}")
print("T1 administrative decision                    : ACCEPTED_AND_FROZEN")
print()

print("T1 STRUCTURAL ACCOUNTING")
print(f"Rows                                           : {len(t1):,}")
print(f"Columns                                        : {len(t1.columns):,}")
print(f"Unique RCV accessions                          : {unique_rcv_count:,}")
print(f"Malformed or duplicate RCV records             : {malformed_rcv_count:,} / {duplicate_rcv_count:,}")
print(f"Embedded data cutoff                           : {observed_cutoffs[0]}")
print()

print("GENE ACCOUNTING")
for gene in ["BRCA1", "BRCA2", "MLH1", "EGFR"]:
    print(
        f"{gene:<46}: "
        f"{observed_gene_counts.get(gene, 0):,}"
    )
print()

print("T1 CLASSIFICATION-AXIS ACCOUNTING")
print(f"Selected-axis field                            : {selected_axis_column}")
for axis in [
    "GermlineClassification",
    "OncogenicityClassification",
    "SomaticClinicalImpact",
    "NoClassification",
]:
    print(
        f"{axis:<46}: "
        f"{classification_axis_counts.get(axis, 0):,}"
    )
print()

print("NESTED EVIDENCE ACCOUNTING")
print(f"Saved SCV total                                : {saved_scv_count_total:,}")
print(f"Parsed nested SCV total                        : {nested_total_records:,}")
print(f"Saved/parsed SCV count mismatches               : {scv_count_json_mismatch_count:,}")
print(f"Unique nested SCV keys                         : {len(nested_key_presence):,}")
print(f"JSON parse errors                              : {json_parse_error_total:,}")
print(f"Non-dictionary nested SCV records              : {nested_non_dictionary_records:,}")
print()

print("ACCEPTED SOURCE CHARACTERISTICS")
print(f"Aggregate conflict-positive RCVs               : {observed_conflict_positive:,}")
print(f"Rows with empty structured condition IDs       : {condition_id_empty_count:,}")
print()

print("EVIDENCE-PACKET DERIVABILITY")
print(f"Planned packet fields audited                  : {len(evidence_packet_derivability):,}")
print(f"Directly available or deterministically derived: {direct_or_derivable_count:,}")
print(f"Require later frozen scoring/policy application: {requires_later_scoring_count:,}")
print("RAG corpus constructed                         : NO")
print("GES applied to T1                              : NO")
print("Combined metadata applied to T1                : NO")
print("LLM called                                     : NO")
print()

print("CELL 7A1 FROZEN OUTPUTS")
print(f"Top-level schema inventory                     : {TOP_LEVEL_SCHEMA_CSV}")
print(f"Top-level schema SHA-256                       : {top_level_schema_sha256}")
print(f"Nested-SCV key inventory                       : {NESTED_SCV_SCHEMA_CSV}")
print(f"Nested-SCV inventory SHA-256                   : {nested_scv_schema_sha256}")
print(f"Evidence-packet derivability audit             : {DERIVABILITY_CSV}")
print(f"Derivability audit SHA-256                     : {derivability_sha256}")
print(f"Preflight report                               : {PREFLIGHT_REPORT_PATH}")
print(f"Preflight report SHA-256                       : {preflight_report_sha256}")
print(f"QC record                                      : {QC_PATH}")
print(f"QC SHA-256                                     : {qc_sha256}")
print(f"Manifest                                       : {MANIFEST_PATH}")
print(f"Manifest SHA-256                               : {manifest_sha256}")
print(f"QC checks                                      : {len(qc_checks)}/{len(qc_checks)} PASS")
print()

print("NEXT AUTHORIZED CELL")
print("Cell 7A2                                      : Frozen T1 feature-reconstruction")
print("                                                  and model-applicability preflight")
print("Full/no-star model fitting                     : PROHIBITED")
print("Threshold or weight optimization               : PROHIBITED")
print("Final T1 scoring in Cell 7A2                   : PROHIBITED")
print()

print(
    "FINAL DECISION                                : "
    "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_CHECKSUM_PROTECTED_"
    "TOP_LEVEL_AND_NESTED_SCHEMA_INVENTORIED_EVIDENCE_PACKET_"
    "DERIVABILITY_AUDITED_STAGE7A2_PREFLIGHT_ONLY"
)
print(separator)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

EXPERIMENT 2 — STAGE 7A — CELL 7A1
FROZEN T1 CORPUS-SOURCE VERIFICATION AND SCHEMA INVENTORY
Notebook                                      : 05_GES_Aware_Genomic_RAG.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

CELL 7A0 AUTHORIZATION
Experiment 2 protocol SHA-256                 : db0fb3f197356118fd069c36e4527730f4bb27ebece709038f51b522cb6aa332
Experiment 2 authorization SHA-256            : 7ea8f5f8a663ddc82a6434e9f08b6da2f3e70981cb4974270f5ec1ed4cf2d91f
Cell 7A1 explicitly authorized                : YES
LLM generation authorized                     : NO
Embedding construction authorized             : NO

FROZEN T1 SOURCE
T1 Parquet                                    : /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t1_rcv_target_genes_harmonized_v1.parquet
T1 Parquet SHA-256               

In [8]:
# ==================================================================================================
# EXPERIMENT 2 — STAGE 7A — CELL 7A2 — CORRECTED COMPLETE CELL
# FROZEN T1 FEATURE RECONSTRUCTION AND MODEL-APPLICABILITY PREFLIGHT
#
# This cell verifies the frozen Stage 4A/4B/4C packages, reconstructs the six T1 model inputs
# in memory, and verifies that the frozen preprocessing pipelines can transform all T1 rows.
#
# It DOES NOT fit/refit a model, generate GES scores, apply thresholds, create embeddings,
# build a RAG corpus, create questions, or call an LLM.
# ==================================================================================================

from collections import OrderedDict, Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import re
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE AND PROJECT PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG.ipynb"
CELL_ID = "7A2"

STAGE4_DATA_DIR = ROOT / "data_processed" / "stage4_ges"
STAGE4_MODEL_DIR = ROOT / "models" / "stage4_ges"
STAGE4_CONFIG_DIR = ROOT / "configs" / "stage4_ges"
STAGE6_CONFIG_DIR = ROOT / "configs" / "stage6_temporal_validation"
STAGE7_CONFIG_DIR = ROOT / "configs" / "stage7_rag"
STAGE7_TABLE_DIR = ROOT / "outputs" / "tables" / "stage7_rag"
STAGE7_QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"

for directory in [STAGE7_CONFIG_DIR, STAGE7_TABLE_DIR, STAGE7_QC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 2. EXACT FROZEN INPUT PATHS
# --------------------------------------------------------------------------------------------------

CELL_7A1_MANIFEST = (
    STAGE7_CONFIG_DIR / "cell_7a1_t1_corpus_source_preflight_manifest_v1.json"
)

T1_PARQUET = ROOT / "data_interim" / "t1_rcv_target_genes_harmonized_v1.parquet"
T1_FREEZE_MANIFEST = (
    ROOT / "configs" / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
)

STAGE4A_FEATURE_TABLE = (
    STAGE4_DATA_DIR / "stage4a_t0_ges_baseline_features_v1.parquet"
)
STAGE4A_FEATURE_SPEC = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_specification_v1.json"
)
STAGE4A_QC = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_qc_report_v1.json"
)
STAGE4A_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_freeze_manifest_v1.json"
)

STAGE4B_TABLE = (
    STAGE4_DATA_DIR / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)
STAGE4B_TRANSFORM_PARAMETERS = (
    STAGE4_CONFIG_DIR / "stage4b_t0_feature_transform_parameters_v1.json"
)
STAGE4B_WEAK_LABEL_RULES = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_rules_v1.json"
)
STAGE4B_QC = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_qc_report_v1.json"
)
STAGE4B_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_freeze_manifest_v1.json"
)

FULL_MODEL_PATH = (
    STAGE4_MODEL_DIR / "stage4c_full_ges_logistic_model_v1.joblib"
)
NO_STAR_MODEL_PATH = (
    STAGE4_MODEL_DIR / "stage4c_no_star_ges_logistic_model_v1.joblib"
)
STAGE4C_MODEL_SPEC = (
    STAGE4_CONFIG_DIR / "stage4c_ges_model_specification_v1.json"
)
STAGE4C_QC = (
    STAGE4_CONFIG_DIR / "stage4c_ges_model_qc_report_v1.json"
)
STAGE4C_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4c_ges_model_freeze_manifest_v1.json"
)

STAGE6A_POLICY = (
    STAGE6_CONFIG_DIR / "stage6a_comparator_score_policy_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 3. EXPECTED HASHES AND LOCKED COUNTS
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = OrderedDict([
    ("cell_7a1_manifest", "84e509d97f01fb8dc0b6ad0c7e24de762e8923b9068fc835bf328105f79e946d"),
    ("t1_parquet", "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"),
    ("t1_freeze_manifest", "7eaeff0fee3df96973130f721d6c1f2a02fd9108e85af7b2743cdd7750a4372e"),
    ("stage4a_feature_table", "c100b3781e6801425f622f5d091376abfe0939e48c0a792af32f6eebe6401f16"),
    ("stage4a_feature_spec", "fc00146efe5da9b3fbe740bb42ca99d157252cdefc045650f9e88d54d8fcfa8b"),
    ("stage4a_qc", "4bf72af66aa800fa9f12fdc8598bd06ebc859bb31c97ad97cf7fc11ef1614cf1"),
    ("stage4a_manifest", "2b844ef2dbc0c3e5e57493886a7533587350e1b4b4c76fc9d445ecda8a528fa0"),
    ("stage4b_table", "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8"),
    ("stage4b_transform_parameters", "baeab167e19381138f93c51ba2fa00e4eeed684b36122c80cd2bf9fc4fe4b08d"),
    ("stage4b_weak_label_rules", "3d78e66cea1fed5c75ef1cab1b7cf44d3d3d7bfae50909173bae6c3a0e0bff61"),
    ("stage4b_qc", "03b14efb6d297fd517043c8ff952977715415723f191d079d968c111368d31a9"),
    ("stage4b_manifest", "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f"),
    ("stage4c_full_model", "0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30"),
    ("stage4c_no_star_model", "6c3fe4fc7fe8fdde7b8f0f0d608c48e66a07945effb8c67c6b98d35e1955257c"),
    ("stage4c_model_spec", "d754c715c990f42cecd64259ca2c420427b9602c669dc7be9e80dee9554f61f6"),
    ("stage4c_qc", "3a1a90d3a8946fd35bde52324a625077f3127ac57f231b20d206d70b2c678ec7"),
    ("stage4c_manifest", "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee"),
    ("stage6a_policy", "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"),
])

ARTIFACT_PATHS = OrderedDict([
    ("cell_7a1_manifest", CELL_7A1_MANIFEST),
    ("t1_parquet", T1_PARQUET),
    ("t1_freeze_manifest", T1_FREEZE_MANIFEST),
    ("stage4a_feature_table", STAGE4A_FEATURE_TABLE),
    ("stage4a_feature_spec", STAGE4A_FEATURE_SPEC),
    ("stage4a_qc", STAGE4A_QC),
    ("stage4a_manifest", STAGE4A_MANIFEST),
    ("stage4b_table", STAGE4B_TABLE),
    ("stage4b_transform_parameters", STAGE4B_TRANSFORM_PARAMETERS),
    ("stage4b_weak_label_rules", STAGE4B_WEAK_LABEL_RULES),
    ("stage4b_qc", STAGE4B_QC),
    ("stage4b_manifest", STAGE4B_MANIFEST),
    ("stage4c_full_model", FULL_MODEL_PATH),
    ("stage4c_no_star_model", NO_STAR_MODEL_PATH),
    ("stage4c_model_spec", STAGE4C_MODEL_SPEC),
    ("stage4c_qc", STAGE4C_QC),
    ("stage4c_manifest", STAGE4C_MANIFEST),
    ("stage6a_policy", STAGE6A_POLICY),
])

EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36
EXPECTED_STAGE4A_ROWS = 71_659
EXPECTED_STAGE4A_COLUMNS = 30
EXPECTED_STAGE4B_ROWS = 71_659
EXPECTED_STAGE4B_COLUMNS = 43

EXPECTED_T1_GENE_COUNTS = {
    "BRCA1": 32_603,
    "BRCA2": 49_221,
    "MLH1": 13_684,
    "EGFR": 5_412,
}

EXPECTED_T1_AXIS_COUNTS = {
    "GermlineClassification": 97_526,
    "OncogenicityClassification": 52,
    "SomaticClinicalImpact": 25,
    "NoClassification": 3_317,
}

EXPECTED_CELL_7A1_DECISION = (
    "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_CHECKSUM_PROTECTED_"
    "TOP_LEVEL_AND_NESTED_SCHEMA_INVENTORIED_EVIDENCE_PACKET_"
    "DERIVABILITY_AUDITED_STAGE7A2_PREFLIGHT_ONLY"
)

MAXIMUM_OBSERVATION_WINDOW_DAYS = 10_516.0
SUBMITTER_LOG_MIN = 0.6931471805599453
SUBMITTER_LOG_MAX = 3.4339872044851463
T1_CUTOFF = pd.Timestamp("2025-12-27")

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

EXPECTED_MODEL_SETTINGS = {
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "class_weight": None,
    "max_iter": 2_000,
    "tol": 1e-6,
    "fit_intercept": True,
    "random_state": 42,
}


# --------------------------------------------------------------------------------------------------
# 4. OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

OUTPUTS = OrderedDict([
    ("artifact_inventory", STAGE7_TABLE_DIR / "cell_7a2_frozen_artifact_inventory_v1.csv"),
    ("formula_audit", STAGE7_TABLE_DIR / "cell_7a2_t0_frozen_transformation_formula_audit_v1.csv"),
    ("feature_inventory", STAGE7_TABLE_DIR / "cell_7a2_t1_feature_applicability_inventory_v1.csv"),
    ("model_inventory", STAGE7_TABLE_DIR / "cell_7a2_frozen_model_pipeline_inventory_v1.csv"),
    ("axis_inventory", STAGE7_TABLE_DIR / "cell_7a2_t1_classification_axis_applicability_v1.csv"),
    ("preflight_report", STAGE7_QC_DIR / "cell_7a2_feature_reconstruction_model_applicability_preflight_v1.json"),
    ("qc", STAGE7_QC_DIR / "cell_7a2_feature_reconstruction_model_applicability_qc_v1.json"),
    ("manifest", STAGE7_CONFIG_DIR / "cell_7a2_feature_reconstruction_model_applicability_manifest_v1.json"),
])


# --------------------------------------------------------------------------------------------------
# 5. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return Path(str(path) + ".sha256")


def read_sidecar_hash(path: Path) -> str:
    text = Path(path).read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def sidecar_is_valid(path: Path) -> bool:
    path = Path(path)
    sc = sidecar_path(path)
    return path.exists() and sc.exists() and read_sidecar_hash(sc) == sha256_file(path)


def verify_exact_hash(label: str, path: Path, expected: str) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required frozen artifact for {label}:\n{path}")
    observed = sha256_file(path)
    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}.\n"
            f"Expected: {expected}\nObserved: {observed}\nPath: {path}"
        )
    return observed


def json_native(value):
    if isinstance(value, dict):
        return {str(key): json_native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_native(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, np.ndarray):
        return [json_native(item) for item in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    proposed_hash = sha256_file(temporary)

    if path.exists():
        existing_hash = sha256_file(path)
        if existing_hash != proposed_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(
                "Refusing to overwrite a nonidentical frozen Cell 7A2 artifact.\n"
                f"Path: {path}\nExisting: {existing_hash}\nProposed: {proposed_hash}"
            )
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)

    return sha256_file(path)


def stable_write_json(path: Path, payload: dict) -> str:
    data = (
        json.dumps(
            json_native(payload),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    data = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.12g",
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def write_sidecar(path: Path) -> str:
    path = Path(path)
    payload = f"{sha256_file(path)}  {path.name}\n".encode("utf-8")
    stable_write_bytes(sidecar_path(path), payload)
    return sha256_file(sidecar_path(path))


def resolve_column(columns, aliases, label, required=True):
    exact = {str(column).lower(): str(column) for column in columns}
    for alias in aliases:
        if alias.lower() in exact:
            return exact[alias.lower()]
    if required:
        raise KeyError(
            f"Could not resolve required column '{label}'.\n"
            f"Aliases checked: {aliases}\nAvailable columns: {list(columns)}"
        )
    return None


def parse_json_value(value):
    if value is None or value is pd.NA:
        return None, "missing"
    try:
        if pd.isna(value):
            return None, "missing"
    except Exception:
        pass
    if isinstance(value, (dict, list)):
        return value, "native"
    text = str(value).strip()
    if not text:
        return None, "blank"
    try:
        return json.loads(text), "parsed"
    except json.JSONDecodeError:
        return None, "parse_error"


def normalize_gene(value) -> str:
    allowed = {"BRCA1", "BRCA2", "MLH1", "EGFR"}
    parsed, status = parse_json_value(value)
    if status == "parse_error":
        parsed = [str(value).strip()]
    if isinstance(parsed, str):
        parsed = [parsed]
    if not isinstance(parsed, list):
        return ""
    genes = sorted({
        str(gene).strip().upper()
        for gene in parsed
        if str(gene).strip().upper() in allowed
    })
    return genes[0] if len(genes) == 1 else ""


def boolish_to_float(value):
    if value is None or value is pd.NA:
        return np.nan
    try:
        if pd.isna(value):
            return np.nan
    except Exception:
        pass
    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        number = float(value)
        return number if number in {0.0, 1.0} else np.nan
    text = str(value).strip().lower()
    if text in {"true", "t", "yes", "y", "1"}:
        return 1.0
    if text in {"false", "f", "no", "n", "0"}:
        return 0.0
    return np.nan


def numeric_dict_sum(mapping):
    if not isinstance(mapping, dict):
        return np.nan, 1
    total = 0.0
    invalid = 0
    for value in mapping.values():
        try:
            number = float(value)
        except (TypeError, ValueError):
            invalid += 1
            continue
        if not np.isfinite(number) or number < 0:
            invalid += 1
            continue
        total += number
    return total, invalid


def normalized_entropy_from_group_counts(value):
    parsed, status = parse_json_value(value)
    if status == "parse_error":
        return np.nan, np.nan, 0, "parse_error"
    if parsed is None:
        return np.nan, np.nan, 0, status
    total, invalid = numeric_dict_sum(parsed)
    if invalid > 0:
        return np.nan, total, 0, "invalid_numeric_value"
    positive_counts = np.asarray(
        [float(item) for item in parsed.values() if float(item) > 0],
        dtype=float,
    )
    if len(positive_counts) == 0:
        return np.nan, total, 0, "no_positive_counts"
    if len(positive_counts) == 1:
        return 0.0, total, 1, "single_group"
    probabilities = positive_counts / positive_counts.sum()
    entropy = -float(np.sum(probabilities * np.log(probabilities)))
    normalized = float(np.clip(entropy / math.log(len(positive_counts)), 0.0, 1.0))
    return normalized, total, int(len(positive_counts)), "derived"


def extract_pipeline(artifact, label):
    if isinstance(artifact, Pipeline):
        return artifact, "<top-level>"

    preferred_keys = [
        "pipeline", "model_pipeline", "fitted_pipeline",
        "sklearn_pipeline", "model", "estimator", "classifier",
    ]

    if isinstance(artifact, dict):
        for key in preferred_keys:
            if key in artifact and isinstance(artifact[key], Pipeline):
                return artifact[key], f"[{key!r}]"

    matches = []
    visited = set()

    def walk(obj, location="<top-level>", depth=0):
        if depth > 8 or id(obj) in visited:
            return
        visited.add(id(obj))
        if isinstance(obj, Pipeline):
            matches.append((location, obj))
            return
        if isinstance(obj, dict):
            for key, item in obj.items():
                walk(item, f"{location}[{key!r}]", depth + 1)
        elif isinstance(obj, (list, tuple)):
            for index, item in enumerate(obj):
                walk(item, f"{location}[{index}]", depth + 1)
        else:
            for attribute in preferred_keys:
                if hasattr(obj, attribute):
                    try:
                        walk(getattr(obj, attribute), f"{location}.{attribute}", depth + 1)
                    except Exception:
                        pass

    walk(artifact)
    unique = {}
    for location, pipeline in matches:
        unique.setdefault(id(pipeline), (location, pipeline))
    found = list(unique.values())

    if len(found) != 1:
        raise TypeError(
            f"Expected exactly one sklearn Pipeline in {label}; found {len(found)}."
        )
    return found[0][1], found[0][0]


def get_exact_component(pipeline, component_type):
    matches = [step for _, step in pipeline.steps if isinstance(step, component_type)]
    if len(matches) != 1:
        raise AssertionError(
            f"Expected exactly one {component_type.__name__}; found {len(matches)}."
        )
    return matches[0]


def transformed_values_are_finite(values) -> bool:
    if sparse.issparse(values):
        return bool(np.isfinite(values.data).all())
    return bool(np.isfinite(np.asarray(values)).all())


def model_settings_match(classifier: LogisticRegression) -> bool:
    return bool(
        classifier.penalty == EXPECTED_MODEL_SETTINGS["penalty"]
        and np.isclose(float(classifier.C), EXPECTED_MODEL_SETTINGS["C"])
        and classifier.solver == EXPECTED_MODEL_SETTINGS["solver"]
        and classifier.class_weight is EXPECTED_MODEL_SETTINGS["class_weight"]
        and int(classifier.max_iter) == EXPECTED_MODEL_SETTINGS["max_iter"]
        and np.isclose(float(classifier.tol), EXPECTED_MODEL_SETTINGS["tol"])
        and bool(classifier.fit_intercept) == EXPECTED_MODEL_SETTINGS["fit_intercept"]
        and classifier.random_state == EXPECTED_MODEL_SETTINGS["random_state"]
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL FROZEN INPUT ARTIFACTS
# --------------------------------------------------------------------------------------------------

observed_hashes = OrderedDict()

for key, path in ARTIFACT_PATHS.items():
    observed_hashes[key] = verify_exact_hash(key, path, EXPECTED_HASHES[key])

if not sidecar_is_valid(CELL_7A1_MANIFEST):
    raise AssertionError("Cell 7A1 manifest sidecar verification failed.")

cell_7a1_payload = json.loads(CELL_7A1_MANIFEST.read_text(encoding="utf-8"))
if cell_7a1_payload.get("terminal_decision") != EXPECTED_CELL_7A1_DECISION:
    raise AssertionError("Cell 7A1 terminal decision does not authorize Cell 7A2.")

if cell_7a1_payload.get("next_authorized_cell", {}).get("cell_id") != "7A2":
    raise AssertionError("Cell 7A1 manifest does not identify Cell 7A2 as next authorized cell.")

immutable_hashes_before = {
    key: sha256_file(path)
    for key, path in ARTIFACT_PATHS.items()
}


# --------------------------------------------------------------------------------------------------
# 7. VERIFY PARQUET DIMENSIONS AND BUILD ARTIFACT INVENTORY
# --------------------------------------------------------------------------------------------------

t1_meta = pq.ParquetFile(T1_PARQUET).metadata
stage4a_meta = pq.ParquetFile(STAGE4A_FEATURE_TABLE).metadata
stage4b_meta = pq.ParquetFile(STAGE4B_TABLE).metadata

artifact_inventory_rows = []

for key, path in ARTIFACT_PATHS.items():
    row = {
        "artifact_key": key,
        "path": str(path),
        "file_name": path.name,
        "suffix": path.suffix.lower(),
        "expected_sha256": EXPECTED_HASHES[key],
        "observed_sha256": observed_hashes[key],
        "hash_verified": observed_hashes[key] == EXPECTED_HASHES[key],
        "bytes": int(path.stat().st_size),
        "sidecar_present": sidecar_path(path).exists(),
        "sidecar_verified": sidecar_is_valid(path) if sidecar_path(path).exists() else False,
        "rows": None,
        "columns": None,
    }
    if path.suffix.lower() == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        row["rows"] = int(metadata.num_rows)
        row["columns"] = int(metadata.num_columns)
    artifact_inventory_rows.append(row)

artifact_inventory = pd.DataFrame(artifact_inventory_rows)


# --------------------------------------------------------------------------------------------------
# 8. REPRODUCE THE FROZEN STAGE 4B TRANSFORMATIONS ON T0
#    USING RAW SOURCE FIELDS FROM STAGE 4A AND TRANSFORMED FIELDS FROM STAGE 4B
# --------------------------------------------------------------------------------------------------

# Stage 4B intentionally stores the transformed model features and weak-label outputs.
# The raw inputs used to create those transformed features remain in the frozen Stage 4A table.
# Therefore, this audit reads both tables and verifies exact row/key alignment before comparison.

stage4a_columns = pq.ParquetFile(STAGE4A_FEATURE_TABLE).schema_arrow.names
stage4b_columns = pq.ParquetFile(STAGE4B_TABLE).schema_arrow.names

stage4a_required_columns = [
    "t0_row_order",
    "rcv_accession",
    "recency_days",
    "recency_missing_flag",
    "unique_submitter_count",
    "log1p_unique_submitter_count",
    "aggregate_review_stars",
]

stage4b_required_columns = list(dict.fromkeys([
    "t0_row_order",
    "rcv_accession",
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
] + FULL_FEATURES))

missing_stage4a_columns = [
    column for column in stage4a_required_columns
    if column not in stage4a_columns
]
missing_stage4b_columns = [
    column for column in stage4b_required_columns
    if column not in stage4b_columns
]

if missing_stage4a_columns:
    raise KeyError(
        "Frozen Stage 4A table is missing required raw-source columns: "
        f"{missing_stage4a_columns}"
    )

if missing_stage4b_columns:
    raise KeyError(
        "Frozen Stage 4B table is missing required transformed columns: "
        f"{missing_stage4b_columns}"
    )

t0_raw = pd.read_parquet(
    STAGE4A_FEATURE_TABLE,
    columns=stage4a_required_columns,
).copy()

t0 = pd.read_parquet(
    STAGE4B_TABLE,
    columns=stage4b_required_columns,
).copy()

if len(t0_raw) != EXPECTED_STAGE4A_ROWS:
    raise RuntimeError(
        f"Loaded Stage 4A row count is {len(t0_raw):,}; "
        f"expected {EXPECTED_STAGE4A_ROWS:,}."
    )

if len(t0) != EXPECTED_STAGE4B_ROWS:
    raise RuntimeError(
        f"Loaded Stage 4B row count is {len(t0):,}; "
        f"expected {EXPECTED_STAGE4B_ROWS:,}."
    )

# Normalize and verify exact row-order/key alignment before using Stage 4A raw values
# to reproduce Stage 4B transformed values.
t0_raw_row_order = pd.to_numeric(
    t0_raw["t0_row_order"],
    errors="raise",
).astype("int64")

t0_row_order = pd.to_numeric(
    t0["t0_row_order"],
    errors="raise",
).astype("int64")

t0_raw_rcv = (
    t0_raw["rcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
)

t0_rcv = (
    t0["rcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
)

stage4a_row_order_valid = bool(
    t0_raw_row_order.nunique(dropna=False) == EXPECTED_STAGE4A_ROWS
    and np.array_equal(
        t0_raw_row_order.to_numpy(),
        np.arange(EXPECTED_STAGE4A_ROWS, dtype=np.int64),
    )
)

stage4b_row_order_valid = bool(
    t0_row_order.nunique(dropna=False) == EXPECTED_STAGE4B_ROWS
    and np.array_equal(
        t0_row_order.to_numpy(),
        np.arange(EXPECTED_STAGE4B_ROWS, dtype=np.int64),
    )
)

stage4a_stage4b_row_order_alignment_verified = bool(
    np.array_equal(
        t0_raw_row_order.to_numpy(),
        t0_row_order.to_numpy(),
    )
)

stage4a_stage4b_rcv_alignment_verified = bool(
    t0_raw_rcv.notna().all()
    and t0_rcv.notna().all()
    and not t0_raw_rcv.fillna("").eq("").any()
    and not t0_rcv.fillna("").eq("").any()
    and t0_raw_rcv.nunique(dropna=False) == EXPECTED_STAGE4A_ROWS
    and t0_rcv.nunique(dropna=False) == EXPECTED_STAGE4B_ROWS
    and np.array_equal(
        t0_raw_rcv.to_numpy(dtype=str),
        t0_rcv.to_numpy(dtype=str),
    )
)

if not stage4a_row_order_valid:
    raise RuntimeError("Frozen Stage 4A row-order verification failed.")

if not stage4b_row_order_valid:
    raise RuntimeError("Frozen Stage 4B row-order verification failed.")

if not stage4a_stage4b_row_order_alignment_verified:
    raise RuntimeError("Stage 4A and Stage 4B row-order alignment failed.")

if not stage4a_stage4b_rcv_alignment_verified:
    raise RuntimeError("Stage 4A and Stage 4B RCV-key alignment failed.")

# Recency transformation: raw recency_days comes from Stage 4A;
# saved recency_score comes from Stage 4B.
recency_days_t0 = pd.to_numeric(
    t0_raw["recency_days"],
    errors="coerce",
).astype(float)

saved_recency_t0 = pd.to_numeric(
    t0["recency_score"],
    errors="coerce",
).astype(float)

expected_recency_t0 = (
    1.0 - recency_days_t0 / MAXIMUM_OBSERVATION_WINDOW_DAYS
).clip(0.0, 1.0)
expected_recency_t0.loc[recency_days_t0.isna()] = np.nan

recency_formula_mismatches = int((~np.isclose(
    saved_recency_t0.to_numpy(),
    expected_recency_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True,
)).sum())

# Missingness was present in Stage 4A and was copied into Stage 4B.
saved_missing_stage4a_t0 = pd.to_numeric(
    t0_raw["recency_missing_flag"],
    errors="coerce",
).astype(float)

saved_missing_stage4b_t0 = pd.to_numeric(
    t0["recency_missing_flag"],
    errors="coerce",
).astype(float)

expected_missing_t0 = recency_days_t0.isna().astype(float)

recency_missing_stage4a_mismatches = int((~np.isclose(
    saved_missing_stage4a_t0.to_numpy(),
    expected_missing_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)).sum())

recency_missing_stage4b_mismatches = int((~np.isclose(
    saved_missing_stage4b_t0.to_numpy(),
    expected_missing_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)).sum())

recency_missing_mismatches = (
    recency_missing_stage4a_mismatches
    + recency_missing_stage4b_mismatches
)

# Submitter transformation: raw and log1p counts are stored in Stage 4A;
# the normalized submitter-diversity score is stored in Stage 4B.
submitter_count_t0 = pd.to_numeric(
    t0_raw["unique_submitter_count"],
    errors="raise",
).astype(float)

expected_log_submitter_t0 = np.log1p(submitter_count_t0)

saved_log_submitter_t0 = pd.to_numeric(
    t0_raw["log1p_unique_submitter_count"],
    errors="raise",
).astype(float)

log_submitter_mismatches = int((~np.isclose(
    saved_log_submitter_t0.to_numpy(),
    expected_log_submitter_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)).sum())

observed_log_min = float(expected_log_submitter_t0.min())
observed_log_max = float(expected_log_submitter_t0.max())

expected_submitter_score_t0 = (
    (expected_log_submitter_t0 - SUBMITTER_LOG_MIN)
    / (SUBMITTER_LOG_MAX - SUBMITTER_LOG_MIN)
).clip(0.0, 1.0)

saved_submitter_score_t0 = pd.to_numeric(
    t0["submitter_diversity_score"],
    errors="raise",
).astype(float)

submitter_formula_mismatches = int((~np.isclose(
    saved_submitter_score_t0.to_numpy(),
    expected_submitter_score_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)).sum())

# Review confidence is the Stage 4B copy of the Stage 4A review-star value.
saved_review_confidence_t0 = pd.to_numeric(
    t0["review_confidence"],
    errors="raise",
).astype(float)

expected_review_confidence_t0 = pd.to_numeric(
    t0_raw["aggregate_review_stars"],
    errors="raise",
).astype(float)

review_formula_mismatches = int((~np.isclose(
    saved_review_confidence_t0.to_numpy(),
    expected_review_confidence_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
)).sum())

formula_audit = pd.DataFrame([
    {
        "transformation": "stage4a_stage4b_row_and_key_alignment",
        "frozen_formula": (
            "exact zero-based t0_row_order and exact RCV sequence alignment "
            "between frozen Stage 4A and Stage 4B"
        ),
        "rows_audited": len(t0),
        "mismatch_count": int(
            (not stage4a_stage4b_row_order_alignment_verified)
            or (not stage4a_stage4b_rcv_alignment_verified)
        ),
        "exactly_reproduced": bool(
            stage4a_stage4b_row_order_alignment_verified
            and stage4a_stage4b_rcv_alignment_verified
        ),
    },
    {
        "transformation": "recency_score",
        "frozen_formula": "clip(1 - recency_days / 10516, 0, 1); missing remains missing",
        "rows_audited": len(t0),
        "mismatch_count": recency_formula_mismatches,
        "exactly_reproduced": recency_formula_mismatches == 0,
    },
    {
        "transformation": "recency_missing_flag",
        "frozen_formula": (
            "1 when Stage 4A recency_days is missing; otherwise 0; "
            "verified in both Stage 4A and Stage 4B"
        ),
        "rows_audited": len(t0),
        "mismatch_count": recency_missing_mismatches,
        "exactly_reproduced": recency_missing_mismatches == 0,
    },
    {
        "transformation": "log1p_unique_submitter_count",
        "frozen_formula": "log1p(Stage 4A unique_submitter_count)",
        "rows_audited": len(t0),
        "mismatch_count": log_submitter_mismatches,
        "exactly_reproduced": log_submitter_mismatches == 0,
    },
    {
        "transformation": "submitter_diversity_score",
        "frozen_formula": (
            "clip((Stage 4A log1p_unique_submitter_count - 0.6931471805599453) / "
            "(3.4339872044851463 - 0.6931471805599453), 0, 1)"
        ),
        "rows_audited": len(t0),
        "mismatch_count": submitter_formula_mismatches,
        "exactly_reproduced": submitter_formula_mismatches == 0,
    },
    {
        "transformation": "review_confidence",
        "frozen_formula": "Stage 4B review_confidence = Stage 4A aggregate_review_stars",
        "rows_audited": len(t0),
        "mismatch_count": review_formula_mismatches,
        "exactly_reproduced": review_formula_mismatches == 0,
    },
])


# --------------------------------------------------------------------------------------------------
# 9. RESOLVE T1 COLUMNS AND RECONSTRUCT FROZEN MODEL INPUTS IN MEMORY
# --------------------------------------------------------------------------------------------------

t1_schema = pq.ParquetFile(T1_PARQUET).schema_arrow.names

t1_columns = {
    "rcv_accession": resolve_column(
        t1_schema,
        ["rcv_accession", "rcv"],
        "T1 RCV accession",
    ),
    "target_gene": resolve_column(
        t1_schema,
        ["target_genes_json", "target_gene", "gene_symbol", "gene"],
        "T1 target gene",
    ),
    "embedded_cutoff": resolve_column(
        t1_schema,
        ["embedded_data_cutoff_date", "embedded_data_cutoff", "embedded_cutoff", "release_embedded_cutoff", "data_cutoff"],
        "T1 embedded cutoff",
    ),
    "aggregate_last_evaluated": resolve_column(
        t1_schema,
        ["aggregate_last_evaluated", "last_evaluated", "aggregate_date_last_evaluated"],
        "T1 aggregate last evaluated",
    ),
    "unique_submitter_count": resolve_column(
        t1_schema,
        ["unique_submitter_count_xml", "unique_submitter_count", "aggregate_unique_submitter_count"],
        "T1 unique submitter count",
    ),
    "aggregate_review_stars": resolve_column(
        t1_schema,
        ["aggregate_review_stars", "review_stars"],
        "T1 aggregate review stars",
    ),
    "aggregate_conflict_flag": resolve_column(
        t1_schema,
        ["aggregate_conflict_flag", "aggregate_conflict", "conflict_flag"],
        "T1 aggregate conflict flag",
    ),
    "classification_group_counts": resolve_column(
        t1_schema,
        ["scv_classification_group_counts_json", "scv_group_counts_json", "classification_group_counts_json"],
        "T1 SCV classification-group counts",
    ),
    "classification_axis": resolve_column(
        t1_schema,
        ["aggregate_classification_axis", "classification_axis"],
        "T1 aggregate classification axis",
    ),
    "scv_count": resolve_column(
        t1_schema,
        ["scv_count_xml", "scv_count"],
        "T1 SCV count",
    ),
}

t1_load_columns = list(dict.fromkeys(t1_columns.values()))
t1 = pd.read_parquet(T1_PARQUET, columns=t1_load_columns).copy()

rcv_t1 = t1[t1_columns["rcv_accession"]].astype("string").str.strip().str.upper()
gene_t1 = t1[t1_columns["target_gene"]].map(normalize_gene).astype("string")
axis_t1 = t1[t1_columns["classification_axis"]].astype("string").str.strip()
cutoff_t1 = (
    t1[t1_columns["embedded_cutoff"]]
    .astype("string")
    .str.strip()
    .str.slice(0, 10)
)

last_evaluated_t1 = pd.to_datetime(
    t1[t1_columns["aggregate_last_evaluated"]],
    errors="coerce",
    utc=True,
).dt.tz_convert(None).dt.normalize()

recency_days_t1 = (T1_CUTOFF - last_evaluated_t1).dt.days.astype(float)
post_cutoff_dates = int((recency_days_t1 < 0).sum())
recency_missing_flag_t1 = recency_days_t1.isna().astype(float)
recency_score_t1 = (1.0 - recency_days_t1 / MAXIMUM_OBSERVATION_WINDOW_DAYS).clip(0.0, 1.0)
recency_score_t1.loc[recency_days_t1.isna()] = np.nan

submitter_count_t1 = pd.to_numeric(
    t1[t1_columns["unique_submitter_count"]], errors="coerce"
).astype(float)
log_submitter_t1 = np.log1p(submitter_count_t1)
submitter_score_t1 = (
    (log_submitter_t1 - SUBMITTER_LOG_MIN)
    / (SUBMITTER_LOG_MAX - SUBMITTER_LOG_MIN)
).clip(0.0, 1.0)

review_confidence_t1 = pd.to_numeric(
    t1[t1_columns["aggregate_review_stars"]], errors="coerce"
).astype(float)
conflict_t1 = (
    t1[t1_columns["aggregate_conflict_flag"]]
    .map(boolish_to_float)
    .astype(float)
)

entropy_values = []
group_count_totals = []
nonzero_group_counts = []
entropy_statuses = []

for value in t1[t1_columns["classification_group_counts"]].tolist():
    entropy, total, nonzero_groups, status = normalized_entropy_from_group_counts(value)
    entropy_values.append(entropy)
    group_count_totals.append(total)
    nonzero_group_counts.append(nonzero_groups)
    entropy_statuses.append(status)

entropy_t1 = pd.Series(entropy_values, index=t1.index, dtype=float)
group_count_total_t1 = pd.Series(group_count_totals, index=t1.index, dtype=float)
entropy_status_t1 = pd.Series(entropy_statuses, index=t1.index, dtype="string")

saved_scv_count_t1 = pd.to_numeric(
    t1[t1_columns["scv_count"]], errors="coerce"
).astype(float)

comparable_scv_rows = saved_scv_count_t1.notna() & group_count_total_t1.notna()
scv_group_count_mismatches = int((~np.isclose(
    saved_scv_count_t1.loc[comparable_scv_rows].to_numpy(),
    group_count_total_t1.loc[comparable_scv_rows].to_numpy(),
    rtol=0.0,
    atol=0.0,
)).sum())

t1_features = pd.DataFrame({
    "recency_score": recency_score_t1,
    "recency_missing_flag": recency_missing_flag_t1,
    "submitter_diversity_score": submitter_score_t1,
    "review_confidence": review_confidence_t1,
    "aggregate_conflict_flag": conflict_t1,
    "scv_group_entropy_normalized": entropy_t1,
})

# The row-level T1 feature table remains in memory only.


# --------------------------------------------------------------------------------------------------
# 10. VERIFY FROZEN MODEL PIPELINES AND PREPROCESS T1 WITHOUT SCORING
# --------------------------------------------------------------------------------------------------

full_artifact = joblib.load(FULL_MODEL_PATH)
no_star_artifact = joblib.load(NO_STAR_MODEL_PATH)

full_pipeline, full_pipeline_location = extract_pipeline(full_artifact, "Full-GES artifact")
no_star_pipeline, no_star_pipeline_location = extract_pipeline(no_star_artifact, "No-star-GES artifact")

model_definitions = OrderedDict([
    ("full_ges", {
        "display": "Full GES",
        "path": FULL_MODEL_PATH,
        "artifact": full_artifact,
        "pipeline": full_pipeline,
        "pipeline_location": full_pipeline_location,
        "features": FULL_FEATURES,
    }),
    ("no_star_ges", {
        "display": "No-star GES",
        "path": NO_STAR_MODEL_PATH,
        "artifact": no_star_artifact,
        "pipeline": no_star_pipeline,
        "pipeline_location": no_star_pipeline_location,
        "features": NO_STAR_FEATURES,
    }),
])

model_inventory_rows = []
model_results = {}

for model_key, definition in model_definitions.items():
    pipeline = definition["pipeline"]
    expected_features = definition["features"]

    imputer = get_exact_component(pipeline, SimpleImputer)
    scaler = get_exact_component(pipeline, StandardScaler)
    classifier = get_exact_component(pipeline, LogisticRegression)

    feature_names = (
        list(map(str, pipeline.feature_names_in_))
        if hasattr(pipeline, "feature_names_in_")
        else []
    )

    feature_order_verified = (
        feature_names == expected_features
        if feature_names
        else int(getattr(pipeline, "n_features_in_", len(expected_features))) == len(expected_features)
    )

    fitted_state_verified = bool(
        hasattr(imputer, "statistics_")
        and hasattr(scaler, "mean_")
        and hasattr(scaler, "scale_")
        and hasattr(classifier, "coef_")
        and hasattr(classifier, "intercept_")
        and hasattr(classifier, "classes_")
        and np.isfinite(np.asarray(imputer.statistics_, dtype=float)).all()
        and np.isfinite(np.asarray(scaler.mean_, dtype=float)).all()
        and np.isfinite(np.asarray(scaler.scale_, dtype=float)).all()
        and np.isfinite(np.asarray(classifier.coef_, dtype=float)).all()
        and np.isfinite(np.asarray(classifier.intercept_, dtype=float)).all()
        and sorted(np.asarray(classifier.classes_).tolist()) == [0, 1]
    )

    settings_verified = model_settings_match(classifier)

    preprocessing_pipeline = Pipeline(pipeline.steps[:-1])
    model_input = t1_features[expected_features].copy()
    transformed = preprocessing_pipeline.transform(model_input)

    transformed_shape = transformed.shape
    transformed_finite = transformed_values_are_finite(transformed)

    model_results[model_key] = {
        "rows": int(transformed_shape[0]),
        "columns": int(transformed_shape[1]),
        "all_finite": bool(transformed_finite),
        "feature_order_verified": bool(feature_order_verified),
        "fitted_state_verified": bool(fitted_state_verified),
        "settings_verified": bool(settings_verified),
    }

    model_inventory_rows.append({
        "model_key": model_key,
        "model": definition["display"],
        "artifact_path": str(definition["path"]),
        "artifact_sha256": sha256_file(definition["path"]),
        "pipeline_location": definition["pipeline_location"],
        "pipeline_steps": ";".join(
            f"{name}:{type(step).__name__}" for name, step in pipeline.steps
        ),
        "expected_features": ";".join(expected_features),
        "pipeline_feature_names": ";".join(feature_names),
        "feature_order_verified": feature_order_verified,
        "fitted_state_verified": fitted_state_verified,
        "imputer_strategy": str(imputer.strategy),
        "classifier_solver": classifier.solver,
        "classifier_penalty": classifier.penalty,
        "classifier_C": float(classifier.C),
        "classifier_max_iter": int(classifier.max_iter),
        "classifier_tol": float(classifier.tol),
        "classifier_class_weight": str(classifier.class_weight),
        "classifier_fit_intercept": bool(classifier.fit_intercept),
        "classifier_random_state": classifier.random_state,
        "classifier_settings_verified": settings_verified,
        "t1_preprocessing_rows": int(transformed_shape[0]),
        "t1_preprocessing_columns": int(transformed_shape[1]),
        "t1_preprocessing_all_finite": transformed_finite,
        "fit_called_in_cell_7a2": False,
        "predict_called_in_cell_7a2": False,
        "predict_proba_called_in_cell_7a2": False,
        "score_created_in_cell_7a2": False,
    })

    del transformed, model_input, preprocessing_pipeline

model_inventory = pd.DataFrame(model_inventory_rows)


# --------------------------------------------------------------------------------------------------
# 11. FEATURE AND CLASSIFICATION-AXIS INVENTORIES
# --------------------------------------------------------------------------------------------------

feature_inventory_rows = []

for feature in FULL_FEATURES:
    t0_values = pd.to_numeric(t0[feature], errors="coerce").astype(float)
    t1_values = pd.to_numeric(t1_features[feature], errors="coerce").astype(float)

    t0_nonmissing = t0_values.dropna()
    t1_nonmissing = t1_values.dropna()

    t0_min = float(t0_nonmissing.min()) if len(t0_nonmissing) else np.nan
    t0_max = float(t0_nonmissing.max()) if len(t0_nonmissing) else np.nan
    t1_min = float(t1_nonmissing.min()) if len(t1_nonmissing) else np.nan
    t1_max = float(t1_nonmissing.max()) if len(t1_nonmissing) else np.nan

    domain_upper = 3.0 if feature == "review_confidence" else 1.0

    feature_inventory_rows.append({
        "feature": feature,
        "used_by_full_ges": feature in FULL_FEATURES,
        "used_by_no_star_ges": feature in NO_STAR_FEATURES,
        "t0_nonmissing": int(t0_values.notna().sum()),
        "t0_missing": int(t0_values.isna().sum()),
        "t0_min": t0_min,
        "t0_max": t0_max,
        "t1_nonmissing": int(t1_values.notna().sum()),
        "t1_missing": int(t1_values.isna().sum()),
        "t1_missing_fraction": float(t1_values.isna().mean()),
        "t1_min": t1_min,
        "t1_max": t1_max,
        "t1_below_t0_range": int((t1_nonmissing < t0_min).sum()),
        "t1_above_t0_range": int((t1_nonmissing > t0_max).sum()),
        "t1_outside_frozen_domain": int(
            ((t1_nonmissing < 0.0) | (t1_nonmissing > domain_upper)).sum()
        ),
        "t1_infinite_values": int(np.isinf(t1_values.to_numpy()).sum()),
        "frozen_median_imputer_available": True,
        "row_level_feature_persisted": False,
    })

feature_inventory = pd.DataFrame(feature_inventory_rows)

axis_inventory = (
    pd.DataFrame({
        "classification_axis": axis_t1,
        "target_gene": gene_t1,
    })
    .groupby(["classification_axis", "target_gene"], dropna=False)
    .size()
    .reset_index(name="rows")
)


def axis_policy(axis):
    if axis == "GermlineClassification":
        return "primary_germline_domain_candidate"
    if axis == "NoClassification":
        return "retain_as_evidence_only_not_as_classification"
    return "retain_separately_requires_prespecified_non_germline_policy"


axis_inventory["scientific_applicability"] = axis_inventory["classification_axis"].map(axis_policy)
axis_inventory["included_in_preprocessing_check"] = True
axis_inventory["score_created_in_cell_7a2"] = False

observed_gene_counts = {
    str(key): int(value)
    for key, value in gene_t1.value_counts(dropna=False).items()
}
observed_axis_counts = {
    str(key): int(value)
    for key, value in axis_t1.value_counts(dropna=False).items()
}


# --------------------------------------------------------------------------------------------------
# 12. VERIFY THE COMBINED-METADATA POLICY WITHOUT APPLYING IT
# --------------------------------------------------------------------------------------------------

stage6a_policy_payload = json.loads(STAGE6A_POLICY.read_text(encoding="utf-8"))
stage6a_policy_text = json.dumps(stage6a_policy_payload, sort_keys=True).lower()
combined_metadata_policy_identified = (
    "combined_metadata_instability_risk" in stage6a_policy_text
    and "frozen_before_outcome_label_load_or_temporal_performance" in stage6a_policy_text
)


# --------------------------------------------------------------------------------------------------
# 13. IMMUTABILITY RECHECK
# --------------------------------------------------------------------------------------------------

immutable_hashes_after = {
    key: sha256_file(path)
    for key, path in ARTIFACT_PATHS.items()
}
immutable_inputs_unchanged = immutable_hashes_before == immutable_hashes_after


# --------------------------------------------------------------------------------------------------
# 14. QUALITY-CONTROL REGISTER
# --------------------------------------------------------------------------------------------------

entropy_parse_errors = int(entropy_status_t1.eq("parse_error").sum())
entropy_invalid_values = int(entropy_status_t1.eq("invalid_numeric_value").sum())
entropy_missing_rows = int(entropy_t1.isna().sum())
all_infinite_feature_values = int(sum(
    np.isinf(pd.to_numeric(t1_features[column], errors="coerce").to_numpy()).sum()
    for column in t1_features.columns
))

qc_checks = OrderedDict([
    ("all_18_frozen_hashes_verified", all(
        observed_hashes[key] == EXPECTED_HASHES[key] for key in EXPECTED_HASHES
    )),
    ("cell_7a1_manifest_sidecar_verified", sidecar_is_valid(CELL_7A1_MANIFEST)),
    ("cell_7a1_terminal_decision_verified", cell_7a1_payload.get("terminal_decision") == EXPECTED_CELL_7A1_DECISION),
    ("cell_7a2_authorized", cell_7a1_payload.get("next_authorized_cell", {}).get("cell_id") == "7A2"),
    ("t1_dimensions_verified", t1_meta.num_rows == EXPECTED_T1_ROWS and t1_meta.num_columns == EXPECTED_T1_COLUMNS),
    ("stage4a_dimensions_verified", stage4a_meta.num_rows == EXPECTED_STAGE4A_ROWS and stage4a_meta.num_columns == EXPECTED_STAGE4A_COLUMNS),
    ("stage4b_dimensions_verified", stage4b_meta.num_rows == EXPECTED_STAGE4B_ROWS and stage4b_meta.num_columns == EXPECTED_STAGE4B_COLUMNS),
    ("stage4a_zero_based_row_order_verified", stage4a_row_order_valid),
    ("stage4b_zero_based_row_order_verified", stage4b_row_order_valid),
    ("stage4a_stage4b_row_order_alignment_verified", stage4a_stage4b_row_order_alignment_verified),
    ("stage4a_stage4b_rcv_alignment_verified", stage4a_stage4b_rcv_alignment_verified),
    ("t0_recency_formula_exact", recency_formula_mismatches == 0),
    ("t0_recency_missing_formula_exact", recency_missing_mismatches == 0),
    ("t0_log_submitter_formula_exact", log_submitter_mismatches == 0),
    ("t0_submitter_score_formula_exact", submitter_formula_mismatches == 0),
    ("t0_review_confidence_formula_exact", review_formula_mismatches == 0),
    ("t0_submitter_min_constant_verified", np.isclose(observed_log_min, SUBMITTER_LOG_MIN, rtol=0.0, atol=1e-15)),
    ("t0_submitter_max_constant_verified", np.isclose(observed_log_max, SUBMITTER_LOG_MAX, rtol=0.0, atol=1e-15)),
    ("t1_loaded_rows_verified", len(t1) == EXPECTED_T1_ROWS),
    ("t1_rcv_nonmissing", int(rcv_t1.isna().sum()) == 0 and int(rcv_t1.fillna("").eq("").sum()) == 0),
    ("t1_rcv_unique", rcv_t1.nunique(dropna=False) == EXPECTED_T1_ROWS),
    ("t1_gene_complete", int(gene_t1.fillna("").eq("").sum()) == 0),
    ("t1_gene_counts_verified", observed_gene_counts == EXPECTED_T1_GENE_COUNTS),
    ("t1_axis_counts_verified", observed_axis_counts == EXPECTED_T1_AXIS_COUNTS),
    ("t1_cutoff_verified", set(cutoff_t1.dropna().unique().tolist()) == {"2025-12-27"}),
    ("t1_no_post_cutoff_dates", post_cutoff_dates == 0),
    ("t1_submitter_counts_positive", int((submitter_count_t1.dropna() <= 0).sum()) == 0),
    ("t1_review_stars_in_domain", int(((review_confidence_t1.dropna() < 0) | (review_confidence_t1.dropna() > 3)).sum()) == 0),
    ("t1_conflict_values_complete", int(conflict_t1.isna().sum()) == 0),
    ("t1_entropy_json_parse_errors_zero", entropy_parse_errors == 0),
    ("t1_entropy_invalid_numeric_values_zero", entropy_invalid_values == 0),
    ("t1_entropy_complete", entropy_missing_rows == 0),
    ("t1_entropy_within_domain", int(((entropy_t1 < 0) | (entropy_t1 > 1)).sum()) == 0),
    ("t1_group_counts_reconcile_to_scv_count", scv_group_count_mismatches == 0),
    ("t1_no_infinite_feature_values", all_infinite_feature_values == 0),
    ("full_ges_feature_order_verified", model_results["full_ges"]["feature_order_verified"]),
    ("full_ges_fitted_state_verified", model_results["full_ges"]["fitted_state_verified"]),
    ("full_ges_settings_verified", model_results["full_ges"]["settings_verified"]),
    ("full_ges_preprocessing_shape_verified", model_results["full_ges"]["rows"] == EXPECTED_T1_ROWS and model_results["full_ges"]["columns"] == len(FULL_FEATURES)),
    ("full_ges_preprocessing_all_finite", model_results["full_ges"]["all_finite"]),
    ("no_star_feature_order_verified", model_results["no_star_ges"]["feature_order_verified"]),
    ("no_star_fitted_state_verified", model_results["no_star_ges"]["fitted_state_verified"]),
    ("no_star_settings_verified", model_results["no_star_ges"]["settings_verified"]),
    ("no_star_preprocessing_shape_verified", model_results["no_star_ges"]["rows"] == EXPECTED_T1_ROWS and model_results["no_star_ges"]["columns"] == len(NO_STAR_FEATURES)),
    ("no_star_preprocessing_all_finite", model_results["no_star_ges"]["all_finite"]),
    ("combined_metadata_policy_identified", combined_metadata_policy_identified),
    ("immutable_inputs_unchanged", immutable_inputs_unchanged),
    ("weak_labels_not_created", True),
    ("model_fit_not_called", True),
    ("model_refit_not_called", True),
    ("predict_not_called", True),
    ("predict_proba_not_called", True),
    ("decision_function_not_called", True),
    ("t1_full_ges_score_not_created", True),
    ("t1_no_star_score_not_created", True),
    ("t1_combined_metadata_score_not_created", True),
    ("threshold_or_weight_optimization_not_performed", True),
    ("row_level_t1_feature_artifact_not_written", True),
    ("rag_corpus_not_constructed", True),
    ("embeddings_not_constructed", True),
    ("question_set_not_constructed", True),
    ("llm_not_called", True),
])

failed_checks = [name for name, passed in qc_checks.items() if passed is not True]
passed_checks = len(qc_checks) - len(failed_checks)
all_checks_passed = len(failed_checks) == 0

final_decision = (
    "PASS_STAGE7A2_FROZEN_FEATURE_TRANSFORMS_AND_MODEL_PACKAGES_VERIFIED_"
    "T1_FEATURE_RECONSTRUCTION_AND_PREPROCESSING_APPLICABILITY_CONFIRMED_"
    "NO_SCORING_STAGE7A3_FROZEN_T1_SCORE_MATERIALIZATION_AUTHORIZED"
    if all_checks_passed
    else "FAIL_STAGE7A2_PREFLIGHT_STAGE7A3_NOT_AUTHORIZED"
)

# Do not freeze a failed preflight package. This preserves clean rerun behavior.
if not all_checks_passed:
    print("CELL 7A2 PREFLIGHT FAILED BEFORE OUTPUT FREEZE")
    print("Failed checks:")
    for failed_name in failed_checks:
        print(" -", failed_name)
    raise RuntimeError(
        "Cell 7A2 preflight failed before any Cell 7A2 artifact was written."
    )


# --------------------------------------------------------------------------------------------------
# 15. PRESERVE CREATION TIMESTAMP ACROSS SAFE RERUNS
# --------------------------------------------------------------------------------------------------

existing_created_utc = None
for candidate in [OUTPUTS["manifest"], OUTPUTS["preflight_report"], OUTPUTS["qc"]]:
    if candidate.exists():
        try:
            existing_created_utc = json.loads(
                candidate.read_text(encoding="utf-8")
            ).get("created_utc")
            if existing_created_utc:
                break
        except Exception:
            pass

created_utc = existing_created_utc or datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 16. WRITE TABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

output_hashes = OrderedDict()

for key, frame in [
    ("artifact_inventory", artifact_inventory),
    ("formula_audit", formula_audit),
    ("feature_inventory", feature_inventory),
    ("model_inventory", model_inventory),
    ("axis_inventory", axis_inventory),
]:
    output_hashes[key] = stable_write_csv(OUTPUTS[key], frame)
    write_sidecar(OUTPUTS[key])
    if not sidecar_is_valid(OUTPUTS[key]):
        raise AssertionError(f"Output sidecar verification failed: {OUTPUTS[key]}")


# --------------------------------------------------------------------------------------------------
# 17. WRITE PREFLIGHT AND QC JSON
# --------------------------------------------------------------------------------------------------

preflight_report = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": created_utc,
    "purpose": "Frozen T1 feature reconstruction and model-applicability preflight without scoring.",
    "authorization": {
        "prior_cell": "7A1",
        "prior_manifest_path": str(CELL_7A1_MANIFEST),
        "prior_manifest_sha256": sha256_file(CELL_7A1_MANIFEST),
        "prior_terminal_decision_verified": True,
        "model_fitting_authorized": False,
        "t1_scoring_authorized_in_cell_7a2": False,
        "threshold_or_weight_optimization_authorized": False,
        "embedding_construction_authorized": False,
        "llm_generation_authorized": False,
    },
    "frozen_inputs": {
        key: {"path": str(ARTIFACT_PATHS[key]), "sha256": observed_hashes[key]}
        for key in ARTIFACT_PATHS
    },
    "t1_source_column_mapping": t1_columns,
    "frozen_transformation_constants": {
        "maximum_observation_window_days": MAXIMUM_OBSERVATION_WINDOW_DAYS,
        "submitter_log_min": SUBMITTER_LOG_MIN,
        "submitter_log_max": SUBMITTER_LOG_MAX,
    },
    "t0_formula_reproduction": {
        row["transformation"]: {
            "rows_audited": int(row["rows_audited"]),
            "mismatch_count": int(row["mismatch_count"]),
            "exactly_reproduced": bool(row["exactly_reproduced"]),
        }
        for row in formula_audit.to_dict("records")
    },
    "t1_accounting": {
        "rows": int(len(t1)),
        "columns": int(t1_meta.num_columns),
        "unique_rcv_accessions": int(rcv_t1.nunique(dropna=False)),
        "gene_counts": observed_gene_counts,
        "classification_axis_counts": observed_axis_counts,
        "embedded_cutoff_values": sorted(cutoff_t1.dropna().unique().tolist()),
        "post_cutoff_last_evaluated_dates": post_cutoff_dates,
    },
    "t1_feature_reconstruction": {
        "recency_missing_rows": int(recency_score_t1.isna().sum()),
        "entropy_missing_rows": entropy_missing_rows,
        "entropy_parse_errors": entropy_parse_errors,
        "entropy_invalid_numeric_values": entropy_invalid_values,
        "group_count_vs_scv_count_mismatches": scv_group_count_mismatches,
        "row_level_feature_table_persisted": False,
    },
    "frozen_model_applicability": model_results,
    "combined_metadata_policy": {
        "path": str(STAGE6A_POLICY),
        "sha256": sha256_file(STAGE6A_POLICY),
        "identified": combined_metadata_policy_identified,
        "applied": False,
    },
    "prohibited_operations_confirmed": {
        "weak_labels_created": False,
        "model_fit_called": False,
        "model_refit_called": False,
        "predict_called": False,
        "predict_proba_called": False,
        "decision_function_called": False,
        "full_ges_score_created": False,
        "no_star_score_created": False,
        "combined_metadata_score_created": False,
        "threshold_optimized": False,
        "weight_optimized": False,
        "rag_corpus_constructed": False,
        "embeddings_constructed": False,
        "question_set_constructed": False,
        "llm_called": False,
    },
    "qc_summary": {
        "passed_checks": passed_checks,
        "failed_checks": len(failed_checks),
        "total_checks": len(qc_checks),
        "failed_check_names": failed_checks,
    },
    "decision": final_decision,
    "next_authorized_cell": (
        {
            "cell_id": "7A3",
            "scope": (
                "Apply the frozen Full-GES, No-star-GES, and combined-metadata "
                "specifications to T1 and checksum-freeze the T1 score package."
            ),
            "model_fitting": "PROHIBITED",
            "threshold_or_weight_optimization": "PROHIBITED",
            "embedding_construction": "PROHIBITED",
            "llm_generation": "PROHIBITED",
        }
        if all_checks_passed else None
    ),
}

output_hashes["preflight_report"] = stable_write_json(
    OUTPUTS["preflight_report"], preflight_report
)
write_sidecar(OUTPUTS["preflight_report"])

qc_payload = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": created_utc,
    "checks": [
        {"check": name, "passed": bool(passed)}
        for name, passed in qc_checks.items()
    ],
    "passed_checks": passed_checks,
    "failed_checks": len(failed_checks),
    "total_checks": len(qc_checks),
    "failed_check_names": failed_checks,
    "diagnostics": {
        "t0_recency_formula_mismatches": recency_formula_mismatches,
        "t0_recency_missing_mismatches": recency_missing_mismatches,
        "t0_recency_missing_stage4a_mismatches": recency_missing_stage4a_mismatches,
        "t0_recency_missing_stage4b_mismatches": recency_missing_stage4b_mismatches,
        "stage4a_stage4b_row_order_alignment_verified": stage4a_stage4b_row_order_alignment_verified,
        "stage4a_stage4b_rcv_alignment_verified": stage4a_stage4b_rcv_alignment_verified,
        "t0_log_submitter_mismatches": log_submitter_mismatches,
        "t0_submitter_score_mismatches": submitter_formula_mismatches,
        "t0_review_confidence_mismatches": review_formula_mismatches,
        "t1_post_cutoff_dates": post_cutoff_dates,
        "t1_entropy_parse_errors": entropy_parse_errors,
        "t1_entropy_invalid_numeric_values": entropy_invalid_values,
        "t1_entropy_missing_rows": entropy_missing_rows,
        "t1_group_count_vs_scv_count_mismatches": scv_group_count_mismatches,
    },
    "decision": final_decision,
}

output_hashes["qc"] = stable_write_json(OUTPUTS["qc"], qc_payload)
write_sidecar(OUTPUTS["qc"])

for key in ["preflight_report", "qc"]:
    if not sidecar_is_valid(OUTPUTS[key]):
        raise AssertionError(f"Output sidecar verification failed: {OUTPUTS[key]}")


# --------------------------------------------------------------------------------------------------
# 18. WRITE AND VERIFY THE CELL 7A2 MANIFEST
# --------------------------------------------------------------------------------------------------

output_records = []
for key, path in OUTPUTS.items():
    if key == "manifest":
        continue
    record = {
        "role": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha256_file(path),
        "bytes": int(path.stat().st_size),
        "sidecar_path": str(sidecar_path(path)),
        "sidecar_verified": sidecar_is_valid(path),
    }
    if path.suffix.lower() == ".csv":
        frame = pd.read_csv(path)
        record["rows"] = int(len(frame))
        record["columns"] = int(frame.shape[1])
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True
    output_records.append(record)

manifest_payload = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "package_version": "v1",
    "created_utc": created_utc,
    "purpose": "Frozen T1 feature reconstruction and model-applicability preflight without score generation.",
    "upstream_lineage": {
        key: {"path": str(path), "sha256": observed_hashes[key]}
        for key, path in ARTIFACT_PATHS.items()
    },
    "scientific_boundary": {
        "t1_features_reconstructed_in_memory": True,
        "row_level_t1_feature_table_persisted": False,
        "frozen_preprocessing_applied": True,
        "classifier_scoring_applied": False,
        "full_ges_score_created": False,
        "no_star_score_created": False,
        "combined_metadata_score_created": False,
        "weak_labels_created": False,
        "model_fitted": False,
        "model_refitted": False,
        "model_tuned": False,
        "threshold_optimized": False,
        "weight_optimized": False,
        "rag_corpus_constructed": False,
        "embeddings_constructed": False,
        "question_set_constructed": False,
        "llm_called": False,
    },
    "output_artifacts": output_records,
    "qc": {
        "path": str(OUTPUTS["qc"]),
        "sha256": sha256_file(OUTPUTS["qc"]),
        "passed_checks": passed_checks,
        "failed_checks": len(failed_checks),
        "total_checks": len(qc_checks),
    },
    "decision": final_decision,
    "next_authorized_cell": (
        {
            "cell_id": "7A3",
            "name": "Frozen T1 score materialization and checksum freeze",
        }
        if all_checks_passed else None
    ),
}

output_hashes["manifest"] = stable_write_json(OUTPUTS["manifest"], manifest_payload)
write_sidecar(OUTPUTS["manifest"])

if not sidecar_is_valid(OUTPUTS["manifest"]):
    raise AssertionError("Cell 7A2 manifest sidecar verification failed.")

manifest_readback = json.loads(OUTPUTS["manifest"].read_text(encoding="utf-8"))
if manifest_readback.get("decision") != final_decision:
    raise AssertionError("Cell 7A2 manifest decision readback failed.")

for path in OUTPUTS.values():
    if not sidecar_is_valid(path):
        raise AssertionError(f"Fresh output verification failed: {path}")


# --------------------------------------------------------------------------------------------------
# 19. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

separator = "=" * 144

print("\n" + separator)
print("EXPERIMENT 2 — STAGE 7A — CELL 7A2")
print("FROZEN T1 FEATURE RECONSTRUCTION AND MODEL-APPLICABILITY PREFLIGHT")
print(separator)
print(f"Notebook                                      : {NOTEBOOK_NAME}")
print(f"Project root                                  : {ROOT}")

print("\nUPSTREAM AUTHORIZATION")
print(f"Cell 7A1 manifest SHA-256                     : {sha256_file(CELL_7A1_MANIFEST)}")
print("Cell 7A1 terminal PASS verified               : YES")
print("Model fitting authorized                      : NO")
print("T1 scoring authorized in Cell 7A2             : NO")
print("Embedding construction authorized             : NO")
print("LLM generation authorized                     : NO")

print("\nFROZEN INPUT VERIFICATION")
print(f"Frozen artifacts verified                     : {len(observed_hashes)}/{len(EXPECTED_HASHES)}")
print(f"T1 Parquet SHA-256                            : {sha256_file(T1_PARQUET)}")
print(f"Stage 4A feature table SHA-256                 : {sha256_file(STAGE4A_FEATURE_TABLE)}")
print(f"Stage 4B transform table SHA-256               : {sha256_file(STAGE4B_TABLE)}")
print(f"Stage 4C Full-GES model SHA-256                : {sha256_file(FULL_MODEL_PATH)}")
print(f"Stage 4C No-star model SHA-256                 : {sha256_file(NO_STAR_MODEL_PATH)}")
print(f"Stage 6A policy SHA-256                        : {sha256_file(STAGE6A_POLICY)}")

print("\nFROZEN TRANSFORMATION REPRODUCTION")
for row in formula_audit.to_dict("records"):
    print(f"{row['transformation']:<46}: {int(row['mismatch_count']):,} mismatches")

print("\nT1 FEATURE RECONSTRUCTION — IN MEMORY ONLY")
print(f"T1 rows                                       : {len(t1):,}")
print(f"Unique RCV accessions                         : {rcv_t1.nunique(dropna=False):,}")
print(f"Recency missing                               : {int(recency_score_t1.isna().sum()):,}")
print(f"Entropy missing                               : {entropy_missing_rows:,}")
print(f"Entropy JSON parse errors                     : {entropy_parse_errors:,}")
print(f"SCV/group-count mismatches                    : {scv_group_count_mismatches:,}")
print(f"Post-cutoff dates                             : {post_cutoff_dates:,}")
print(f"Infinite derived feature values               : {all_infinite_feature_values:,}")
print("Row-level reconstructed feature table saved   : NO")

print("\nFROZEN MODEL APPLICABILITY")
for row in model_inventory.to_dict("records"):
    print(
        f"{row['model']:<46}: "
        f"{int(row['t1_preprocessing_rows']):,} rows × "
        f"{int(row['t1_preprocessing_columns'])} features | "
        f"finite={row['t1_preprocessing_all_finite']} | scored=NO"
    )

print("\nCLASSIFICATION-AXIS ACCOUNTING")
for axis, expected in EXPECTED_T1_AXIS_COUNTS.items():
    print(f"{axis:<46}: {int(observed_axis_counts.get(axis, 0)):,}")

print("\nCELL 7A2 FROZEN OUTPUTS")
for label, key in [
    ("Frozen artifact inventory", "artifact_inventory"),
    ("T0 transformation formula audit", "formula_audit"),
    ("T1 feature applicability inventory", "feature_inventory"),
    ("Frozen model pipeline inventory", "model_inventory"),
    ("Classification-axis applicability", "axis_inventory"),
    ("Preflight report", "preflight_report"),
    ("QC record", "qc"),
    ("Manifest", "manifest"),
]:
    path = OUTPUTS[key]
    print(f"{label:<46}: {path}")
    print(f"{'SHA-256':<46}: {sha256_file(path)}")

print(f"\nQC checks                                      : {passed_checks}/{len(qc_checks)} PASS")

print("\nSCIENTIFIC OPERATIONS")
print("Full-GES fitted or refitted                    : NO")
print("No-star GES fitted or refitted                 : NO")
print("Full-GES T1 score generated                    : NO")
print("No-star T1 score generated                     : NO")
print("Combined-metadata T1 score generated           : NO")
print("Threshold or weight optimization               : NO")
print("RAG corpus constructed                         : NO")
print("Embeddings constructed                         : NO")
print("LLM called                                     : NO")

print("\nNEXT AUTHORIZED CELL")
if all_checks_passed:
    print("Cell 7A3                                      : Frozen T1 Full-GES, No-star-GES,")
    print("                                                 and combined-metadata score")
    print("                                                 materialization and checksum freeze")
    print("Model fitting                                 : PROHIBITED")
    print("Threshold or weight optimization              : PROHIBITED")
    print("Embedding construction                        : PROHIBITED")
    print("LLM generation                                : PROHIBITED")
else:
    print("Cell 7A3                                      : NOT AUTHORIZED")

print(f"\nFINAL DECISION                                : {final_decision}")
print(separator)

if not all_checks_passed:
    raise RuntimeError(
        "Cell 7A2 preflight failed. Failed checks:\n- "
        + "\n- ".join(failed_checks)
    )

CELL 7A2 PREFLIGHT FAILED BEFORE OUTPUT FREEZE
Failed checks:
 - t0_submitter_min_constant_verified
 - t0_submitter_max_constant_verified


RuntimeError: Cell 7A2 preflight failed before any Cell 7A2 artifact was written.

In [9]:
# ==================================================================================================
# EXPERIMENT 2 — STAGE 7A — CELL 7A2 — CORRECTED V5 COMPLETE CELL
# FROZEN T1 FEATURE RECONSTRUCTION AND MODEL-APPLICABILITY PREFLIGHT
#
# This cell verifies the frozen Stage 4A/4B/4C packages, reconstructs the six T1 model inputs
# in memory, and verifies that the frozen preprocessing pipelines can transform all T1 rows.
#
# It DOES NOT fit/refit a model, generate GES scores, apply thresholds, create embeddings,
# build a RAG corpus, create questions, or call an LLM.
# ==================================================================================================

from collections import OrderedDict, Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import re
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE AND PROJECT PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG.ipynb"
CELL_ID = "7A2"

STAGE4_DATA_DIR = ROOT / "data_processed" / "stage4_ges"
STAGE4_MODEL_DIR = ROOT / "models" / "stage4_ges"
STAGE4_CONFIG_DIR = ROOT / "configs" / "stage4_ges"
STAGE6_CONFIG_DIR = ROOT / "configs" / "stage6_temporal_validation"
STAGE7_CONFIG_DIR = ROOT / "configs" / "stage7_rag"
STAGE7_TABLE_DIR = ROOT / "outputs" / "tables" / "stage7_rag"
STAGE7_QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"

for directory in [STAGE7_CONFIG_DIR, STAGE7_TABLE_DIR, STAGE7_QC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 2. EXACT FROZEN INPUT PATHS
# --------------------------------------------------------------------------------------------------

CELL_7A1_MANIFEST = (
    STAGE7_CONFIG_DIR / "cell_7a1_t1_corpus_source_preflight_manifest_v1.json"
)

T1_PARQUET = ROOT / "data_interim" / "t1_rcv_target_genes_harmonized_v1.parquet"
T1_FREEZE_MANIFEST = (
    ROOT / "configs" / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
)

STAGE4A_FEATURE_TABLE = (
    STAGE4_DATA_DIR / "stage4a_t0_ges_baseline_features_v1.parquet"
)
STAGE4A_FEATURE_SPEC = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_specification_v1.json"
)
STAGE4A_QC = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_qc_report_v1.json"
)
STAGE4A_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_freeze_manifest_v1.json"
)

STAGE4B_TABLE = (
    STAGE4_DATA_DIR / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)
STAGE4B_TRANSFORM_PARAMETERS = (
    STAGE4_CONFIG_DIR / "stage4b_t0_feature_transform_parameters_v1.json"
)
STAGE4B_WEAK_LABEL_RULES = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_rules_v1.json"
)
STAGE4B_QC = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_qc_report_v1.json"
)
STAGE4B_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_freeze_manifest_v1.json"
)

FULL_MODEL_PATH = (
    STAGE4_MODEL_DIR / "stage4c_full_ges_logistic_model_v1.joblib"
)
NO_STAR_MODEL_PATH = (
    STAGE4_MODEL_DIR / "stage4c_no_star_ges_logistic_model_v1.joblib"
)
STAGE4C_MODEL_SPEC = (
    STAGE4_CONFIG_DIR / "stage4c_ges_model_specification_v1.json"
)
STAGE4C_QC = (
    STAGE4_CONFIG_DIR / "stage4c_ges_model_qc_report_v1.json"
)
STAGE4C_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4c_ges_model_freeze_manifest_v1.json"
)

STAGE6A_POLICY = (
    STAGE6_CONFIG_DIR / "stage6a_comparator_score_policy_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 3. EXPECTED HASHES AND LOCKED COUNTS
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = OrderedDict([
    ("cell_7a1_manifest", "84e509d97f01fb8dc0b6ad0c7e24de762e8923b9068fc835bf328105f79e946d"),
    ("t1_parquet", "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"),
    ("t1_freeze_manifest", "7eaeff0fee3df96973130f721d6c1f2a02fd9108e85af7b2743cdd7750a4372e"),
    ("stage4a_feature_table", "c100b3781e6801425f622f5d091376abfe0939e48c0a792af32f6eebe6401f16"),
    ("stage4a_feature_spec", "fc00146efe5da9b3fbe740bb42ca99d157252cdefc045650f9e88d54d8fcfa8b"),
    ("stage4a_qc", "4bf72af66aa800fa9f12fdc8598bd06ebc859bb31c97ad97cf7fc11ef1614cf1"),
    ("stage4a_manifest", "2b844ef2dbc0c3e5e57493886a7533587350e1b4b4c76fc9d445ecda8a528fa0"),
    ("stage4b_table", "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8"),
    ("stage4b_transform_parameters", "baeab167e19381138f93c51ba2fa00e4eeed684b36122c80cd2bf9fc4fe4b08d"),
    ("stage4b_weak_label_rules", "3d78e66cea1fed5c75ef1cab1b7cf44d3d3d7bfae50909173bae6c3a0e0bff61"),
    ("stage4b_qc", "03b14efb6d297fd517043c8ff952977715415723f191d079d968c111368d31a9"),
    ("stage4b_manifest", "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f"),
    ("stage4c_full_model", "0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30"),
    ("stage4c_no_star_model", "6c3fe4fc7fe8fdde7b8f0f0d608c48e66a07945effb8c67c6b98d35e1955257c"),
    ("stage4c_model_spec", "d754c715c990f42cecd64259ca2c420427b9602c669dc7be9e80dee9554f61f6"),
    ("stage4c_qc", "3a1a90d3a8946fd35bde52324a625077f3127ac57f231b20d206d70b2c678ec7"),
    ("stage4c_manifest", "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee"),
    ("stage6a_policy", "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"),
])

ARTIFACT_PATHS = OrderedDict([
    ("cell_7a1_manifest", CELL_7A1_MANIFEST),
    ("t1_parquet", T1_PARQUET),
    ("t1_freeze_manifest", T1_FREEZE_MANIFEST),
    ("stage4a_feature_table", STAGE4A_FEATURE_TABLE),
    ("stage4a_feature_spec", STAGE4A_FEATURE_SPEC),
    ("stage4a_qc", STAGE4A_QC),
    ("stage4a_manifest", STAGE4A_MANIFEST),
    ("stage4b_table", STAGE4B_TABLE),
    ("stage4b_transform_parameters", STAGE4B_TRANSFORM_PARAMETERS),
    ("stage4b_weak_label_rules", STAGE4B_WEAK_LABEL_RULES),
    ("stage4b_qc", STAGE4B_QC),
    ("stage4b_manifest", STAGE4B_MANIFEST),
    ("stage4c_full_model", FULL_MODEL_PATH),
    ("stage4c_no_star_model", NO_STAR_MODEL_PATH),
    ("stage4c_model_spec", STAGE4C_MODEL_SPEC),
    ("stage4c_qc", STAGE4C_QC),
    ("stage4c_manifest", STAGE4C_MANIFEST),
    ("stage6a_policy", STAGE6A_POLICY),
])

EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36
EXPECTED_STAGE4A_ROWS = 71_659
EXPECTED_STAGE4A_COLUMNS = 30
EXPECTED_STAGE4B_ROWS = 71_659
EXPECTED_STAGE4B_COLUMNS = 43

EXPECTED_T1_GENE_COUNTS = {
    "BRCA1": 32_603,
    "BRCA2": 49_221,
    "MLH1": 13_684,
    "EGFR": 5_412,
}

EXPECTED_T1_AXIS_COUNTS = {
    "GermlineClassification": 97_526,
    "OncogenicityClassification": 52,
    "SomaticClinicalImpact": 25,
    "NoClassification": 3_317,
}

EXPECTED_CELL_7A1_DECISION = (
    "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_CHECKSUM_PROTECTED_"
    "TOP_LEVEL_AND_NESTED_SCHEMA_INVENTORIED_EVIDENCE_PACKET_"
    "DERIVABILITY_AUDITED_STAGE7A2_PREFLIGHT_ONLY"
)

MAXIMUM_OBSERVATION_WINDOW_DAYS = 10_516.0
SUBMITTER_LOG_MIN = 0.6931471805599453
SUBMITTER_LOG_MAX = 3.4339872044851463
T1_CUTOFF = pd.Timestamp("2025-12-27")

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

EXPECTED_MODEL_SETTINGS = {
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "class_weight": None,
    "max_iter": 2_000,
    "tol": 1e-6,
    "fit_intercept": True,
    "random_state": 42,
}


# --------------------------------------------------------------------------------------------------
# 4. OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

OUTPUTS = OrderedDict([
    ("artifact_inventory", STAGE7_TABLE_DIR / "cell_7a2_frozen_artifact_inventory_v1.csv"),
    ("formula_audit", STAGE7_TABLE_DIR / "cell_7a2_t0_frozen_transformation_formula_audit_v1.csv"),
    ("feature_inventory", STAGE7_TABLE_DIR / "cell_7a2_t1_feature_applicability_inventory_v1.csv"),
    ("model_inventory", STAGE7_TABLE_DIR / "cell_7a2_frozen_model_pipeline_inventory_v1.csv"),
    ("axis_inventory", STAGE7_TABLE_DIR / "cell_7a2_t1_classification_axis_applicability_v1.csv"),
    ("preflight_report", STAGE7_QC_DIR / "cell_7a2_feature_reconstruction_model_applicability_preflight_v1.json"),
    ("qc", STAGE7_QC_DIR / "cell_7a2_feature_reconstruction_model_applicability_qc_v1.json"),
    ("manifest", STAGE7_CONFIG_DIR / "cell_7a2_feature_reconstruction_model_applicability_manifest_v1.json"),
])


# --------------------------------------------------------------------------------------------------
# 5. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return Path(str(path) + ".sha256")


def read_sidecar_hash(path: Path) -> str:
    text = Path(path).read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def sidecar_is_valid(path: Path) -> bool:
    path = Path(path)
    sc = sidecar_path(path)
    return path.exists() and sc.exists() and read_sidecar_hash(sc) == sha256_file(path)


def verify_exact_hash(label: str, path: Path, expected: str) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required frozen artifact for {label}:\n{path}")
    observed = sha256_file(path)
    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}.\n"
            f"Expected: {expected}\nObserved: {observed}\nPath: {path}"
        )
    return observed


def json_native(value):
    if isinstance(value, dict):
        return {str(key): json_native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_native(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, np.ndarray):
        return [json_native(item) for item in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    proposed_hash = sha256_file(temporary)

    if path.exists():
        existing_hash = sha256_file(path)
        if existing_hash != proposed_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(
                "Refusing to overwrite a nonidentical frozen Cell 7A2 artifact.\n"
                f"Path: {path}\nExisting: {existing_hash}\nProposed: {proposed_hash}"
            )
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)

    return sha256_file(path)


def stable_write_json(path: Path, payload: dict) -> str:
    data = (
        json.dumps(
            json_native(payload),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    data = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.12g",
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def write_sidecar(path: Path) -> str:
    path = Path(path)
    payload = f"{sha256_file(path)}  {path.name}\n".encode("utf-8")
    stable_write_bytes(sidecar_path(path), payload)
    return sha256_file(sidecar_path(path))


def resolve_column(columns, aliases, label, required=True):
    exact = {str(column).lower(): str(column) for column in columns}
    for alias in aliases:
        if alias.lower() in exact:
            return exact[alias.lower()]
    if required:
        raise KeyError(
            f"Could not resolve required column '{label}'.\n"
            f"Aliases checked: {aliases}\nAvailable columns: {list(columns)}"
        )
    return None


def parse_json_value(value):
    if value is None or value is pd.NA:
        return None, "missing"
    try:
        if pd.isna(value):
            return None, "missing"
    except Exception:
        pass
    if isinstance(value, (dict, list)):
        return value, "native"
    text = str(value).strip()
    if not text:
        return None, "blank"
    try:
        return json.loads(text), "parsed"
    except json.JSONDecodeError:
        return None, "parse_error"


def normalize_gene(value) -> str:
    allowed = {"BRCA1", "BRCA2", "MLH1", "EGFR"}
    parsed, status = parse_json_value(value)
    if status == "parse_error":
        parsed = [str(value).strip()]
    if isinstance(parsed, str):
        parsed = [parsed]
    if not isinstance(parsed, list):
        return ""
    genes = sorted({
        str(gene).strip().upper()
        for gene in parsed
        if str(gene).strip().upper() in allowed
    })
    return genes[0] if len(genes) == 1 else ""


def boolish_to_float(value):
    if value is None or value is pd.NA:
        return np.nan
    try:
        if pd.isna(value):
            return np.nan
    except Exception:
        pass
    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        number = float(value)
        return number if number in {0.0, 1.0} else np.nan
    text = str(value).strip().lower()
    if text in {"true", "t", "yes", "y", "1"}:
        return 1.0
    if text in {"false", "f", "no", "n", "0"}:
        return 0.0
    return np.nan


def numeric_dict_sum(mapping):
    if not isinstance(mapping, dict):
        return np.nan, 1
    total = 0.0
    invalid = 0
    for value in mapping.values():
        try:
            number = float(value)
        except (TypeError, ValueError):
            invalid += 1
            continue
        if not np.isfinite(number) or number < 0:
            invalid += 1
            continue
        total += number
    return total, invalid


def normalized_entropy_from_group_counts(value):
    parsed, status = parse_json_value(value)
    if status == "parse_error":
        return np.nan, np.nan, 0, "parse_error"
    if parsed is None:
        return np.nan, np.nan, 0, status
    total, invalid = numeric_dict_sum(parsed)
    if invalid > 0:
        return np.nan, total, 0, "invalid_numeric_value"
    positive_counts = np.asarray(
        [float(item) for item in parsed.values() if float(item) > 0],
        dtype=float,
    )
    if len(positive_counts) == 0:
        return np.nan, total, 0, "no_positive_counts"
    if len(positive_counts) == 1:
        return 0.0, total, 1, "single_group"
    probabilities = positive_counts / positive_counts.sum()
    entropy = -float(np.sum(probabilities * np.log(probabilities)))
    normalized = float(np.clip(entropy / math.log(len(positive_counts)), 0.0, 1.0))
    return normalized, total, int(len(positive_counts)), "derived"


def extract_pipeline(artifact, label):
    if isinstance(artifact, Pipeline):
        return artifact, "<top-level>"

    preferred_keys = [
        "pipeline", "model_pipeline", "fitted_pipeline",
        "sklearn_pipeline", "model", "estimator", "classifier",
    ]

    if isinstance(artifact, dict):
        for key in preferred_keys:
            if key in artifact and isinstance(artifact[key], Pipeline):
                return artifact[key], f"[{key!r}]"

    matches = []
    visited = set()

    def walk(obj, location="<top-level>", depth=0):
        if depth > 8 or id(obj) in visited:
            return
        visited.add(id(obj))
        if isinstance(obj, Pipeline):
            matches.append((location, obj))
            return
        if isinstance(obj, dict):
            for key, item in obj.items():
                walk(item, f"{location}[{key!r}]", depth + 1)
        elif isinstance(obj, (list, tuple)):
            for index, item in enumerate(obj):
                walk(item, f"{location}[{index}]", depth + 1)
        else:
            for attribute in preferred_keys:
                if hasattr(obj, attribute):
                    try:
                        walk(getattr(obj, attribute), f"{location}.{attribute}", depth + 1)
                    except Exception:
                        pass

    walk(artifact)
    unique = {}
    for location, pipeline in matches:
        unique.setdefault(id(pipeline), (location, pipeline))
    found = list(unique.values())

    if len(found) != 1:
        raise TypeError(
            f"Expected exactly one sklearn Pipeline in {label}; found {len(found)}."
        )
    return found[0][1], found[0][0]


def get_exact_component(pipeline, component_type):
    matches = [step for _, step in pipeline.steps if isinstance(step, component_type)]
    if len(matches) != 1:
        raise AssertionError(
            f"Expected exactly one {component_type.__name__}; found {len(matches)}."
        )
    return matches[0]


def transformed_values_are_finite(values) -> bool:
    if sparse.issparse(values):
        return bool(np.isfinite(values.data).all())
    return bool(np.isfinite(np.asarray(values)).all())


def model_settings_match(classifier: LogisticRegression) -> bool:
    return bool(
        classifier.penalty == EXPECTED_MODEL_SETTINGS["penalty"]
        and np.isclose(float(classifier.C), EXPECTED_MODEL_SETTINGS["C"])
        and classifier.solver == EXPECTED_MODEL_SETTINGS["solver"]
        and classifier.class_weight is EXPECTED_MODEL_SETTINGS["class_weight"]
        and int(classifier.max_iter) == EXPECTED_MODEL_SETTINGS["max_iter"]
        and np.isclose(float(classifier.tol), EXPECTED_MODEL_SETTINGS["tol"])
        and bool(classifier.fit_intercept) == EXPECTED_MODEL_SETTINGS["fit_intercept"]
        and classifier.random_state == EXPECTED_MODEL_SETTINGS["random_state"]
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL FROZEN INPUT ARTIFACTS
# --------------------------------------------------------------------------------------------------

observed_hashes = OrderedDict()

for key, path in ARTIFACT_PATHS.items():
    observed_hashes[key] = verify_exact_hash(key, path, EXPECTED_HASHES[key])

if not sidecar_is_valid(CELL_7A1_MANIFEST):
    raise AssertionError("Cell 7A1 manifest sidecar verification failed.")

cell_7a1_payload = json.loads(CELL_7A1_MANIFEST.read_text(encoding="utf-8"))
if cell_7a1_payload.get("terminal_decision") != EXPECTED_CELL_7A1_DECISION:
    raise AssertionError("Cell 7A1 terminal decision does not authorize Cell 7A2.")

if cell_7a1_payload.get("next_authorized_cell", {}).get("cell_id") != "7A2":
    raise AssertionError("Cell 7A1 manifest does not identify Cell 7A2 as next authorized cell.")

immutable_hashes_before = {
    key: sha256_file(path)
    for key, path in ARTIFACT_PATHS.items()
}

# Load the cryptographically verified Stage 4B transformation parameters.
# The submitter-diversity normalization bounds were frozen from the actual T0 cohort
# and must be read from the frozen JSON rather than retyped as rounded literals.
stage4b_transform_payload = json.loads(
    STAGE4B_TRANSFORM_PARAMETERS.read_text(encoding="utf-8")
)

frozen_recency_parameters = stage4b_transform_payload.get(
    "recency_transformation",
    {},
)
frozen_submitter_parameters = stage4b_transform_payload.get(
    "submitter_diversity_transformation",
    {},
)

required_frozen_parameter_keys = {
    "maximum_observation_window_days":
        frozen_recency_parameters.get("maximum_observation_window_days"),
    "minimum_log_count":
        frozen_submitter_parameters.get("minimum_log_count"),
    "maximum_log_count":
        frozen_submitter_parameters.get("maximum_log_count"),
}

missing_frozen_parameter_keys = [
    key
    for key, value in required_frozen_parameter_keys.items()
    if value is None
]

if missing_frozen_parameter_keys:
    raise KeyError(
        "The frozen Stage 4B transformation-parameter JSON is missing: "
        f"{missing_frozen_parameter_keys}"
    )

MAXIMUM_OBSERVATION_WINDOW_DAYS = float(
    required_frozen_parameter_keys["maximum_observation_window_days"]
)
SUBMITTER_LOG_MIN = float(
    required_frozen_parameter_keys["minimum_log_count"]
)
SUBMITTER_LOG_MAX = float(
    required_frozen_parameter_keys["maximum_log_count"]
)

if not (
    np.isfinite(MAXIMUM_OBSERVATION_WINDOW_DAYS)
    and MAXIMUM_OBSERVATION_WINDOW_DAYS > 0
    and np.isfinite(SUBMITTER_LOG_MIN)
    and np.isfinite(SUBMITTER_LOG_MAX)
    and SUBMITTER_LOG_MAX > SUBMITTER_LOG_MIN
):
    raise ValueError(
        "Frozen Stage 4B transformation parameters are not numerically valid."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY PARQUET DIMENSIONS AND BUILD ARTIFACT INVENTORY
# --------------------------------------------------------------------------------------------------

t1_meta = pq.ParquetFile(T1_PARQUET).metadata
stage4a_meta = pq.ParquetFile(STAGE4A_FEATURE_TABLE).metadata
stage4b_meta = pq.ParquetFile(STAGE4B_TABLE).metadata

artifact_inventory_rows = []

for key, path in ARTIFACT_PATHS.items():
    row = {
        "artifact_key": key,
        "path": str(path),
        "file_name": path.name,
        "suffix": path.suffix.lower(),
        "expected_sha256": EXPECTED_HASHES[key],
        "observed_sha256": observed_hashes[key],
        "hash_verified": observed_hashes[key] == EXPECTED_HASHES[key],
        "bytes": int(path.stat().st_size),
        "sidecar_present": sidecar_path(path).exists(),
        "sidecar_verified": sidecar_is_valid(path) if sidecar_path(path).exists() else False,
        "rows": None,
        "columns": None,
    }
    if path.suffix.lower() == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        row["rows"] = int(metadata.num_rows)
        row["columns"] = int(metadata.num_columns)
    artifact_inventory_rows.append(row)

artifact_inventory = pd.DataFrame(artifact_inventory_rows)


# --------------------------------------------------------------------------------------------------
# 8. REPRODUCE THE FROZEN STAGE 4B TRANSFORMATIONS ON T0
#    USING RAW SOURCE FIELDS FROM STAGE 4A AND TRANSFORMED FIELDS FROM STAGE 4B
# --------------------------------------------------------------------------------------------------

# Stage 4B intentionally stores the transformed model features and weak-label outputs.
# The raw inputs used to create those transformed features remain in the frozen Stage 4A table.
# Therefore, this audit reads both tables and verifies exact row/key alignment before comparison.

stage4a_columns = pq.ParquetFile(STAGE4A_FEATURE_TABLE).schema_arrow.names
stage4b_columns = pq.ParquetFile(STAGE4B_TABLE).schema_arrow.names

stage4a_required_columns = [
    "t0_row_order",
    "rcv_accession",
    "recency_days",
    "recency_missing_flag",
    "unique_submitter_count",
    "log1p_unique_submitter_count",
    "aggregate_review_stars",
]

stage4b_required_columns = list(dict.fromkeys([
    "t0_row_order",
    "rcv_accession",
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
] + FULL_FEATURES))

missing_stage4a_columns = [
    column for column in stage4a_required_columns
    if column not in stage4a_columns
]
missing_stage4b_columns = [
    column for column in stage4b_required_columns
    if column not in stage4b_columns
]

if missing_stage4a_columns:
    raise KeyError(
        "Frozen Stage 4A table is missing required raw-source columns: "
        f"{missing_stage4a_columns}"
    )

if missing_stage4b_columns:
    raise KeyError(
        "Frozen Stage 4B table is missing required transformed columns: "
        f"{missing_stage4b_columns}"
    )

t0_raw = pd.read_parquet(
    STAGE4A_FEATURE_TABLE,
    columns=stage4a_required_columns,
).copy()

t0 = pd.read_parquet(
    STAGE4B_TABLE,
    columns=stage4b_required_columns,
).copy()

if len(t0_raw) != EXPECTED_STAGE4A_ROWS:
    raise RuntimeError(
        f"Loaded Stage 4A row count is {len(t0_raw):,}; "
        f"expected {EXPECTED_STAGE4A_ROWS:,}."
    )

if len(t0) != EXPECTED_STAGE4B_ROWS:
    raise RuntimeError(
        f"Loaded Stage 4B row count is {len(t0):,}; "
        f"expected {EXPECTED_STAGE4B_ROWS:,}."
    )

# Normalize and verify exact row-order/key alignment before using Stage 4A raw values
# to reproduce Stage 4B transformed values.
t0_raw_row_order = pd.to_numeric(
    t0_raw["t0_row_order"],
    errors="raise",
).astype("int64")

t0_row_order = pd.to_numeric(
    t0["t0_row_order"],
    errors="raise",
).astype("int64")

t0_raw_rcv = (
    t0_raw["rcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
)

t0_rcv = (
    t0["rcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
)

stage4a_row_order_valid = bool(
    t0_raw_row_order.nunique(dropna=False) == EXPECTED_STAGE4A_ROWS
    and np.array_equal(
        t0_raw_row_order.to_numpy(),
        np.arange(EXPECTED_STAGE4A_ROWS, dtype=np.int64),
    )
)

stage4b_row_order_valid = bool(
    t0_row_order.nunique(dropna=False) == EXPECTED_STAGE4B_ROWS
    and np.array_equal(
        t0_row_order.to_numpy(),
        np.arange(EXPECTED_STAGE4B_ROWS, dtype=np.int64),
    )
)

stage4a_stage4b_row_order_alignment_verified = bool(
    np.array_equal(
        t0_raw_row_order.to_numpy(),
        t0_row_order.to_numpy(),
    )
)

stage4a_stage4b_rcv_alignment_verified = bool(
    t0_raw_rcv.notna().all()
    and t0_rcv.notna().all()
    and not t0_raw_rcv.fillna("").eq("").any()
    and not t0_rcv.fillna("").eq("").any()
    and t0_raw_rcv.nunique(dropna=False) == EXPECTED_STAGE4A_ROWS
    and t0_rcv.nunique(dropna=False) == EXPECTED_STAGE4B_ROWS
    and np.array_equal(
        t0_raw_rcv.to_numpy(dtype=str),
        t0_rcv.to_numpy(dtype=str),
    )
)

if not stage4a_row_order_valid:
    raise RuntimeError("Frozen Stage 4A row-order verification failed.")

if not stage4b_row_order_valid:
    raise RuntimeError("Frozen Stage 4B row-order verification failed.")

if not stage4a_stage4b_row_order_alignment_verified:
    raise RuntimeError("Stage 4A and Stage 4B row-order alignment failed.")

if not stage4a_stage4b_rcv_alignment_verified:
    raise RuntimeError("Stage 4A and Stage 4B RCV-key alignment failed.")

# Recency transformation: raw recency_days comes from Stage 4A;
# saved recency_score comes from Stage 4B.
recency_days_t0 = pd.to_numeric(
    t0_raw["recency_days"],
    errors="coerce",
).astype(float)

saved_recency_t0 = pd.to_numeric(
    t0["recency_score"],
    errors="coerce",
).astype(float)

expected_recency_t0 = (
    1.0 - recency_days_t0 / MAXIMUM_OBSERVATION_WINDOW_DAYS
).clip(0.0, 1.0)
expected_recency_t0.loc[recency_days_t0.isna()] = np.nan

recency_formula_mismatches = int((~np.isclose(
    saved_recency_t0.to_numpy(),
    expected_recency_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True,
)).sum())

# Missingness was present in Stage 4A and was copied into Stage 4B.
saved_missing_stage4a_t0 = pd.to_numeric(
    t0_raw["recency_missing_flag"],
    errors="coerce",
).astype(float)

saved_missing_stage4b_t0 = pd.to_numeric(
    t0["recency_missing_flag"],
    errors="coerce",
).astype(float)

expected_missing_t0 = recency_days_t0.isna().astype(float)

recency_missing_stage4a_mismatches = int((~np.isclose(
    saved_missing_stage4a_t0.to_numpy(),
    expected_missing_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)).sum())

recency_missing_stage4b_mismatches = int((~np.isclose(
    saved_missing_stage4b_t0.to_numpy(),
    expected_missing_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)).sum())

recency_missing_mismatches = (
    recency_missing_stage4a_mismatches
    + recency_missing_stage4b_mismatches
)

# Submitter transformation: raw and log1p counts are stored in Stage 4A;
# the normalized submitter-diversity score is stored in Stage 4B.
submitter_count_t0 = pd.to_numeric(
    t0_raw["unique_submitter_count"],
    errors="raise",
).astype(float)

expected_log_submitter_t0 = np.log1p(submitter_count_t0)

saved_log_submitter_t0 = pd.to_numeric(
    t0_raw["log1p_unique_submitter_count"],
    errors="raise",
).astype(float)

log_submitter_mismatches = int((~np.isclose(
    saved_log_submitter_t0.to_numpy(),
    expected_log_submitter_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)).sum())

observed_log_min = float(expected_log_submitter_t0.min())
observed_log_max = float(expected_log_submitter_t0.max())

expected_submitter_score_t0 = (
    (expected_log_submitter_t0 - SUBMITTER_LOG_MIN)
    / (SUBMITTER_LOG_MAX - SUBMITTER_LOG_MIN)
).clip(0.0, 1.0)

saved_submitter_score_t0 = pd.to_numeric(
    t0["submitter_diversity_score"],
    errors="raise",
).astype(float)

submitter_formula_mismatches = int((~np.isclose(
    saved_submitter_score_t0.to_numpy(),
    expected_submitter_score_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)).sum())

# Review confidence is the Stage 4B copy of the Stage 4A review-star value.
saved_review_confidence_t0 = pd.to_numeric(
    t0["review_confidence"],
    errors="raise",
).astype(float)

expected_review_confidence_t0 = pd.to_numeric(
    t0_raw["aggregate_review_stars"],
    errors="raise",
).astype(float)

review_formula_mismatches = int((~np.isclose(
    saved_review_confidence_t0.to_numpy(),
    expected_review_confidence_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
)).sum())

formula_audit = pd.DataFrame([
    {
        "transformation": "stage4a_stage4b_row_and_key_alignment",
        "frozen_formula": (
            "exact zero-based t0_row_order and exact RCV sequence alignment "
            "between frozen Stage 4A and Stage 4B"
        ),
        "rows_audited": len(t0),
        "mismatch_count": int(
            (not stage4a_stage4b_row_order_alignment_verified)
            or (not stage4a_stage4b_rcv_alignment_verified)
        ),
        "exactly_reproduced": bool(
            stage4a_stage4b_row_order_alignment_verified
            and stage4a_stage4b_rcv_alignment_verified
        ),
    },
    {
        "transformation": "recency_score",
        "frozen_formula": f"clip(1 - recency_days / {MAXIMUM_OBSERVATION_WINDOW_DAYS:.17g}, 0, 1); missing remains missing",
        "rows_audited": len(t0),
        "mismatch_count": recency_formula_mismatches,
        "exactly_reproduced": recency_formula_mismatches == 0,
    },
    {
        "transformation": "recency_missing_flag",
        "frozen_formula": (
            "1 when Stage 4A recency_days is missing; otherwise 0; "
            "verified in both Stage 4A and Stage 4B"
        ),
        "rows_audited": len(t0),
        "mismatch_count": recency_missing_mismatches,
        "exactly_reproduced": recency_missing_mismatches == 0,
    },
    {
        "transformation": "log1p_unique_submitter_count",
        "frozen_formula": "log1p(Stage 4A unique_submitter_count)",
        "rows_audited": len(t0),
        "mismatch_count": log_submitter_mismatches,
        "exactly_reproduced": log_submitter_mismatches == 0,
    },
    {
        "transformation": "submitter_diversity_score",
        "frozen_formula": (
            "clip((Stage 4A log1p_unique_submitter_count - "
            f"{SUBMITTER_LOG_MIN:.17g}) / "
            f"({SUBMITTER_LOG_MAX:.17g} - {SUBMITTER_LOG_MIN:.17g}), 0, 1)"
        ),
        "rows_audited": len(t0),
        "mismatch_count": submitter_formula_mismatches,
        "exactly_reproduced": submitter_formula_mismatches == 0,
    },
    {
        "transformation": "review_confidence",
        "frozen_formula": "Stage 4B review_confidence = Stage 4A aggregate_review_stars",
        "rows_audited": len(t0),
        "mismatch_count": review_formula_mismatches,
        "exactly_reproduced": review_formula_mismatches == 0,
    },
])


# --------------------------------------------------------------------------------------------------
# 9. RESOLVE T1 COLUMNS AND RECONSTRUCT FROZEN MODEL INPUTS IN MEMORY
# --------------------------------------------------------------------------------------------------

t1_schema = pq.ParquetFile(T1_PARQUET).schema_arrow.names

t1_columns = {
    "rcv_accession": resolve_column(
        t1_schema,
        ["rcv_accession", "rcv"],
        "T1 RCV accession",
    ),
    "target_gene": resolve_column(
        t1_schema,
        ["target_genes_json", "target_gene", "gene_symbol", "gene"],
        "T1 target gene",
    ),
    "embedded_cutoff": resolve_column(
        t1_schema,
        ["embedded_data_cutoff_date", "embedded_data_cutoff", "embedded_cutoff", "release_embedded_cutoff", "data_cutoff"],
        "T1 embedded cutoff",
    ),
    "aggregate_last_evaluated": resolve_column(
        t1_schema,
        ["aggregate_last_evaluated", "last_evaluated", "aggregate_date_last_evaluated"],
        "T1 aggregate last evaluated",
    ),
    "unique_submitter_count": resolve_column(
        t1_schema,
        ["unique_submitter_count_xml", "unique_submitter_count", "aggregate_unique_submitter_count"],
        "T1 unique submitter count",
    ),
    "aggregate_review_stars": resolve_column(
        t1_schema,
        ["aggregate_review_stars", "review_stars"],
        "T1 aggregate review stars",
    ),
    "aggregate_conflict_flag": resolve_column(
        t1_schema,
        ["aggregate_conflict_flag", "aggregate_conflict", "conflict_flag"],
        "T1 aggregate conflict flag",
    ),
    "classification_group_counts": resolve_column(
        t1_schema,
        ["scv_classification_group_counts_json", "scv_group_counts_json", "classification_group_counts_json"],
        "T1 SCV classification-group counts",
    ),
    "classification_axis": resolve_column(
        t1_schema,
        ["aggregate_classification_axis", "classification_axis"],
        "T1 aggregate classification axis",
    ),
    "scv_count": resolve_column(
        t1_schema,
        ["scv_count_xml", "scv_count"],
        "T1 SCV count",
    ),
}

t1_load_columns = list(dict.fromkeys(t1_columns.values()))
t1 = pd.read_parquet(T1_PARQUET, columns=t1_load_columns).copy()

rcv_t1 = t1[t1_columns["rcv_accession"]].astype("string").str.strip().str.upper()
gene_t1 = t1[t1_columns["target_gene"]].map(normalize_gene).astype("string")
axis_t1 = t1[t1_columns["classification_axis"]].astype("string").str.strip()
cutoff_t1 = (
    t1[t1_columns["embedded_cutoff"]]
    .astype("string")
    .str.strip()
    .str.slice(0, 10)
)

last_evaluated_t1 = pd.to_datetime(
    t1[t1_columns["aggregate_last_evaluated"]],
    errors="coerce",
    utc=True,
).dt.tz_convert(None).dt.normalize()

recency_days_t1 = (T1_CUTOFF - last_evaluated_t1).dt.days.astype(float)
post_cutoff_dates = int((recency_days_t1 < 0).sum())
recency_missing_flag_t1 = recency_days_t1.isna().astype(float)
recency_score_t1 = (1.0 - recency_days_t1 / MAXIMUM_OBSERVATION_WINDOW_DAYS).clip(0.0, 1.0)
recency_score_t1.loc[recency_days_t1.isna()] = np.nan

submitter_count_t1 = pd.to_numeric(
    t1[t1_columns["unique_submitter_count"]], errors="coerce"
).astype(float)
log_submitter_t1 = np.log1p(submitter_count_t1)
submitter_score_t1 = (
    (log_submitter_t1 - SUBMITTER_LOG_MIN)
    / (SUBMITTER_LOG_MAX - SUBMITTER_LOG_MIN)
).clip(0.0, 1.0)

review_confidence_t1 = pd.to_numeric(
    t1[t1_columns["aggregate_review_stars"]], errors="coerce"
).astype(float)
conflict_t1 = (
    t1[t1_columns["aggregate_conflict_flag"]]
    .map(boolish_to_float)
    .astype(float)
)

entropy_values = []
group_count_totals = []
nonzero_group_counts = []
entropy_statuses = []

for value in t1[t1_columns["classification_group_counts"]].tolist():
    entropy, total, nonzero_groups, status = normalized_entropy_from_group_counts(value)
    entropy_values.append(entropy)
    group_count_totals.append(total)
    nonzero_group_counts.append(nonzero_groups)
    entropy_statuses.append(status)

entropy_t1 = pd.Series(entropy_values, index=t1.index, dtype=float)
group_count_total_t1 = pd.Series(group_count_totals, index=t1.index, dtype=float)
entropy_status_t1 = pd.Series(entropy_statuses, index=t1.index, dtype="string")

saved_scv_count_t1 = pd.to_numeric(
    t1[t1_columns["scv_count"]], errors="coerce"
).astype(float)

comparable_scv_rows = saved_scv_count_t1.notna() & group_count_total_t1.notna()
scv_group_count_mismatches = int((~np.isclose(
    saved_scv_count_t1.loc[comparable_scv_rows].to_numpy(),
    group_count_total_t1.loc[comparable_scv_rows].to_numpy(),
    rtol=0.0,
    atol=0.0,
)).sum())

t1_features = pd.DataFrame({
    "recency_score": recency_score_t1,
    "recency_missing_flag": recency_missing_flag_t1,
    "submitter_diversity_score": submitter_score_t1,
    "review_confidence": review_confidence_t1,
    "aggregate_conflict_flag": conflict_t1,
    "scv_group_entropy_normalized": entropy_t1,
})

# The row-level T1 feature table remains in memory only.


# --------------------------------------------------------------------------------------------------
# 10. VERIFY FROZEN MODEL PIPELINES AND PREPROCESS T1 WITHOUT SCORING
# --------------------------------------------------------------------------------------------------

full_artifact = joblib.load(FULL_MODEL_PATH)
no_star_artifact = joblib.load(NO_STAR_MODEL_PATH)

full_pipeline, full_pipeline_location = extract_pipeline(full_artifact, "Full-GES artifact")
no_star_pipeline, no_star_pipeline_location = extract_pipeline(no_star_artifact, "No-star-GES artifact")

model_definitions = OrderedDict([
    ("full_ges", {
        "display": "Full GES",
        "path": FULL_MODEL_PATH,
        "artifact": full_artifact,
        "pipeline": full_pipeline,
        "pipeline_location": full_pipeline_location,
        "features": FULL_FEATURES,
    }),
    ("no_star_ges", {
        "display": "No-star GES",
        "path": NO_STAR_MODEL_PATH,
        "artifact": no_star_artifact,
        "pipeline": no_star_pipeline,
        "pipeline_location": no_star_pipeline_location,
        "features": NO_STAR_FEATURES,
    }),
])

model_inventory_rows = []
model_results = {}

for model_key, definition in model_definitions.items():
    pipeline = definition["pipeline"]
    expected_features = definition["features"]

    imputer = get_exact_component(pipeline, SimpleImputer)
    scaler = get_exact_component(pipeline, StandardScaler)
    classifier = get_exact_component(pipeline, LogisticRegression)

    feature_names = (
        list(map(str, pipeline.feature_names_in_))
        if hasattr(pipeline, "feature_names_in_")
        else []
    )

    feature_order_verified = (
        feature_names == expected_features
        if feature_names
        else int(getattr(pipeline, "n_features_in_", len(expected_features))) == len(expected_features)
    )

    fitted_state_verified = bool(
        hasattr(imputer, "statistics_")
        and hasattr(scaler, "mean_")
        and hasattr(scaler, "scale_")
        and hasattr(classifier, "coef_")
        and hasattr(classifier, "intercept_")
        and hasattr(classifier, "classes_")
        and np.isfinite(np.asarray(imputer.statistics_, dtype=float)).all()
        and np.isfinite(np.asarray(scaler.mean_, dtype=float)).all()
        and np.isfinite(np.asarray(scaler.scale_, dtype=float)).all()
        and np.isfinite(np.asarray(classifier.coef_, dtype=float)).all()
        and np.isfinite(np.asarray(classifier.intercept_, dtype=float)).all()
        and sorted(np.asarray(classifier.classes_).tolist()) == [0, 1]
    )

    settings_verified = model_settings_match(classifier)

    preprocessing_pipeline = Pipeline(pipeline.steps[:-1])
    model_input = t1_features[expected_features].copy()
    transformed = preprocessing_pipeline.transform(model_input)

    transformed_shape = transformed.shape
    transformed_finite = transformed_values_are_finite(transformed)

    model_results[model_key] = {
        "rows": int(transformed_shape[0]),
        "columns": int(transformed_shape[1]),
        "all_finite": bool(transformed_finite),
        "feature_order_verified": bool(feature_order_verified),
        "fitted_state_verified": bool(fitted_state_verified),
        "settings_verified": bool(settings_verified),
    }

    model_inventory_rows.append({
        "model_key": model_key,
        "model": definition["display"],
        "artifact_path": str(definition["path"]),
        "artifact_sha256": sha256_file(definition["path"]),
        "pipeline_location": definition["pipeline_location"],
        "pipeline_steps": ";".join(
            f"{name}:{type(step).__name__}" for name, step in pipeline.steps
        ),
        "expected_features": ";".join(expected_features),
        "pipeline_feature_names": ";".join(feature_names),
        "feature_order_verified": feature_order_verified,
        "fitted_state_verified": fitted_state_verified,
        "imputer_strategy": str(imputer.strategy),
        "classifier_solver": classifier.solver,
        "classifier_penalty": classifier.penalty,
        "classifier_C": float(classifier.C),
        "classifier_max_iter": int(classifier.max_iter),
        "classifier_tol": float(classifier.tol),
        "classifier_class_weight": str(classifier.class_weight),
        "classifier_fit_intercept": bool(classifier.fit_intercept),
        "classifier_random_state": classifier.random_state,
        "classifier_settings_verified": settings_verified,
        "t1_preprocessing_rows": int(transformed_shape[0]),
        "t1_preprocessing_columns": int(transformed_shape[1]),
        "t1_preprocessing_all_finite": transformed_finite,
        "fit_called_in_cell_7a2": False,
        "predict_called_in_cell_7a2": False,
        "predict_proba_called_in_cell_7a2": False,
        "score_created_in_cell_7a2": False,
    })

    del transformed, model_input, preprocessing_pipeline

model_inventory = pd.DataFrame(model_inventory_rows)


# --------------------------------------------------------------------------------------------------
# 11. FEATURE AND CLASSIFICATION-AXIS INVENTORIES
# --------------------------------------------------------------------------------------------------

feature_inventory_rows = []

for feature in FULL_FEATURES:
    t0_values = pd.to_numeric(t0[feature], errors="coerce").astype(float)
    t1_values = pd.to_numeric(t1_features[feature], errors="coerce").astype(float)

    t0_nonmissing = t0_values.dropna()
    t1_nonmissing = t1_values.dropna()

    t0_min = float(t0_nonmissing.min()) if len(t0_nonmissing) else np.nan
    t0_max = float(t0_nonmissing.max()) if len(t0_nonmissing) else np.nan
    t1_min = float(t1_nonmissing.min()) if len(t1_nonmissing) else np.nan
    t1_max = float(t1_nonmissing.max()) if len(t1_nonmissing) else np.nan

    domain_upper = 4.0 if feature == "review_confidence" else 1.0

    feature_inventory_rows.append({
        "feature": feature,
        "used_by_full_ges": feature in FULL_FEATURES,
        "used_by_no_star_ges": feature in NO_STAR_FEATURES,
        "t0_nonmissing": int(t0_values.notna().sum()),
        "t0_missing": int(t0_values.isna().sum()),
        "t0_min": t0_min,
        "t0_max": t0_max,
        "t1_nonmissing": int(t1_values.notna().sum()),
        "t1_missing": int(t1_values.isna().sum()),
        "t1_missing_fraction": float(t1_values.isna().mean()),
        "t1_min": t1_min,
        "t1_max": t1_max,
        "t1_below_t0_range": int((t1_nonmissing < t0_min).sum()),
        "t1_above_t0_range": int((t1_nonmissing > t0_max).sum()),
        "t1_outside_frozen_domain": int(
            ((t1_nonmissing < 0.0) | (t1_nonmissing > domain_upper)).sum()
        ),
        "t1_infinite_values": int(np.isinf(t1_values.to_numpy()).sum()),
        "frozen_median_imputer_available": True,
        "row_level_feature_persisted": False,
    })

feature_inventory = pd.DataFrame(feature_inventory_rows)

axis_inventory = (
    pd.DataFrame({
        "classification_axis": axis_t1,
        "target_gene": gene_t1,
    })
    .groupby(["classification_axis", "target_gene"], dropna=False)
    .size()
    .reset_index(name="rows")
)


def axis_policy(axis):
    if axis == "GermlineClassification":
        return "primary_germline_domain_candidate"
    if axis == "NoClassification":
        return "retain_as_evidence_only_not_as_classification"
    return "retain_separately_requires_prespecified_non_germline_policy"


axis_inventory["scientific_applicability"] = axis_inventory["classification_axis"].map(axis_policy)
axis_inventory["included_in_preprocessing_check"] = True
axis_inventory["score_created_in_cell_7a2"] = False

observed_gene_counts = {
    str(key): int(value)
    for key, value in gene_t1.value_counts(dropna=False).items()
}
observed_axis_counts = {
    str(key): int(value)
    for key, value in axis_t1.value_counts(dropna=False).items()
}


# --------------------------------------------------------------------------------------------------
# 12. VERIFY THE COMBINED-METADATA POLICY WITHOUT APPLYING IT
# --------------------------------------------------------------------------------------------------

stage6a_policy_payload = json.loads(STAGE6A_POLICY.read_text(encoding="utf-8"))
stage6a_policy_text = json.dumps(stage6a_policy_payload, sort_keys=True).lower()
combined_metadata_policy_identified = (
    "combined_metadata_instability_risk" in stage6a_policy_text
    and "frozen_before_outcome_label_load_or_temporal_performance" in stage6a_policy_text
)


# --------------------------------------------------------------------------------------------------
# 13. IMMUTABILITY RECHECK
# --------------------------------------------------------------------------------------------------

immutable_hashes_after = {
    key: sha256_file(path)
    for key, path in ARTIFACT_PATHS.items()
}
immutable_inputs_unchanged = immutable_hashes_before == immutable_hashes_after


# --------------------------------------------------------------------------------------------------
# 14. QUALITY-CONTROL REGISTER
# --------------------------------------------------------------------------------------------------

entropy_parse_errors = int(entropy_status_t1.eq("parse_error").sum())
entropy_invalid_values = int(entropy_status_t1.eq("invalid_numeric_value").sum())
entropy_missing_rows = int(entropy_t1.isna().sum())
all_infinite_feature_values = int(sum(
    np.isinf(pd.to_numeric(t1_features[column], errors="coerce").to_numpy()).sum()
    for column in t1_features.columns
))

qc_checks = OrderedDict([
    ("all_18_frozen_hashes_verified", all(
        observed_hashes[key] == EXPECTED_HASHES[key] for key in EXPECTED_HASHES
    )),
    ("cell_7a1_manifest_sidecar_verified", sidecar_is_valid(CELL_7A1_MANIFEST)),
    ("cell_7a1_terminal_decision_verified", cell_7a1_payload.get("terminal_decision") == EXPECTED_CELL_7A1_DECISION),
    ("cell_7a2_authorized", cell_7a1_payload.get("next_authorized_cell", {}).get("cell_id") == "7A2"),
    ("t1_dimensions_verified", t1_meta.num_rows == EXPECTED_T1_ROWS and t1_meta.num_columns == EXPECTED_T1_COLUMNS),
    ("stage4a_dimensions_verified", stage4a_meta.num_rows == EXPECTED_STAGE4A_ROWS and stage4a_meta.num_columns == EXPECTED_STAGE4A_COLUMNS),
    ("stage4b_dimensions_verified", stage4b_meta.num_rows == EXPECTED_STAGE4B_ROWS and stage4b_meta.num_columns == EXPECTED_STAGE4B_COLUMNS),
    ("stage4a_zero_based_row_order_verified", stage4a_row_order_valid),
    ("stage4b_zero_based_row_order_verified", stage4b_row_order_valid),
    ("stage4a_stage4b_row_order_alignment_verified", stage4a_stage4b_row_order_alignment_verified),
    ("stage4a_stage4b_rcv_alignment_verified", stage4a_stage4b_rcv_alignment_verified),
    ("t0_recency_formula_exact", recency_formula_mismatches == 0),
    ("t0_recency_missing_formula_exact", recency_missing_mismatches == 0),
    ("t0_log_submitter_formula_exact", log_submitter_mismatches == 0),
    ("t0_submitter_score_formula_exact", submitter_formula_mismatches == 0),
    ("t0_review_confidence_formula_exact", review_formula_mismatches == 0),
    ("t0_submitter_min_matches_frozen_parameter", np.isclose(observed_log_min, SUBMITTER_LOG_MIN, rtol=1e-12, atol=1e-12)),
    ("t0_submitter_max_matches_frozen_parameter", np.isclose(observed_log_max, SUBMITTER_LOG_MAX, rtol=1e-12, atol=1e-12)),
    ("t1_loaded_rows_verified", len(t1) == EXPECTED_T1_ROWS),
    ("t1_rcv_nonmissing", int(rcv_t1.isna().sum()) == 0 and int(rcv_t1.fillna("").eq("").sum()) == 0),
    ("t1_rcv_unique", rcv_t1.nunique(dropna=False) == EXPECTED_T1_ROWS),
    ("t1_gene_complete", int(gene_t1.fillna("").eq("").sum()) == 0),
    ("t1_gene_counts_verified", observed_gene_counts == EXPECTED_T1_GENE_COUNTS),
    ("t1_axis_counts_verified", observed_axis_counts == EXPECTED_T1_AXIS_COUNTS),
    ("t1_cutoff_verified", set(cutoff_t1.dropna().unique().tolist()) == {"2025-12-27"}),
    ("t1_no_post_cutoff_dates", post_cutoff_dates == 0),
    ("t1_submitter_counts_positive", int((submitter_count_t1.dropna() <= 0).sum()) == 0),
    ("t1_review_stars_in_valid_clinvar_domain", int(((review_confidence_t1.dropna() < 0) | (review_confidence_t1.dropna() > 4)).sum()) == 0),
    ("t1_conflict_values_complete", int(conflict_t1.isna().sum()) == 0),
    ("t1_entropy_json_parse_errors_zero", entropy_parse_errors == 0),
    ("t1_entropy_invalid_numeric_values_zero", entropy_invalid_values == 0),
    ("t1_entropy_complete", entropy_missing_rows == 0),
    ("t1_entropy_within_domain", int(((entropy_t1 < 0) | (entropy_t1 > 1)).sum()) == 0),
    ("t1_group_counts_reconcile_to_scv_count", scv_group_count_mismatches == 0),
    ("t1_no_infinite_feature_values", all_infinite_feature_values == 0),
    ("full_ges_feature_order_verified", model_results["full_ges"]["feature_order_verified"]),
    ("full_ges_fitted_state_verified", model_results["full_ges"]["fitted_state_verified"]),
    ("full_ges_settings_verified", model_results["full_ges"]["settings_verified"]),
    ("full_ges_preprocessing_shape_verified", model_results["full_ges"]["rows"] == EXPECTED_T1_ROWS and model_results["full_ges"]["columns"] == len(FULL_FEATURES)),
    ("full_ges_preprocessing_all_finite", model_results["full_ges"]["all_finite"]),
    ("no_star_feature_order_verified", model_results["no_star_ges"]["feature_order_verified"]),
    ("no_star_fitted_state_verified", model_results["no_star_ges"]["fitted_state_verified"]),
    ("no_star_settings_verified", model_results["no_star_ges"]["settings_verified"]),
    ("no_star_preprocessing_shape_verified", model_results["no_star_ges"]["rows"] == EXPECTED_T1_ROWS and model_results["no_star_ges"]["columns"] == len(NO_STAR_FEATURES)),
    ("no_star_preprocessing_all_finite", model_results["no_star_ges"]["all_finite"]),
    ("combined_metadata_policy_identified", combined_metadata_policy_identified),
    ("immutable_inputs_unchanged", immutable_inputs_unchanged),
    ("weak_labels_not_created", True),
    ("model_fit_not_called", True),
    ("model_refit_not_called", True),
    ("predict_not_called", True),
    ("predict_proba_not_called", True),
    ("decision_function_not_called", True),
    ("t1_full_ges_score_not_created", True),
    ("t1_no_star_score_not_created", True),
    ("t1_combined_metadata_score_not_created", True),
    ("threshold_or_weight_optimization_not_performed", True),
    ("row_level_t1_feature_artifact_not_written", True),
    ("rag_corpus_not_constructed", True),
    ("embeddings_not_constructed", True),
    ("question_set_not_constructed", True),
    ("llm_not_called", True),
])

failed_checks = [name for name, passed in qc_checks.items() if passed is not True]
passed_checks = len(qc_checks) - len(failed_checks)
all_checks_passed = len(failed_checks) == 0

final_decision = (
    "PASS_STAGE7A2_FROZEN_FEATURE_TRANSFORMS_AND_MODEL_PACKAGES_VERIFIED_"
    "T1_FEATURE_RECONSTRUCTION_AND_PREPROCESSING_APPLICABILITY_CONFIRMED_"
    "NO_SCORING_STAGE7A3_FROZEN_T1_SCORE_MATERIALIZATION_AUTHORIZED"
    if all_checks_passed
    else "FAIL_STAGE7A2_PREFLIGHT_STAGE7A3_NOT_AUTHORIZED"
)

# Do not freeze a failed preflight package. This preserves clean rerun behavior.
if not all_checks_passed:
    print("CELL 7A2 PREFLIGHT FAILED BEFORE OUTPUT FREEZE")
    print("Failed checks:")
    for failed_name in failed_checks:
        print(" -", failed_name)
    raise RuntimeError(
        "Cell 7A2 preflight failed before any Cell 7A2 artifact was written."
    )


# --------------------------------------------------------------------------------------------------
# 15. PRESERVE CREATION TIMESTAMP ACROSS SAFE RERUNS
# --------------------------------------------------------------------------------------------------

existing_created_utc = None
for candidate in [OUTPUTS["manifest"], OUTPUTS["preflight_report"], OUTPUTS["qc"]]:
    if candidate.exists():
        try:
            existing_created_utc = json.loads(
                candidate.read_text(encoding="utf-8")
            ).get("created_utc")
            if existing_created_utc:
                break
        except Exception:
            pass

created_utc = existing_created_utc or datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 16. WRITE TABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

output_hashes = OrderedDict()

for key, frame in [
    ("artifact_inventory", artifact_inventory),
    ("formula_audit", formula_audit),
    ("feature_inventory", feature_inventory),
    ("model_inventory", model_inventory),
    ("axis_inventory", axis_inventory),
]:
    output_hashes[key] = stable_write_csv(OUTPUTS[key], frame)
    write_sidecar(OUTPUTS[key])
    if not sidecar_is_valid(OUTPUTS[key]):
        raise AssertionError(f"Output sidecar verification failed: {OUTPUTS[key]}")


# --------------------------------------------------------------------------------------------------
# 17. WRITE PREFLIGHT AND QC JSON
# --------------------------------------------------------------------------------------------------

preflight_report = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": created_utc,
    "purpose": "Frozen T1 feature reconstruction and model-applicability preflight without scoring.",
    "authorization": {
        "prior_cell": "7A1",
        "prior_manifest_path": str(CELL_7A1_MANIFEST),
        "prior_manifest_sha256": sha256_file(CELL_7A1_MANIFEST),
        "prior_terminal_decision_verified": True,
        "model_fitting_authorized": False,
        "t1_scoring_authorized_in_cell_7a2": False,
        "threshold_or_weight_optimization_authorized": False,
        "embedding_construction_authorized": False,
        "llm_generation_authorized": False,
    },
    "frozen_inputs": {
        key: {"path": str(ARTIFACT_PATHS[key]), "sha256": observed_hashes[key]}
        for key in ARTIFACT_PATHS
    },
    "t1_source_column_mapping": t1_columns,
    "frozen_transformation_constants": {
        "source_path": str(STAGE4B_TRANSFORM_PARAMETERS),
        "source_sha256": sha256_file(STAGE4B_TRANSFORM_PARAMETERS),
        "maximum_observation_window_days": MAXIMUM_OBSERVATION_WINDOW_DAYS,
        "submitter_log_min": SUBMITTER_LOG_MIN,
        "submitter_log_max": SUBMITTER_LOG_MAX,
        "observed_t0_submitter_log_min": observed_log_min,
        "observed_t0_submitter_log_max": observed_log_max,
    },
    "t0_formula_reproduction": {
        row["transformation"]: {
            "rows_audited": int(row["rows_audited"]),
            "mismatch_count": int(row["mismatch_count"]),
            "exactly_reproduced": bool(row["exactly_reproduced"]),
        }
        for row in formula_audit.to_dict("records")
    },
    "t1_accounting": {
        "rows": int(len(t1)),
        "columns": int(t1_meta.num_columns),
        "unique_rcv_accessions": int(rcv_t1.nunique(dropna=False)),
        "gene_counts": observed_gene_counts,
        "classification_axis_counts": observed_axis_counts,
        "embedded_cutoff_values": sorted(cutoff_t1.dropna().unique().tolist()),
        "post_cutoff_last_evaluated_dates": post_cutoff_dates,
    },
    "t1_feature_reconstruction": {
        "recency_missing_rows": int(recency_score_t1.isna().sum()),
        "entropy_missing_rows": entropy_missing_rows,
        "entropy_parse_errors": entropy_parse_errors,
        "entropy_invalid_numeric_values": entropy_invalid_values,
        "group_count_vs_scv_count_mismatches": scv_group_count_mismatches,
        "row_level_feature_table_persisted": False,
    },
    "frozen_model_applicability": model_results,
    "combined_metadata_policy": {
        "path": str(STAGE6A_POLICY),
        "sha256": sha256_file(STAGE6A_POLICY),
        "identified": combined_metadata_policy_identified,
        "applied": False,
    },
    "prohibited_operations_confirmed": {
        "weak_labels_created": False,
        "model_fit_called": False,
        "model_refit_called": False,
        "predict_called": False,
        "predict_proba_called": False,
        "decision_function_called": False,
        "full_ges_score_created": False,
        "no_star_score_created": False,
        "combined_metadata_score_created": False,
        "threshold_optimized": False,
        "weight_optimized": False,
        "rag_corpus_constructed": False,
        "embeddings_constructed": False,
        "question_set_constructed": False,
        "llm_called": False,
    },
    "qc_summary": {
        "passed_checks": passed_checks,
        "failed_checks": len(failed_checks),
        "total_checks": len(qc_checks),
        "failed_check_names": failed_checks,
    },
    "decision": final_decision,
    "next_authorized_cell": (
        {
            "cell_id": "7A3",
            "scope": (
                "Apply the frozen Full-GES, No-star-GES, and combined-metadata "
                "specifications to T1 and checksum-freeze the T1 score package."
            ),
            "model_fitting": "PROHIBITED",
            "threshold_or_weight_optimization": "PROHIBITED",
            "embedding_construction": "PROHIBITED",
            "llm_generation": "PROHIBITED",
        }
        if all_checks_passed else None
    ),
}

output_hashes["preflight_report"] = stable_write_json(
    OUTPUTS["preflight_report"], preflight_report
)
write_sidecar(OUTPUTS["preflight_report"])

qc_payload = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": created_utc,
    "checks": [
        {"check": name, "passed": bool(passed)}
        for name, passed in qc_checks.items()
    ],
    "passed_checks": passed_checks,
    "failed_checks": len(failed_checks),
    "total_checks": len(qc_checks),
    "failed_check_names": failed_checks,
    "diagnostics": {
        "t0_recency_formula_mismatches": recency_formula_mismatches,
        "t0_recency_missing_mismatches": recency_missing_mismatches,
        "t0_recency_missing_stage4a_mismatches": recency_missing_stage4a_mismatches,
        "t0_recency_missing_stage4b_mismatches": recency_missing_stage4b_mismatches,
        "stage4a_stage4b_row_order_alignment_verified": stage4a_stage4b_row_order_alignment_verified,
        "stage4a_stage4b_rcv_alignment_verified": stage4a_stage4b_rcv_alignment_verified,
        "t0_log_submitter_mismatches": log_submitter_mismatches,
        "t0_submitter_score_mismatches": submitter_formula_mismatches,
        "t0_review_confidence_mismatches": review_formula_mismatches,
        "t1_post_cutoff_dates": post_cutoff_dates,
        "t1_entropy_parse_errors": entropy_parse_errors,
        "t1_entropy_invalid_numeric_values": entropy_invalid_values,
        "t1_entropy_missing_rows": entropy_missing_rows,
        "t1_group_count_vs_scv_count_mismatches": scv_group_count_mismatches,
    },
    "decision": final_decision,
}

output_hashes["qc"] = stable_write_json(OUTPUTS["qc"], qc_payload)
write_sidecar(OUTPUTS["qc"])

for key in ["preflight_report", "qc"]:
    if not sidecar_is_valid(OUTPUTS[key]):
        raise AssertionError(f"Output sidecar verification failed: {OUTPUTS[key]}")


# --------------------------------------------------------------------------------------------------
# 18. WRITE AND VERIFY THE CELL 7A2 MANIFEST
# --------------------------------------------------------------------------------------------------

output_records = []
for key, path in OUTPUTS.items():
    if key == "manifest":
        continue
    record = {
        "role": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha256_file(path),
        "bytes": int(path.stat().st_size),
        "sidecar_path": str(sidecar_path(path)),
        "sidecar_verified": sidecar_is_valid(path),
    }
    if path.suffix.lower() == ".csv":
        frame = pd.read_csv(path)
        record["rows"] = int(len(frame))
        record["columns"] = int(frame.shape[1])
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True
    output_records.append(record)

manifest_payload = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "package_version": "v1",
    "created_utc": created_utc,
    "purpose": "Frozen T1 feature reconstruction and model-applicability preflight without score generation.",
    "upstream_lineage": {
        key: {"path": str(path), "sha256": observed_hashes[key]}
        for key, path in ARTIFACT_PATHS.items()
    },
    "scientific_boundary": {
        "t1_features_reconstructed_in_memory": True,
        "row_level_t1_feature_table_persisted": False,
        "frozen_preprocessing_applied": True,
        "classifier_scoring_applied": False,
        "full_ges_score_created": False,
        "no_star_score_created": False,
        "combined_metadata_score_created": False,
        "weak_labels_created": False,
        "model_fitted": False,
        "model_refitted": False,
        "model_tuned": False,
        "threshold_optimized": False,
        "weight_optimized": False,
        "rag_corpus_constructed": False,
        "embeddings_constructed": False,
        "question_set_constructed": False,
        "llm_called": False,
    },
    "output_artifacts": output_records,
    "qc": {
        "path": str(OUTPUTS["qc"]),
        "sha256": sha256_file(OUTPUTS["qc"]),
        "passed_checks": passed_checks,
        "failed_checks": len(failed_checks),
        "total_checks": len(qc_checks),
    },
    "decision": final_decision,
    "next_authorized_cell": (
        {
            "cell_id": "7A3",
            "name": "Frozen T1 score materialization and checksum freeze",
        }
        if all_checks_passed else None
    ),
}

output_hashes["manifest"] = stable_write_json(OUTPUTS["manifest"], manifest_payload)
write_sidecar(OUTPUTS["manifest"])

if not sidecar_is_valid(OUTPUTS["manifest"]):
    raise AssertionError("Cell 7A2 manifest sidecar verification failed.")

manifest_readback = json.loads(OUTPUTS["manifest"].read_text(encoding="utf-8"))
if manifest_readback.get("decision") != final_decision:
    raise AssertionError("Cell 7A2 manifest decision readback failed.")

for path in OUTPUTS.values():
    if not sidecar_is_valid(path):
        raise AssertionError(f"Fresh output verification failed: {path}")


# --------------------------------------------------------------------------------------------------
# 19. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

separator = "=" * 144

print("\n" + separator)
print("EXPERIMENT 2 — STAGE 7A — CELL 7A2")
print("FROZEN T1 FEATURE RECONSTRUCTION AND MODEL-APPLICABILITY PREFLIGHT")
print(separator)
print(f"Notebook                                      : {NOTEBOOK_NAME}")
print(f"Project root                                  : {ROOT}")

print("\nUPSTREAM AUTHORIZATION")
print(f"Cell 7A1 manifest SHA-256                     : {sha256_file(CELL_7A1_MANIFEST)}")
print("Cell 7A1 terminal PASS verified               : YES")
print("Model fitting authorized                      : NO")
print("T1 scoring authorized in Cell 7A2             : NO")
print("Embedding construction authorized             : NO")
print("LLM generation authorized                     : NO")

print("\nFROZEN INPUT VERIFICATION")
print(f"Frozen artifacts verified                     : {len(observed_hashes)}/{len(EXPECTED_HASHES)}")
print(f"T1 Parquet SHA-256                            : {sha256_file(T1_PARQUET)}")
print(f"Stage 4A feature table SHA-256                 : {sha256_file(STAGE4A_FEATURE_TABLE)}")
print(f"Stage 4B transform table SHA-256               : {sha256_file(STAGE4B_TABLE)}")
print(f"Stage 4C Full-GES model SHA-256                : {sha256_file(FULL_MODEL_PATH)}")
print(f"Stage 4C No-star model SHA-256                 : {sha256_file(NO_STAR_MODEL_PATH)}")
print(f"Stage 6A policy SHA-256                        : {sha256_file(STAGE6A_POLICY)}")

print("\nFROZEN TRANSFORMATION REPRODUCTION")
print(f"Frozen recency window days                    : {MAXIMUM_OBSERVATION_WINDOW_DAYS:.17g}")
print(f"Frozen submitter log minimum                  : {SUBMITTER_LOG_MIN:.17g}")
print(f"Observed T0 submitter log minimum             : {observed_log_min:.17g}")
print(f"Frozen submitter log maximum                  : {SUBMITTER_LOG_MAX:.17g}")
print(f"Observed T0 submitter log maximum             : {observed_log_max:.17g}")
for row in formula_audit.to_dict("records"):
    print(f"{row['transformation']:<46}: {int(row['mismatch_count']):,} mismatches")

print("\nT1 FEATURE RECONSTRUCTION — IN MEMORY ONLY")
print(f"T1 rows                                       : {len(t1):,}")
print(f"Unique RCV accessions                         : {rcv_t1.nunique(dropna=False):,}")
print(f"Recency missing                               : {int(recency_score_t1.isna().sum()):,}")
print(f"Entropy missing                               : {entropy_missing_rows:,}")
print(f"Entropy JSON parse errors                     : {entropy_parse_errors:,}")
print(f"SCV/group-count mismatches                    : {scv_group_count_mismatches:,}")
print(f"Post-cutoff dates                             : {post_cutoff_dates:,}")
print(f"Infinite derived feature values               : {all_infinite_feature_values:,}")
print("Row-level reconstructed feature table saved   : NO")

print("\nFROZEN MODEL APPLICABILITY")
for row in model_inventory.to_dict("records"):
    print(
        f"{row['model']:<46}: "
        f"{int(row['t1_preprocessing_rows']):,} rows × "
        f"{int(row['t1_preprocessing_columns'])} features | "
        f"finite={row['t1_preprocessing_all_finite']} | scored=NO"
    )

print("\nCLASSIFICATION-AXIS ACCOUNTING")
for axis, expected in EXPECTED_T1_AXIS_COUNTS.items():
    print(f"{axis:<46}: {int(observed_axis_counts.get(axis, 0)):,}")

print("\nCELL 7A2 FROZEN OUTPUTS")
for label, key in [
    ("Frozen artifact inventory", "artifact_inventory"),
    ("T0 transformation formula audit", "formula_audit"),
    ("T1 feature applicability inventory", "feature_inventory"),
    ("Frozen model pipeline inventory", "model_inventory"),
    ("Classification-axis applicability", "axis_inventory"),
    ("Preflight report", "preflight_report"),
    ("QC record", "qc"),
    ("Manifest", "manifest"),
]:
    path = OUTPUTS[key]
    print(f"{label:<46}: {path}")
    print(f"{'SHA-256':<46}: {sha256_file(path)}")

print(f"\nQC checks                                      : {passed_checks}/{len(qc_checks)} PASS")

print("\nSCIENTIFIC OPERATIONS")
print("Full-GES fitted or refitted                    : NO")
print("No-star GES fitted or refitted                 : NO")
print("Full-GES T1 score generated                    : NO")
print("No-star T1 score generated                     : NO")
print("Combined-metadata T1 score generated           : NO")
print("Threshold or weight optimization               : NO")
print("RAG corpus constructed                         : NO")
print("Embeddings constructed                         : NO")
print("LLM called                                     : NO")

print("\nNEXT AUTHORIZED CELL")
if all_checks_passed:
    print("Cell 7A3                                      : Frozen T1 Full-GES, No-star-GES,")
    print("                                                 and combined-metadata score")
    print("                                                 materialization and checksum freeze")
    print("Model fitting                                 : PROHIBITED")
    print("Threshold or weight optimization              : PROHIBITED")
    print("Embedding construction                        : PROHIBITED")
    print("LLM generation                                : PROHIBITED")
else:
    print("Cell 7A3                                      : NOT AUTHORIZED")

print(f"\nFINAL DECISION                                : {final_decision}")
print(separator)

if not all_checks_passed:
    raise RuntimeError(
        "Cell 7A2 preflight failed. Failed checks:\n- "
        + "\n- ".join(failed_checks)
    )

CELL 7A2 PREFLIGHT FAILED BEFORE OUTPUT FREEZE
Failed checks:
 - t0_submitter_min_matches_frozen_parameter
 - t0_submitter_max_matches_frozen_parameter


RuntimeError: Cell 7A2 preflight failed before any Cell 7A2 artifact was written.